# AI Hub 조합 데이터 EDA (외부 데이터 보충)

## EDA의 목적
- 알약 검출 모델의 학습 데이터를 AI Hub 576 "경구약제 이미지" 허용 조합 데이터로 보충하기에 앞서, 외부 데이터의 형태, 라벨 품질, 분포를 대회 원본 EDA와 같은 순서로 확인한다.
- 본 EDA를 통해 AI Hub 라벨의 변환 방법과 정제 목록을 확정하고, 어떤 AI Hub 이미지를 학습에 넣을지 정한다.
- 각 섹션의 분석 결과를 보충 데이터 변환, 라벨 정제, 입력 크기와 증강, 클래스 체계, 보충 이미지 선택, 평가 방법에 대한 결정으로 연결하고, 7번 EDA 결론에서 결정 사항 표로 정리한다.
- 대회 원본 분석과 원본 라벨 정제는 [00_raewoo_first_eda.ipynb](00_raewoo_first_eda.ipynb)에 있다.
    - 이 노트북은 00이 저장한 원본 정제 목록(`00_raewoo_label_fixes.csv`)과 test 기록표(`00_raewoo_test_review.csv`)를 읽어 쓴다. 00을 먼저 실행하지 않아도 된다.
- 학습용 분할은 이 노트북에서 정하지 않는다. (팀원 작업)
- 데이터 준비 (README의 AI Hub 외부 데이터 받기)
    - 대회에서 금지한 `TL_2_조합.zip`, `TS_2_조합.zip`은 받지 않는다. (RULE.md)
    - 라벨 zip 8개(약 59MB)를 받아 푼다.
        - `uv run python script/download_aihub.py 66065 66067 66068 66069 66070 66071 66072 66243`
        - `uv run python script/extract_aihub.py data/external/aihub/train_annotations/*.zip`
    - 이미지 zip 8개(각 약 3GB)는 디스크를 아끼려고 하나씩 받고, 풀고, 지운다.
        - `uv run python script/download_aihub.py 66154`
        - `uv run python script/extract_aihub.py data/external/aihub/train_images/TS_1_조합.zip --delete-images`
        - filekey 66156~66161(TS_3~8), 66254(VS_1)도 같은 방법으로 받는다.

## EDA 목차
> 각 분석 결과 아래에는 **인사이트 / 결론** 마크다운 셀을 두어, 분석 결과와 그에 따른 결정을 기록한다.

0. 사전 설정
    - seed, 데이터 경로, 그림 저장 경로, 한글 폰트, 도우미 함수를 정의한다.
    - 대회 원본 라벨, 00의 정제 목록과 test 기록표, AI Hub 라벨 zip을 읽고, 모든 이미지의 크기·밝기·축소 그림을 한 번에 계산해 둔다.
1. 데이터셋 훑어보기
    - **파일 구성과 금지 범위**
        - AI Hub 576 데이터셋에서 쓸 수 있는 파일과 금지된 파일을 정리하고, 받은 라벨과 이미지의 수, 해상도, 깨진 파일을 확인한다.
        - 1차 정제: 촬영 PNG가 없는 JSON과 `_index.png`를 뺀다.
            - 목적: 어떤 전처리로도 복구할 수 없는 파일을 먼저 거른다.
    - **라벨 구조 비교**
        - 1차 정제한 AI Hub JSON(바꾸기 전)과 Kaggle JSON의 폴더 구조, 키 구성, id 값, 클래스 표기를 비교하고, 같은 알약의 JSON 두 개를 키마다 나란히 놓는다.
    - **Kaggle 형식으로 바꾼 뒤 다시 비교**
        - AI Hub JSON에서 Kaggle과 형식이 다른 키 8개(id, category_id 등)만 바꾸고, 같은 비교를 다시 해 사진에 따라 다른 값만 남는지 확인한다.
            - 목적: AI Hub 라벨을 Kaggle 라벨과 같은 코드로 읽을 수 있게 바꾸는 방법을 정한다. 이후 분석은 바꾼 라벨로 한다.
    - **촬영 조건 비교**
        - 메타데이터의 촬영 조건과, AI Hub·대회 train·test의 채널별 평균 밝기를 비교한다.
            - 목적: AI Hub 사진을 대회 사진과 그대로 섞어 학습해도 되는지 정한다.
    - **이미지 육안 확인**
        - 라벨 오류가 없는 이미지를 zip마다 1장씩 8장 골라 box를 겹쳐 그린다. (seed 고정)
            - 목적: AI Hub 원본 box가 알약을 제대로 감싸는지 확인한다.
    - **누수 점검**
        - 파일 이름, 파일 내용, 축소 그림으로 AI Hub와 대회 train·test, AI Hub zip끼리 같은 사진이 있는지 찾는다.
            - 목적: 대회 데이터와 겹치는 사진을 학습에서 뺄지 정하고, 대회 데이터의 출처를 확인한다.
2. 라벨 품질
    - **라벨 누락과 클래스 불일치**
        - 파일명의 K-코드와 bbox의 category_id를 대조하고, 빠진 라벨을 원인별로 나눠 규모와 예시를 본다.
        - 같은 K-코드의 JSON끼리 알약 정보(제품명, 각인)가 같은지 확인한다.
        - copy 파일이 있는 알약 폴더의 JSON 3개를 세 각도 사진에 그려 보고 맞는 각도에 다시 배정한다. 2-2부터는 다시 배정한 라벨로 본다.
    - **좌표 오류와 고치는 방법**
        - 이미지 밖 bbox, area 오타, 다른 클래스와 같은 bbox, 숫자가 붙은 bbox를 찾는다.
        - 빠진 box를 OpenCV로 사진에서 찾아 그려 넣는 방법을 만들고, 라벨이 있는 box로 정확도를 먼저 확인한다.
        - `-오류` 폴더의 JSON이 왜 오류인지 사진으로 확인한다.
    - **클래스별 crop 갤러리**
        - 모든 bbox를 클래스별로 잘라 시트 여러 장에 나눠 저장하고, 잘못된 클래스와 헐거운 box를 찾는다.
            - 대회 train과 같은 클래스는 대회 crop을 첫 줄에 놓아 같은 알약인지 대조한다.
    - **라벨 정제 목록**
        - 2-1~2-3에서 찾은 라벨 오류를 모아 처리 방법을 정하고 CSV(`01_raewoo_aihub_label_fixes.csv`)로 저장한다.
    - **라벨 정제 처리**
        - 정제 목록을 적용한 AI Hub 라벨을 만든다. 3번부터는 정제 라벨로 본다.
            - 목적: 고치거나 뺄 라벨과 이미지를 정한다.
            - 참고: 평가 지표 mAP@[0.75:0.95]는 IoU 0.75 미만인 box를 오답으로 처리하므로, box 오차에 매우 민감하다. (00 8-2)
3. bbox 분포
    - **객체 수와 크기**
        - 이미지당 객체 수, box 크기를 대회 train과 비교하고, 같은 알약의 box 크기(여백)가 대회 라벨과 같은지 본다.
            - 목적: 00에서 정한 입력 크기와 증강이 AI Hub를 섞은 뒤에도 맞는지 확인하고, 라벨 여백 차이로 점수가 깎이지 않는지 확인한다.
    - **위치와 겹침**
        - box 중심 위치와, 같은 이미지 안 box끼리의 IoU를 대회 train과 비교한다.
            - 목적: 00에서 정한 후처리 설정(NMS IoU, 최대 검출 수)이 그대로 맞는지 확인한다.
4. 클래스 분포
    - **클래스별 객체 수와 등장 조합 수**
        - AI Hub의 클래스별 객체 수와 조합 수를 대회 train 클래스와 train에 없는 클래스로 나눠 본다.
    - **조합당 이미지 수**
        - 조합 하나에 사진이 몇 장인지와 촬영 각도 분포를 확인한다.
    - **같은 조합 사진 비교**
        - 같은 조합의 각도별 사진 3장을 나란히 본다.
    - **대회 train과 합친 분포**
        - 정제한 대회 train과 AI Hub를 합쳤을 때의 클래스 분포와 불균형 정도를 본다.
    - **가장 균일하게 고르기**
        - 클래스별 목표 객체 수를 채우는 이미지를 고르고, 전부 쓸 때와 클래스 분포를 비교한다.
            - 목적: 학습 클래스 체계와 학습에 넣을 AI Hub 이미지, 클래스 불균형 대응 방법을 정한다.
5. test 전용 클래스 채우기
    - **test에서 읽은 각인 대조**
        - 00 5-3 기록표의 각인을 AI Hub 라벨의 각인 필드에서 찾는다.
    - **test 알약과 AI Hub 알약 나란히 보기**
        - train에 없는 test 알약의 사진과, 각인으로 찾은 AI Hub 알약의 crop을 나란히 놓고 같은 알약인지 눈으로 대조한다.
    - **test 사진 단위 커버리지**
        - 모든 알약이 학습 클래스에 있는 test 사진의 비율을 보충 전후로 센다.
            - 목적: test에만 있는 알약이 보충으로 얼마나 채워지는지 확인하고, 남는 알약을 정한다.
6. 평가 기준선
    - **변환 라벨로 mAP 계산 검증**
        - 바꾼 AI Hub 라벨을 정답과 예측으로 함께 넣었을 때 mAP@[0.75:0.95]가 1.0인지 확인한다.
            - 목적: 118개 클래스 체계로도 로컬 평가 코드가 맞게 동작하는지 확인한다.
7. EDA 결론
    - 1~6번의 결과를 보충 데이터 변환, 라벨 정제, 클래스 체계, 보충 이미지 선택, 클래스 불균형, 평가 방법 항목별 결정 사항 표로 정리한다.

## 0. 사전 설정

In [ ]:
import collections
import concurrent.futures
import contextlib
import csv
import hashlib
import io
import json
import os
import shutil
import sys
import unicodedata
import zipfile
from pathlib import Path

import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy.optimize
from IPython.display import Markdown, display
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageFont
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# 금지 파일 검사, 파일명 해석, Kaggle 형식 읽기는 공용 모듈의 함수를 쓴다. AI Hub 원본 JSON은 0번에서 직접 읽는다
from pill_detection.data import check_aihub_file, combo_name, load_annotations, pill_codes

# 실행 환경의 버전 확인 (팀원 간 환경 차이로 오류가 생겼을 때 원인 파악용)
print(f"Python       : {sys.version.split()[0]}")
print(f"numpy        : {np.__version__}")
print(f"matplotlib   : {matplotlib.__version__}")
print(f"OpenCV       : {cv2.__version__}")

In [ ]:
# 랜덤 샘플 시각화 등의 재현성을 위한 seed 고정
SEED = 42
np.random.seed(SEED)

# 노트북을 어디서 실행해도 레포 최상위(pyproject.toml이 있는 폴더)를 기준으로 경로를 잡는다
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
TRAIN_IMG_DIR = RAW_DIR / 'train_images'
TRAIN_ANN_DIR = RAW_DIR / 'train_annotations'
TEST_IMG_DIR = RAW_DIR / 'test_images'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'
AIHUB_DIR = EXTERNAL_DIR / 'aihub'  # script/download_aihub.py로 받은 zip
AIHUB_RAW_ANN_DIR = EXTERNAL_DIR / 'aihub_raw' / 'train_annotations'  # 라벨 zip을 그대로 푼 원본 JSON
AIHUB_ANN_DIR = EXTERNAL_DIR / 'aihub_converted' / 'train_annotations'  # Kaggle 형식으로 바꾼 JSON을 둘 폴더 (지금은 저장하지 않는다. 1-3은 메모리에서만 바꾼다)
AIHUB_IMG_DIR = EXTERNAL_DIR / 'aihub_converted' / 'train_images'  # 이미지 zip의 PNG
AIHUB_SOURCES_CSV = EXTERNAL_DIR / 'aihub_converted' / 'train_images_sources.csv'  # 이미지마다 나온 zip
EDA_DIR = PROJECT_ROOT / 'notebooks' / 'eda'
# 그림과 계산 결과 저장 경로 (runs/는 git에서 제외됨)
FIG_DIR = PROJECT_ROOT / 'runs' / 'eda_raewoo' / 'figures_01'
CACHE_DIR = PROJECT_ROOT / 'runs' / 'eda_raewoo' / 'cache'
FIG_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 대회와 AI Hub 이미지 크기 (너비, 높이)
IMG_W, IMG_H = 976, 1280

# 그래프의 한글이 깨지지 않도록 설치된 한글 폰트 사용 (macOS, Windows, Linux 순)
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ['AppleGothic', 'Malgun Gothic', 'NanumGothic']:
    if font_name in installed_fonts:
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False  # 한글 폰트에서 마이너스 기호가 깨지는 문제 방지


def save_fig(fig, name):
    """그림을 FIG_DIR에 png 파일로 저장한다."""
    fig.savefig(FIG_DIR / f'{name}.png', dpi=120, bbox_inches='tight')


def show_table(header, rows):
    """열 이름 리스트와 행 리스트를 마크다운 표로 출력한다."""
    lines = ['| ' + ' | '.join(header) + ' |', '|' + ' --- |' * len(header)]
    # 값 속 |(예: 분할선 각인 K|50)가 칸 구분자로 읽히지 않게 \|로 바꾼다
    lines += ['| ' + ' | '.join(str(value).replace('|', r'\|') for value in row) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))


def box_iou(box_a, box_b):
    """두 xywh box의 IoU(겹친 넓이 ÷ 합친 넓이, 1이면 완전히 같음)를 계산한다."""
    inter_w = max(0, min(box_a[0] + box_a[2], box_b[0] + box_b[2]) - max(box_a[0], box_b[0]))
    inter_h = max(0, min(box_a[1] + box_a[3], box_b[1] + box_b[3]) - max(box_a[1], box_b[1]))
    inter = inter_w * inter_h
    return inter / (box_a[2] * box_a[3] + box_b[2] * box_b[3] - inter)


def crop_square(image, box, pad=0.1, size=224):
    """box 중심에서 긴 변보다 pad만큼 큰 정사각형을 잘라 size×size로 줄인다. 알약 모양이 늘어나지 않는다."""
    x, y, w, h = box
    side = max(w, h) * (1 + pad)
    cx, cy = x + w / 2, y + h / 2
    return image.crop((int(cx - side / 2), int(cy - side / 2), int(cx + side / 2), int(cy + side / 2))).resize((size, size))


def pill_label(ann):
    """box 위에 적을 K-코드와 각인(메타데이터 print_front/print_back)을 만든다."""
    marks = [ann['image'][key] for key in ('print_front', 'print_back') if ann['image'].get(key)]
    return f"K-{ann['category_id']:06d} ({'/'.join(marks) or '각인 없음'})"


print(f"레포 최상위  : {PROJECT_ROOT}")
print(f"그래프 폰트  : {plt.rcParams['font.family'][0]}")
print(f"그림 저장    : {FIG_DIR}")

- 대회 원본 라벨과 00의 결과 파일을 읽는다.
    - `clean_records`: 00 2-5의 정제 목록(`00_raewoo_label_fixes.csv`)을 원본 라벨에 적용한 대회 train 라벨이다. (이미지 232장, 객체 771개)
        - 그려 넣은 box(`bbox_before`가 빈 줄)의 알약 메타데이터는 같은 클래스의 다른 annotation에서 가져온다. 00 2-5와 같은 방법이다.
    - `review_rows`: 00 5-2에서 test 사진을 보고 적은 기록표이다. train에 없는 알약 한 개가 한 줄이다.
- AI Hub 원본 JSON(`aihub_raw/train_annotations/`, 라벨 zip 8개를 그대로 푼 폴더)을 하나도 거르지 않고 읽는다. (`aihub_jsons`, JSON 파일 하나가 한 줄)
    - 읽은 값은 고치지 않는다. 읽지 못하는 JSON도 오류 메시지와 함께 남긴다. 무엇을 뺄지는 1-1부터 분석하며 정한다.
    - zip을 풀면 JSON이 어느 zip에서 나왔는지 남지 않으므로, 라벨 zip의 파일 목록만 읽어 조합 폴더 이름으로 찾는다. 조합 폴더 하나는 zip 하나에만 들어 있다.
    - macOS가 zip에 넣은 부가 파일(`._`로 시작하는 파일, `.DS_Store`)은 `script/extract_aihub.py`가 풀지 않아서 이 폴더에 없다.

In [ ]:
train_records = load_annotations(TRAIN_ANN_DIR)
train_files = sorted(os.listdir(TRAIN_IMG_DIR))
test_files = sorted(os.listdir(TEST_IMG_DIR), key=lambda name: int(name.split('.')[0]))
train_classes = sorted({a['category_id'] for r in train_records.values() for a in r['annotations']})
train_combos = sorted({combo_name(fn) for fn in train_files})
assert set(train_records) == set(train_files), '라벨이 있는 이미지와 이미지 파일 목록이 다르다'

# 00 2-5의 정제 목록을 원본 라벨에 적용한다
IMAGE_FIELDS = ['back_color', 'camera_la', 'camera_lo', 'drug_S', 'drug_dir', 'file_name', 'height', 'id', 'imgfile',
                'light_color', 'size', 'width']  # 메타데이터 중 같은 이미지의 알약끼리 값이 같은 필드 (나머지는 알약 필드, 00 2-5)
with open(EDA_DIR / '00_raewoo_label_fixes.csv', encoding='utf-8') as f:
    train_fixes = list(csv.DictReader(f))
class_example = {}  # K-코드 → 대회 train에서 그 클래스의 annotation 하나 (그려 넣은 box의 메타데이터용)
for fn in train_files:
    for ann in train_records[fn]['annotations']:
        class_example.setdefault(ann['category_id'], ann)
clean_records = {}
for fn, record in sorted(train_records.items()):
    by_code = {a['category_id']: a for a in record['annotations']}
    anns = [dict(a) for a in record['annotations']]
    for fix in (row for row in train_fixes if row['file_name'] == fn):
        code, box = int(fix['category_id']), json.loads(fix['bbox_after'])
        if not fix['bbox_before']:  # 그려 넣은 box
            example = class_example[code]
            anns.append({**example, 'id': None, 'image_id': record['image']['id'], 'json_path': None, 'bbox': box,
                         'image': {**example['image'], **{key: record['image'][key] for key in IMAGE_FIELDS}}})
            continue
        # 원본 annotation 순서로 찾는다 (라벨을 서로 바꾸면 category_id가 바뀌므로 바뀐 값으로 찾지 않는다)
        ann = anns[next(i for i, a in enumerate(record['annotations']) if a['category_id'] == code)]
        ann['bbox'] = box
        if fix['category_id_after']:  # 라벨이 서로 바뀐 경우: 클래스와 알약 메타데이터를 상대 라벨의 것으로 바꾼다
            other = by_code[int(fix['category_id_after'])]
            ann['category_id'], ann['image'] = other['category_id'], other['image']
    for ann in anns:
        ann['area'] = ann['bbox'][2] * ann['bbox'][3]
    clean_records[fn] = {**record, 'annotations': anns}
clean_object_count = collections.Counter(a['category_id'] for r in clean_records.values() for a in r['annotations'])
assert (len(clean_records), sum(clean_object_count.values()), len(clean_object_count)) == (232, 771, 56), \
    '00 2-5의 정제 결과(이미지 232장, 객체 771개, 클래스 56개)와 다르다. 00_raewoo_label_fixes.csv가 바뀌었는지 확인한다.'

with open(EDA_DIR / '00_raewoo_test_review.csv', encoding='utf-8') as f:
    review_rows = list(csv.DictReader(f))

# AI Hub 라벨 zip (script/download_aihub.py의 폴더 규칙: 라벨 zip은 train_annotations/에 있다)
label_zips = sorted(p for p in (AIHUB_DIR / 'train_annotations').glob('*.zip')
                    if unicodedata.normalize('NFC', p.name).endswith('_조합.zip'))
assert len(label_zips) == 8, (f'허용 조합 라벨 zip 8개가 {AIHUB_DIR / "train_annotations"}에 있어야 한다. 지금 {len(label_zips)}개\n'
                              '받는 명령: uv run python script/download_aihub.py 66065 66067 66068 66069 66070 66071 66072 66243')
# zip을 풀면 JSON이 어느 zip에서 나왔는지 남지 않으므로, zip의 파일 목록만 읽어 조합 폴더 이름으로 찾는다
label_zip_of_combo = {}  # 조합 폴더 이름 → 라벨 zip
zip_json_paths = set()  # zip 안 JSON 경로 (macOS 부가 파일 ._는 extract_aihub.py가 풀지 않으므로 뺀다)
for path in label_zips:
    check_aihub_file(path)  # 금지 파일이면 여기서 오류가 난다
    zip_name = unicodedata.normalize('NFC', path.name)
    with zipfile.ZipFile(path) as zf:
        for name in (unicodedata.normalize('NFC', n) for n in zf.namelist()):
            combo_folder = name.split('/')[0]
            assert label_zip_of_combo.setdefault(combo_folder, zip_name) == zip_name, f'{combo_folder} 폴더가 두 zip에 있다'
            if name.endswith('.json') and not os.path.basename(name).startswith('._'):
                zip_json_paths.add(name)
raw_paths = sorted(AIHUB_RAW_ANN_DIR.rglob('*.json'))
assert {unicodedata.normalize('NFC', p.relative_to(AIHUB_RAW_ANN_DIR).as_posix()) for p in raw_paths} == zip_json_paths, \
    (f'{AIHUB_RAW_ANN_DIR}의 JSON이 라벨 zip과 다르다. 다시 푸는 명령: '
     'uv run python script/extract_aihub.py data/external/aihub/train_annotations/*.zip')
aihub_jsons = []  # 원본 JSON 파일 하나가 한 줄. 아무것도 거르지 않고 읽은 그대로 둔다
for path in raw_paths:
    combo_folder, pill_folder = (unicodedata.normalize('NFC', p.name) for p in (path.parent.parent, path.parent))
    try:
        data, error = json.loads(path.read_text(encoding='utf-8')), None
    except json.JSONDecodeError as e:
        data, error = None, str(e)
    aihub_jsons.append({
        'path': path, 'zip': label_zip_of_combo[combo_folder], 'combo_folder': combo_folder, 'pill_folder': pill_folder,
        # JSON 쪽 이미지 파일명: JSON 안의 file_name이 아니라 JSON 파일 이름에서 ' copy'를 떼고 만든다
        'file_name': unicodedata.normalize('NFC', path.name)[:-len('.json')].split(' copy')[0] + '.png',
        'data': data, 'error': error,
    })
aihub_png = sorted(p.name for p in AIHUB_IMG_DIR.glob('*.png'))
aihub_index_files = [fn for fn in aihub_png if fn.endswith('_index.png')]  # 90° 사진에 K-코드와 제품명을 적은 참고용 사진 (1-1)
aihub_files = [fn for fn in aihub_png if not fn.endswith('_index.png')]  # 촬영 사진

print(f'대회 train {len(train_files)}장 (정제 라벨 객체 {sum(clean_object_count.values())}개), test {len(test_files)}장, '
      f'test 기록표 {len(review_rows)}줄')
print(f"AI Hub 라벨 zip {len(label_zips)}개: 원본 JSON {len(aihub_jsons):,}개 (읽지 못한 JSON {sum(j['error'] is not None for j in aihub_jsons)}개), "
      f'받은 PNG {len(aihub_png):,}장 (그중 _index.png {len(aihub_index_files)}장)')

In [ ]:
print(len(review_rows))

- 모든 이미지(대회 train 232장, test 842장, AI Hub)를 한 번씩 읽어, 뒤 분석에 쓸 값을 계산해 둔다.
    - 파일 내용 해시(MD5): 1-6에서 내용이 완전히 같은 파일을 찾는다.
    - 크기(너비, 높이)와 읽기 성공 여부: 1-1에서 해상도와 깨진 파일을 확인한다.
    - 채널별 평균 밝기(R, G, B): 1-4에서 촬영 조건을 비교한다.
    - 32×40 회색 축소 그림: 1-6에서 다시 저장되거나 조금 달라진 같은 사진을 찾는다.
- 이미지가 많아 처음 실행은 몇 분 걸린다. 결과는 `runs/eda_raewoo/cache/image_stats.npz`에 저장하고, 파일 목록이 같으면 다시 계산하지 않는다.

In [ ]:
THUMB_W, THUMB_H = 32, 40  # 축소 그림 크기 (원본 976×1280과 같은 비율)


def image_stats(path):
    """이미지 하나의 MD5, 크기, 채널 평균 밝기, 회색 축소 그림을 계산한다. 읽지 못하면 크기를 (0, 0)으로 둔다."""
    data = path.read_bytes()
    bgr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        return hashlib.md5(data).hexdigest(), (0, 0), np.zeros(3), np.zeros((THUMB_H, THUMB_W), np.uint8)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    thumb = cv2.resize(gray, (THUMB_W, THUMB_H), interpolation=cv2.INTER_AREA)
    return hashlib.md5(data).hexdigest(), (bgr.shape[1], bgr.shape[0]), bgr.reshape(-1, 3).mean(axis=0)[::-1], thumb


stat_paths = ([('train', TRAIN_IMG_DIR / fn) for fn in train_files] + [('test', TEST_IMG_DIR / fn) for fn in test_files]
              + [('aihub', AIHUB_IMG_DIR / fn) for fn in aihub_png])
stat_keys = np.array([f'{group}/{path.name}' for group, path in stat_paths])
cache_path = CACHE_DIR / 'image_stats.npz'
cache = np.load(cache_path) if cache_path.exists() else None
if cache is not None and np.array_equal(cache['keys'], stat_keys):
    stats = {name: cache[name] for name in cache.files}
else:
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:  # OpenCV는 이미지를 읽는 동안 다른 스레드를 막지 않는다
        results = list(pool.map(image_stats, [path for _, path in stat_paths]))
    stats = {'keys': stat_keys, 'md5': np.array([r[0] for r in results]), 'size': np.array([r[1] for r in results]),
             'mean_rgb': np.array([r[2] for r in results]), 'thumb': np.array([r[3] for r in results])}
    np.savez(cache_path, **stats)
stat_index = {key: i for i, key in enumerate(stats['keys'])}  # 'aihub/<파일명>' → 행 번호
print(f"이미지 {len(stat_keys):,}장의 값을 계산했다. (대회 train {len(train_files)}, test {len(test_files)}, AI Hub {len(aihub_png):,})")

## 1. 데이터셋 훑어보기
- 대회 원본 데이터의 출처는 AI Hub 576 "약품식별 인공지능 개발을 위한 경구약제 이미지 데이터"이다. (DATA.md)
- 대회 규칙은 이 중 `TL_2_조합.zip`, `TS_2_조합.zip` 두 파일만 금지한다. (RULE.md)

### 1-1. 파일 구성과 금지 범위
- `aihubshell -mode l -datasetkey 576`으로 받은 파일 목록을 정리한다. (2026-10-24 조회)
    - TL/VL은 라벨링 데이터(JSON), TS/VS는 원천 데이터(이미지)이다. T는 Training, V는 Validation이다.
    - 용량은 aihubshell이 반올림해 보여 준 값이다.

In [ ]:
AIHUB_FILES = [
    # (구분, 파일, 개수, 파일 하나 용량, filekey, 사용)
    ('Training 조합 라벨', 'TL_1, TL_3~8_조합.zip', 7, '7~8MB', '66065, 66067~66072', '사용 가능'),
    ('Training 조합 라벨', 'TL_2_조합.zip', 1, '7MB', '66066', '**금지**'),
    ('Training 조합 이미지', 'TS_1, TS_3~8_조합.zip', 7, '3GB', '66154, 66156~66161', '사용 가능'),
    ('Training 조합 이미지', 'TS_2_조합.zip', 1, '3GB', '66155', '**금지**'),
    ('Validation 조합 라벨', 'VL_1_조합.zip', 1, '7MB', '66243', '사용 가능'),
    ('Validation 조합 이미지', 'VS_1_조합.zip', 1, '3GB', '66254', '사용 가능'),
    ('Training 단일 라벨', 'TL_1~81_단일.zip', 81, '11~73MB (합계 2.6GB)', '66073~66153', '사용 가능'),
    ('Training 단일 이미지', 'TS_1~81_단일.zip', 81, '13~95GB (합계 3,473GB)', '66162~66242', '사용 가능 (로컬 디스크 부족)'),
    ('Validation 단일 라벨', 'VL_1~10_단일.zip', 10, '15~24MB', '66244~66253', '사용 가능'),
    ('Validation 단일 이미지', 'VS_1~10_단일.zip', 10, '19~31GB (합계 228GB)', '66255~66264', '사용 가능 (로컬 디스크 부족)'),
]
show_table(['구분', '파일', '개수', '용량', 'filekey', '사용'], [list(row) for row in AIHUB_FILES])

**인사이트**
- AI Hub 576 데이터셋은 조합 데이터와 단일 데이터로 나뉜다.
    - 조합 데이터는 사진 한 장에 알약이 3~4개 있다. 대회 사진과 같은 형태이다.
    - 단일 데이터는 이름으로 보아 알약을 하나씩 찍은 사진이다.
- 조합 데이터에서 쓸 수 있는 것은 라벨 zip 8개(TL_1, TL_3~8, VL_1)와 이미지 zip 8개(TS_1, TS_3~8, VS_1)이다.
    - 금지된 파일은 TL_2, TS_2 두 개뿐이다.
    - 이미지 zip은 하나에 약 3GB라 8개를 모두 받아도 약 24GB이다.
- 단일 데이터도 규칙상 쓸 수 있지만, 이미지가 합계 약 3,701GB(Training 3,473GB, Validation 228GB)이다.

**결론**
- 조합 데이터의 허용 zip 8쌍(TL_1·TS_1, TL_3~8·TS_3~8, VL_1·VS_1)만 받아 쓴다.
- TL_2, TS_2는 받지 않고, 코드와 다운로드 스크립트에도 넣지 않는다. (RULE.md)
- 단일 데이터는 쓰지 않는다. 로컬 디스크에 받을 수 없는 크기이다.

- 받은 라벨 zip과 이미지 zip이 서로 맞는지 zip 쌍마다 확인한다. 세는 대상은 아래 세 가지이다.
    - JSON 파일: 라벨 zip에서 나온 파일이다. 알약 하나의 라벨이고, 경로는 `<조합>_json/<K-코드 폴더>/<조합>_0_2_0_2_<각도>_000_200.json`이다. (이름에 ` copy`가 붙은 파일도 있다, 1-2)
    - PNG 파일: 이미지 zip에서 나온 파일이다. 촬영 PNG(`<조합>_0_2_0_2_<각도>_000_200.png`)와, 조합마다 하나씩 있는 `<조합>_index.png` 두 종류가 있다.
    - 이미지 파일명: 사진 한 장의 이름(`<조합>_0_2_0_2_<각도>_000_200.png`)이다.
        - JSON 쪽 이미지 파일명: JSON 파일 이름에서 ` copy`를 떼고 `.json`을 `.png`로 바꾼 이름이다. 사진 한 장에 알약이 3~4개라 JSON 3~4개가 같은 이미지 파일명을 가리킨다.
        - PNG 쪽 이미지 파일명: 촬영 PNG의 파일 이름이다.
- 두 쪽을 이미지 파일명으로 맞대어 "JSON과 촬영 PNG가 둘 다 있음", "JSON만 있음"(그 이름의 촬영 PNG가 없음), "촬영 PNG만 있음"(그 이름을 가리키는 JSON이 없음)으로 나눈다.
    - 라벨 zip(TL_n, VL_1)의 JSON 쪽 이미지 파일명은 같은 번호의 이미지 zip(TS_n, VS_1)에 촬영 PNG로 있어야 한다.
    - JSON이 어느 zip에서 나왔는지는 라벨 zip의 파일 목록에서 조합 폴더 이름으로 찾는다. (0번)
    - PNG가 어느 zip에서 나왔는지는 `script/extract_aihub.py`가 푼 뒤 남긴 `train_images_sources.csv`로 본다. (이미지 zip은 풀고 지웠다)
- PNG 파일마다 실제 크기가 JSON의 width, height(976×1280)와 같은지, 읽지 못하는 파일이 있는지 본다. (0번에서 계산한 값)
- `_index.png`는 가리키는 JSON이 없으므로 따로 센다.
- 같은 번호의 zip 쌍에서 짝이 맞지 않는 이미지 파일명은 두 번째 표에 한 줄씩 적는다.
    - JSON만 있음, 촬영 PNG만 있음: 받은 zip 8쌍 전체에서 짝이 없는 이미지 파일명이다.
    - 다른 번호의 zip 쌍: JSON과 촬영 PNG가 둘 다 있지만 번호가 다른 zip 쌍에 들어 있는 이미지 파일명이다. 첫 번째 표에서는 두 쌍에서 JSON만 있음과 촬영 PNG만 있음으로 한 번씩 세인다.

In [ ]:
with open(AIHUB_SOURCES_CSV, encoding='utf-8') as f:
    image_zip_of = {row['file_name']: row['image_zip'] for row in csv.DictReader(f)}  # PNG 파일명 → 나온 이미지 zip
label_zip_of = {j['file_name']: j['zip'] for j in aihub_jsons}  # JSON 쪽 이미지 파일명 → 라벨 zip
PAIR = {f'{kind}L_{n}_조합.zip': f'{kind}S_{n}_조합.zip' for kind, n in [('T', 1), *[('T', n) for n in range(3, 9)], ('V', 1)]}
aihub_sizes = {fn: tuple(stats['size'][stat_index[f'aihub/{fn}']]) for fn in aihub_png}

rows = []
for label_zip, image_zip in PAIR.items():
    labeled = {fn for fn, z in label_zip_of.items() if z == label_zip}
    images = {fn for fn, z in image_zip_of.items() if z == image_zip and not fn.endswith('_index.png')}
    index = {fn for fn, z in image_zip_of.items() if z == image_zip and fn.endswith('_index.png')}
    on_disk = (images | index) & set(aihub_sizes)
    rows.append([f'{label_zip} / {image_zip}', f'{len(labeled):,}', f'{len(images):,}', len(index), len(labeled - images), len(images - labeled),
                 sum(aihub_sizes[fn] == (0, 0) for fn in on_disk),
                 sum(aihub_sizes[fn] not in ((0, 0), (IMG_W, IMG_H)) for fn in on_disk)])
rows.append(['**합계**', *[f'**{sum(int(str(r[i]).replace(",", "")) for r in rows):,}**' for i in range(1, 8)]])
show_table(['라벨 zip / 이미지 zip', 'JSON 쪽 이미지 파일명', '촬영 PNG', '_index.png', 'JSON만 있음', '촬영 PNG만 있음', '읽지 못한 PNG',
            f'{IMG_W}×{IMG_H}이 아닌 PNG'], rows)

In [ ]:
# 위 표에서 짝이 맞지 않는 이미지 파일명을 한 줄씩 적는다
# JSON과 촬영 PNG가 번호가 다른 zip 쌍에 있으면 위 표에서는 두 쌍에서 JSON만 있음과 촬영 PNG만 있음으로 한 번씩 세인다
photo_names = {fn for fn in image_zip_of if not fn.endswith('_index.png')}
rows = []
for fn in sorted(set(label_zip_of) | photo_names):
    label_zip, image_zip = label_zip_of.get(fn), image_zip_of.get(fn)
    if label_zip and image_zip and PAIR[label_zip] == image_zip:
        continue  # 같은 번호의 zip 쌍에 JSON과 촬영 PNG가 둘 다 있다
    if not image_zip:
        kind = 'JSON만 있음'
    elif not label_zip:
        kind = '촬영 PNG만 있음'
    else:
        kind = '다른 번호의 zip 쌍'
    rows.append([f'`{fn}`', label_zip or '-', image_zip or '-', kind])
show_table(['이미지 파일명', '라벨 zip', '이미지 zip', '구분'], rows)

**인사이트**
- zip파일 이름의 '조합'은 한 이미지에 3~4개의 알약 조합이 들어있다는 의미다.
- TL_{N}은 TS_{N}이랑 짝이다. 
    - TL: json이 들어있음, 즉 annotation. 
    - TS: .png가 들어있음. 즉, 이미지 데이터.

- TL_1의 JSON 쪽 이미지 파일명(`<조합>_0_2_0_2_<각도>_000_200.png`)은 1503개다. 이에 정확히 대응되려면 TS_1의 촬영 PNG도 1503개여야 한다. 
    - 그러면 TS_1에만 있는 2조합, 촬영 PNG 6장의 JSON은 어디에 있을까?
    - 그건 알 수 없다. 아마도 TL_2에 있을 것으로 추정된다. 
- TS_3에 존재하는 K-004378-005002-005094-019552_0_2_0_2_70_000_200.png 촬영 PNG는 TL_3이 아니라 TL_4에 그에 대응되는 JSON이 있었다. 이게 3,4,5,6,7까지 이어진다.
    - 아마도 앞 TS_{N-1}의 마지막 2조합 촬영 PNG에 대응되는 JSON이 TL_{N}에 존재하는 것, 즉 하나씩 밀린 것으로 보인다. 
- 하지만 TS_7의 촬영 PNG만 있는 2조합, 총 6장은 대응되는 JSON을 찾을 수 없다. 

**결론**
- '다른 번호의 zip 쌍'의 이미지 파일명은 JSON과 촬영 PNG가 둘 다 있으므로 그대로 사용한다.
- 하지만 TL_3_조합.zip의 JSON만 있는 이미지 파일명 6개(JSON 24개)는 촬영 PNG가 없기 때문에 사용할 수 없다. 1차 정제에서 뺀다. 
- TS_1_조합.zip, TS_7_조합.zip의 촬영 PNG만 있는 이미지 파일명 12개는, 같은 알약을 공유하는 다른 JSON에서 정보를 가져오고, bbox는 2절에서 새로 채워서 사용한다.

- `_index.png`가 무엇인지, 같은 조합의 90° 촬영 사진과 나란히 놓고 본다.
    - 왼쪽은 90° 촬영 사진, 가운데는 `_index.png`, 오른쪽은 `_index.png`에서 알약 하나 옆을 확대한 것이다.
    - 두 사진의 픽셀 차이(R, G, B 차이의 평균)도 제목에 적는다.

In [ ]:
def example_box(photo):
    """그 사진의 첫 JSON(알약 폴더 순서, copy 파일은 뺌)에서 (bbox, K-코드)를 꺼낸다. 읽을 수 없거나 bbox가 숫자 4개가 아니면 None이다."""
    j = next((j for j in aihub_jsons if j['file_name'] == photo and ' copy' not in j['path'].name), None)
    if j is None or j['data'] is None or not j['data']['annotations']:
        return None
    bbox = j['data']['annotations'][0].get('bbox')
    if not (isinstance(bbox, list) and len(bbox) == 4 and all(isinstance(v, (int, float)) for v in bbox)):
        return None
    return bbox, j['data']['images'][0]['drug_N']


photo_set = set(aihub_files)
candidates = ((fn, fn.replace('_index.png', '_0_2_0_2_90_000_200.png')) for fn in sorted(aihub_index_files))
index_example, photo_example, (bbox, drug_code) = next((index_fn, photo, found) for index_fn, photo in candidates
                                                       if photo in photo_set and (found := example_box(photo)))
index_image, photo_image = Image.open(AIHUB_IMG_DIR / index_example).convert('RGB'), Image.open(AIHUB_IMG_DIR / photo_example).convert('RGB')
pixel_diff = np.abs(np.asarray(index_image, float) - np.asarray(photo_image, float)).mean()
x, y, w, h = bbox
zoom = (max(0, x - 40), max(0, y - 40), min(IMG_W, x + w + 260), min(IMG_H, y + h + 60))
fig, axes = plt.subplots(1, 3, figsize=(20, 8), layout='constrained', gridspec_kw={'width_ratios': [1, 1, 1.3]})
axes[0].imshow(photo_image)
axes[0].set_title(f'90° 촬영 사진\n{photo_example}', fontsize=10)
axes[1].imshow(index_image)
axes[1].set_title(f'_index.png (90° 사진과 픽셀 차이 평균 {pixel_diff:.2f})\n{index_example}', fontsize=10)
axes[2].imshow(index_image.crop(zoom))
axes[2].set_title(f'_index.png 확대 ({drug_code} 주변)', fontsize=10)
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('이미지 zip의 _index.png와 같은 조합의 90° 촬영 사진', fontsize=14)
save_fig(fig, '1-1b_index_png')
plt.show()

**인사이트**
- 한 조합당 3 각도 촬영분이 있고, _index.png에는 그 사진에 대응되는 정답 라벨이 표시되어있다. 

**결론**
- _index.png는 학습에 사용하지 않는다. 

- 1차 정제: 어떤 전처리로도 복구할 수 없는 파일을 먼저 뺀다. 사진을 보지 않고 파일의 짝만으로 정한다.
    - JSON만 있는 이미지 파일명(1-1의 두 번째 표)의 JSON은 뺀다. 그 이름의 촬영 PNG가 없어서 학습에 쓸 수 없다.
    - `_index.png`는 뺀다. 정답 K-코드와 제품명이 그려진 참고 PNG라 학습 사진으로 쓸 수 없다. (위 그림)
- 아래는 남기고 뒤에서 판단한다.
    - 촬영 PNG만 있는 이미지 파일명: 2절에서 같은 알약의 라벨로 box를 그려 넣을 수 있다.
    - 다른 번호의 zip 쌍: JSON과 촬영 PNG가 둘 다 있다.
    - 읽지 못하는 JSON, `-오류` 폴더, copy 파일, bbox 오류: 그 이름의 촬영 PNG가 있어서, 사진으로 고칠 수 있는지 2절에서 본다. (1-2, 2-1, 2-2)
- 1차 정제 전후의 수를 비교한다. 항목마다 무엇을 세는지 `정의` 열에 적는다.
    - `변화` 열은 1차 정제 후의 수에서 전의 수를 뺀 값이다. 괄호나 `/`로 나눈 값은 자리마다 따로 뺀다.
    - JSON 쪽 이미지 파일명 = JSON과 촬영 PNG가 둘 다 있는 이미지 파일명 + JSON만 있는 이미지 파일명
    - 촬영 PNG = JSON과 촬영 PNG가 둘 다 있는 이미지 파일명 + 촬영 PNG만 있는 이미지 파일명

In [ ]:
aihub_jsons_1st = [j for j in aihub_jsons if j['file_name'] in set(aihub_files)]  # 1차 정제 후 JSON: 촬영 PNG가 있는 JSON만 남긴다
aihub_photos_1st = aihub_files  # 1차 정제 후 PNG: 촬영 PNG만 남긴다 (_index.png를 뺀다)

CLEAN_ITEMS = {  # 항목 → (정의, 값을 적는 형식)
    'JSON 파일': ('라벨 zip에서 나온 파일. 알약 하나의 라벨', '{}'),
    'JSON 쪽 이미지 파일명': ('JSON 파일 이름에서 ` copy`를 떼고 .png로 바꾼 이름의 종류 수', '{}'),
    'PNG 파일 (촬영 PNG / _index.png)': ('이미지 zip에서 나온 파일', '{} ({} / {})'),
    'JSON과 촬영 PNG가 둘 다 있는 이미지 파일명': ('JSON 쪽과 PNG 쪽에 모두 있는 이름', '{}'),
    'JSON만 있는 이미지 파일명': ('JSON은 있는데 그 이름의 촬영 PNG가 없는 이름', '{} (JSON {}개)'),
    '촬영 PNG만 있는 이미지 파일명': ('촬영 PNG는 있는데 그 이름을 가리키는 JSON이 없는 이름', '{}'),
    '조합 수 (JSON 쪽 / PNG 쪽 / 둘 다)': ('이미지 파일명의 `<조합>` 부분의 종류 수', '{} / {} / {}'),
    '클래스 수': ('JSON이 든 K-코드 폴더 이름(앞 8자)의 종류 수. `-오류` 폴더도 앞 8자가 K-코드이다', '{}'),
}


def clean_counts(jsons, pngs):
    """JSON 목록과 PNG 파일 이름 목록으로 CLEAN_ITEMS의 값을 센다. 항목마다 형식의 {} 수만큼 숫자가 있다."""
    json_files = {j['file_name'] for j in jsons}
    photos = {fn for fn in pngs if not fn.endswith('_index.png')}
    json_only = json_files - photos
    json_combos, photo_combos = {combo_name(fn) for fn in json_files}, {combo_name(fn) for fn in photos}
    return {
        'JSON 파일': (len(jsons),),
        'JSON 쪽 이미지 파일명': (len(json_files),),
        'PNG 파일 (촬영 PNG / _index.png)': (len(pngs), len(photos), len(pngs) - len(photos)),
        'JSON과 촬영 PNG가 둘 다 있는 이미지 파일명': (len(json_files & photos),),
        'JSON만 있는 이미지 파일명': (len(json_only), sum(j['file_name'] in json_only for j in jsons)),
        '촬영 PNG만 있는 이미지 파일명': (len(photos - json_files),),
        '조합 수 (JSON 쪽 / PNG 쪽 / 둘 다)': (len(json_combos), len(photo_combos), len(json_combos & photo_combos)),
        '클래스 수': (len({j['pill_folder'][:8] for j in jsons}),),
    }


def signed(n):
    """변화량을 부호와 함께 적는다. 0은 부호 없이 적는다."""
    return f'{n:+,}' if n else '0'


before, after = clean_counts(aihub_jsons, aihub_png), clean_counts(aihub_jsons_1st, aihub_photos_1st)
class_before = collections.Counter(j['pill_folder'][:8] for j in aihub_jsons)
class_after = collections.Counter(j['pill_folder'][:8] for j in aihub_jsons_1st)
reduced = [f'{code} {class_before[code]:,}→{class_after[code]:,}' for code in sorted(class_before) if class_after[code] != class_before[code]]
rows = [[item, definition, form.format(*(f'{n:,}' for n in before[item])), form.format(*(f'{n:,}' for n in after[item])),
         form.format(*(signed(a - b) for a, b in zip(after[item], before[item])))]
        for item, (definition, form) in CLEAN_ITEMS.items()]
rows.append(['JSON이 줄어든 클래스', '1차 정제로 JSON 수가 바뀐 K-코드 (전→후)', '-', f'{len(reduced)}개: ' + ', '.join(reduced), '-'])
show_table(['항목', '정의', '1차 정제 전', '1차 정제 후', '변화 (후 - 전)'], rows)

**인사이트**
- 촬영 PNG가 없는 JSON 24개를 뺐다. 이 JSON이 가리키는 이미지 파일명은 아래 6개이다.
    * K-003544-012247-016548-029345_0_2_0_2_70_000_200.png
    * K-003544-012247-016548-029345_0_2_0_2_75_000_200.png
    * K-003544-012247-016548-029345_0_2_0_2_90_000_200.png
    * K-003544-012247-016548-029451_0_2_0_2_70_000_200.png
    * K-003544-012247-016548-029451_0_2_0_2_75_000_200.png
    * K-003544-012247-016548-029451_0_2_0_2_90_000_200.png
- 이는 조합 2개, 이미지 파일명 6개를 안 쓰는 셈이다.

**결론**
- 1차 정제한 JSON 45,910개와 촬영 PNG 12,009장으로 1-2부터 분석한다.
- JSON만 있는 이미지 파일명 6개(조합 2개, JSON 24개)는 쓰지 않는다. 촬영 PNG가 없어서 고칠 방법이 없다.
- 촬영 PNG만 있는 이미지 12장은 남긴다. 2-2에서 사진의 알약을 찾아 box를 그려 넣는다.

### 1-2. 라벨 구조 비교
- 1-1에서 1차 정제한 JSON을 Kaggle 형식으로 바꾸기 전 그대로 두고, Kaggle train 라벨과 같은 항목을 비교한다.
    - dl_idx는 알약 식별자 필드이고, drug_N은 K-코드(`K-016548`)이다.
- JSON 키 구성은 최상위, images, annotations, categories 각각의 키 이름을 순서대로 모아 Kaggle과 같은지 센다.
- `비고` 열에는 Kaggle과 다른 예외와 그 이유를 적는다.

In [ ]:
def key_structure(data):
    """JSON의 최상위, images, annotations, categories에 있는 키 이름을 순서대로 모은다."""
    return tuple(tuple(part) for part in (data, data['images'][0], data['annotations'][0], data['categories'][0]))


def json_name(j):
    """표에 적을 JSON 경로(`<조합 폴더>/<K-코드 폴더>/<파일 이름>`)를 만든다."""
    return f"`{j['combo_folder']}/{j['pill_folder']}/{unicodedata.normalize('NFC', j['path'].name)}`"


def broken_line(path):
    """읽지 못한 JSON에서 오류가 난 줄 번호와 그 줄의 내용을 찾는다."""
    text = path.read_text(encoding='utf-8')
    try:
        json.loads(text)
    except json.JSONDecodeError as e:
        return e.lineno, text.splitlines()[e.lineno - 1].strip()


def key_difference(data):
    """JSON 키 구성에서 Kaggle과 다른 곳을 문장으로 적는다."""
    notes = []
    for part, keys, kaggle_keys in zip(['최상위', 'images', 'annotations', 'categories'], key_structure(data), kaggle_structure):
        missing, extra = [k for k in kaggle_keys if k not in keys], [k for k in keys if k not in kaggle_keys]
        if missing:
            notes.append(f"{part}에 {', '.join(missing)} 키가 없다")
        if extra:
            notes.append(f"{part}에 {', '.join(extra)} 키가 더 있다")
        if not missing and not extra and keys != kaggle_keys:
            notes.append(f'{part}의 키 순서가 다르다')
    return ', '.join(notes)


def top_values(counter):
    """가장 많은 값 두 개를 '값 (개수)' 문자열로 만든다."""
    return ', '.join(f'{v} ({n})' for v, n in counter.most_common(2))


kaggle_json_paths = sorted(TRAIN_ANN_DIR.rglob('*.json'))
kaggle_structures = collections.Counter(key_structure(json.loads(p.read_text(encoding='utf-8'))) for p in kaggle_json_paths)
kaggle_structure = kaggle_structures.most_common(1)[0][0]
kaggle_anns = [ann for r in train_records.values() for ann in r['annotations']]

# 1차 정제 후 JSON(1-1)을 센다. JSON 내용으로 세는 항목은 읽을 수 있는 JSON만 센다
broken = [j for j in aihub_jsons_1st if j['error']]
valid = [j for j in aihub_jsons_1st if not j['error']]
images = [j['data']['images'][0] for j in valid]
outside_error = [j for j in valid if not j['pill_folder'].endswith('-오류')]
key_mismatch = [j for j in valid if key_structure(j['data']) != kaggle_structure]
aihub_ids = {
    'image': collections.Counter(image['id'] for image in images),
    'category': collections.Counter(a.get('category_id') for j in valid for a in j['data']['annotations']),
    'annotation': collections.Counter(a.get('id') for j in valid for a in j['data']['annotations']),
    'category_name': collections.Counter(j['data']['categories'][0]['name'] for j in valid),
}
dl_idx_other = collections.defaultdict(list)  # drug_N → dl_idx가 K-코드 - 1이 아닌 JSON의 images
for image in images:
    if int(image['dl_idx']) != int(image['drug_N'][2:]) - 1:
        dl_idx_other[image['drug_N']].append(image)
error_folder_jsons = [j for j in aihub_jsons_1st if j['pill_folder'].endswith('-오류')]
error_folders = sorted({f"{j['combo_folder']}/{j['pill_folder']}" for j in error_folder_jsons})
copy_jsons = [j for j in aihub_jsons_1st if ' copy' in j['path'].name]
copy_kinds = collections.defaultdict(list)  # 파일 이름에 붙은 ' copy', ' copy 2' → 그 JSON
for j in copy_jsons:
    name = unicodedata.normalize('NFC', j['path'].name)
    copy_kinds[name[name.index(' copy'):-len('.json')]].append(j)

broken_note = '; '.join(f'{json_name(j)}의 {line_no}번째 줄 `{line}`에 값이 없어 JSON으로 읽을 수 없다'
                        for j in broken for line_no, line in [broken_line(j['path'])])
dl_idx_note = f'나머지 {sum(len(v) for v in dl_idx_other.values())}개는 ' + '; '.join(
    f"{code}({imgs[0]['dl_name']})의 JSON {len(imgs)}개이다. dl_idx가 모두 {'/'.join(sorted({str(i['dl_idx']) for i in imgs}))}이고, "
    f"dl_mapping_code는 {'/'.join(sorted({i['dl_mapping_code'] for i in imgs}))}이다" for code, imgs in sorted(dl_idx_other.items()))
rows = [
    ['읽을 수 없는 JSON', '0', len(broken),
     f'{broken_note}. JSON 내용으로 세는 아래 행(폴더 이름 = drug_N, 키 구성, id, name, dl_idx)은 이 {len(broken)}개를 뺀 {len(valid)}개로 센다'],
    ['폴더 구조', '`<조합>_json/K-<코드>/<이미지명>.json`', '같음',
     f'예외 2가지: K-코드 뒤에 `-오류`가 붙은 폴더 {len(error_folders)}개 (`-오류` 폴더 JSON 행), '
     f'파일 이름에 ` copy`가 붙은 JSON {len(copy_jsons)}개 (copy 중복 파일 행). `-오류` 폴더 외의 JSON은 K-코드 폴더 이름 = drug_N이다 '
     f"({sum(j['pill_folder'] == j['data']['images'][0]['drug_N'] for j in outside_error)}/{len(outside_error)})"],
    ['JSON 키 구성 (최상위, images, annotations, categories)',
     f"한 가지 {kaggle_structures[kaggle_structure]}/{len(kaggle_json_paths)} (키 {', '.join(str(len(p)) for p in kaggle_structure)}개)",
     f'Kaggle과 같음 {len(valid) - len(key_mismatch)}/{len(valid)}',
     '; '.join(f"다른 {len(key_mismatch)}개: {json_name(j)}의 {key_difference(j['data'])}" for j in key_mismatch)],
    ['JSON 파일 수', len(kaggle_json_paths), len(aihub_jsons_1st), f'1-1의 1차 정제 후 수이다. 읽을 수 없는 JSON {len(broken)}개도 센다'],
    ['image id', f'이미지마다 다름 ({len({r["image"]["id"] for r in train_records.values()})}종)', top_values(aihub_ids['image']), ''],
    ['category_id', f'K-코드 숫자 ({len(train_classes)}종)', top_values(aihub_ids['category']), ''],
    ['annotation id', f'annotation마다 다름 ({len({a["id"] for a in kaggle_anns})}종)', top_values(aihub_ids['annotation']), ''],
    ['categories의 name', '제품명 (dl_name)', top_values(aihub_ids['category_name']), ''],
    ['dl_idx = K-코드', f"{sum(int(a['image']['dl_idx']) == a['category_id'] for a in kaggle_anns)}/{len(kaggle_anns)}",
     f"{sum(int(i['dl_idx']) == int(i['drug_N'][2:]) for i in images)}/{len(valid)}", ''],
    ['dl_idx = K-코드 - 1', '0', f"{len(valid) - sum(len(v) for v in dl_idx_other.values())}/{len(valid)}", dl_idx_note],
    ['-오류 폴더 JSON', '0', len(error_folder_jsons),
     f"폴더 {len(error_folders)}개: {', '.join(f'`{folder}`' for folder in error_folders)}. "
     f"읽을 수 없는 JSON {sum(j['error'] is not None for j in error_folder_jsons)}개도 이 폴더에 있다"],
    ['copy.json, copy 2.json파일', '0', len(copy_jsons),
     ', '.join(f'`<이미지명>{kind}.json` {len(js)}개' for kind, js in sorted(copy_kinds.items()))
     + '. 예: ' + ', '.join(json_name(js[0]) for kind, js in sorted(copy_kinds.items()))],
]
show_table(['항목', 'Kaggle train', 'AI Hub 1차 정제 후', '비고'], rows)

**인사이트**
- 폴더 구조는 Kaggle과 AI Hub 원본이 동일하다. 
- JSON 키 구성도 Kaggle과 같다. 
    - 하지만 `K-006835-020004-038972-053384_json/K-020004/K-006835-020004-038972-053384_0_2_0_2_90_000_200.json` 이 파일에는 `iscrowd` 키가 하나 빠져있다. 
    - 이 파일의 annotation 값은 {'area': 106845, 'bbox': [215, 652, 255, 419], 'category_id': 1, 'ignore': 0, 'segmentation': [], 'id': 1, 'image_id': 1}이다. bbox는 정상으로 들어 있다.
    - 같은 알약을 70°, 75°에서 찍은 JSON에는 iscrowd가 있다. 이 파일만 키가 빠진 것이라, AI Hub가 라벨을 만들 때 생긴 누락으로 보인다.
- JSON 파일 수는 1차 정제 후 45910개고, 읽을 수 없는 JSON파일이 1개 있다. `K-012247-016551-033009-044199_json/K-016551-오류/K-012247-016551-033009-044199_0_2_0_2_90_000_200.json`: `"area": ,` 로 되어있어서 읽을 수 없다. 
- `image_id`, `category_id`, `annotation.id`, `categories.name` 키는 Kaggle에서는 각각 고유한 값 또는 유의미한 식별자였는데, AI Hub 원본에는 이게 다 1이고, name은 Drug으로 퉁쳐져있다. 
- AIHub 원본 데이터에는 `dl_idx = K-코드 - 1`이다. 
    - 단, 450개는 예외다. K-053384(디카테오엘씨정 10mg/병) 한 알약의 json 450개는 모두 `dl_idx = 130376`, `dl_mapping_code = K-053384`, `item_seq = 202009337`로 똑같이 들어있다. 그래서 뭐가 잘못된 건 아니다. 
    - Kaggle과 형식을 맞추기 위해서 1-3에서 `dl_idx`와 `category_id`는 `drug_N`의 숫자로 덮어씌어진다. 이 때 K-053384의 dl_idx도 53384로 바뀐다. 
- AIHub 원본 데이터에는 -오류 이름이 붙은 폴더가 총 3개 있고, 여기에는 각 3개의 .json이 있다. 총 9개의 .json이 오류로 표시되어있다. 
    - 이 중 하나는 `"area": ,`으로, 잘못된 json 형식이고, 
    - 나머지 8개 모두 bbox가 빈 리스트 []이다. AI Hub는 bbox가 없는 라벨을 -오류 폴더로 따로 모아 둔 것으로 보인다.
    - JSON 안의 drug_N은 K-023319, K-016551로 정상이다.
    - 즉, bbox와 라벨 오류만 전처리로 잡으면 되는 것으로 보인다. 

**결론**
- 라벨 오류의 유형이 Kaggle 제공 데이터에서보다 훨씬 다양한 것으로 보인다. 아래 내용은 모두 전처리에서 다뤄야 할 내용이다. 
    1. '-오류'로 표시된 K-코드 폴더가 따로 있고, 거기에는 3개 각도로 찍힌 이미지들에 bbox가 비어있다. 그리고 여기서 `"area": ,`로 json 오류도 있다. 
    2. 키 구성 다름: `iscrowd` 키가 빠져있는 게 하나 있다. 이건 1-3에서 Kaggle 형식으로 바꿀 때 0으로 채운다. 
    3. copy.json: `<이미지명> copy.json` 또는 `<이미지명> copy 2.json` 이름으로 copy 21개, copy 2 3개, 총 24개 있다. 
        - 예시: `data/external/aihub_raw/train_annotations/K-001900-003544-016548-029345_json/K-016548/K-001900-003544-016548-029345_0_2_0_2_75_000_200 copy.json`
        - 예시: `data/external/aihub_raw/train_annotations/K-001900-003544-004543-016551_json/K-016551/K-001900-003544-004543-016551_0_2_0_2_75_000_200 copy.json`
        - (이미지,알약) 묶음으로는 21 묶음 있다. 
        - 일단 지금으로서는 `K-코드/.json 세 개` 이 구조는 모두 통일된다. 그리고 copy든 copy 2든 이름만 이상한 거고, 세 파일이 세 각도에 하나씩 맞는 거로 확인됐다. 자세한 건 2절 라벨 품질 점검에서 시각화와 함께 자세하게 살펴본다. 

- 같은 알약 K-001900(보령부스파정 5mg)을 같은 각도(70°)에서 찍은 Kaggle JSON 하나와 AI Hub 원본 JSON 하나를 키마다 나란히 놓는다.
    - Kaggle 쪽은 `K-001900-016548-019607-029451_..._70_...` 이미지에 있는 K-001900의 JSON이다. (00 0번의 `train_records` 예시)
    - AI Hub 쪽은 같은 K-코드 폴더의 70° JSON 중 경로 순서로 첫 번째이다. copy 파일과 `-오류` 폴더는 뺀다.
    - 같은 알약이므로 알약 정보는 같아야 하고, 사진이 달라서 생기는 값과 대회 측이 고친 값만 달라야 한다.
- 값이 다른 키는 `비교` 열에 **다름**으로 표시한다. 긴 값은 앞 30자만 보인다.

In [ ]:
EXAMPLE_CODE = 'K-001900'
kaggle_json_path = (TRAIN_ANN_DIR / 'K-001900-016548-019607-029451_json' / EXAMPLE_CODE
                    / 'K-001900-016548-019607-029451_0_2_0_2_70_000_200.json')
kaggle_json = json.loads(kaggle_json_path.read_text(encoding='utf-8'))
aihub_example = next(j for j in aihub_jsons_1st if j['pill_folder'] == EXAMPLE_CODE
                     and '_70_' in j['path'].name and ' copy' not in j['path'].name)
aihub_json_path, aihub_json = aihub_example['path'], aihub_example['data']


def short(value, n=30):
    """표에 넣을 값을 앞 n자만 남긴다. 빈 문자열은 (빈 값)으로 쓴다."""
    text = str(value).replace('|', '/')
    if text == '':
        return '(빈 값)'
    return text if len(text) <= n else text[:n] + '…'


def compare_json(left, right, left_name, right_name):
    """두 JSON의 키를 하나씩 나란히 놓은 표 행과, 값이 다른 키 이름을 만든다."""
    rows = [['파일', '-', short(left_name, 40), short(right_name, 40), '-'],
            ['최상위', '(키 목록)', ', '.join(left), ', '.join(right), '같음' if list(left) == list(right) else '**다름**']]
    diff = []
    for part in ['images', 'annotations', 'categories']:
        for key, left_value in left[part][0].items():
            right_value = right[part][0].get(key, '(키 없음)')
            if left_value != right_value:
                diff.append(f'{part}.{key}')
            rows.append([part, key, short(left_value), short(right_value), '같음' if left_value == right_value else '**다름**'])
    n_keys = sum(len(left[part][0]) for part in ['images', 'annotations', 'categories'])
    rows.append([f'**값이 다른 키 {len(diff)}/{n_keys}개**', '', '', '', ''])
    return rows, diff


rows, raw_diff = compare_json(kaggle_json, aihub_json, kaggle_json_path.name, aihub_json_path.name)
show_table(['위치', '키', 'Kaggle train', 'AI Hub 원본', '비교'], rows)

**인사이트**
- 같은 알약 K-001900의 JSON 두 개는 61개 키 중 11개만 값이 다르다.
- 다른 키는 세 종류이다.
    - 사진이 달라서 다른 값: file_name, imgfile, bbox, area
    - 이미지와 annotation마다 매기는 번호: images.id, annotations.id, annotations.image_id (AI Hub 원본은 모두 1)
    - 클래스 표기: category_id와 categories.id(AI Hub 1, Kaggle 1900), categories.name(AI Hub Drug, Kaggle 제품명), dl_idx(AI Hub 1899, Kaggle 1900)
- 제품명, 각인, 모양, 색, 크기 같은 알약 정보 필드는 모두 같다.
- 대회 측은 AI Hub 원본에서 번호와 클래스 표기만 바꾼 것으로 보인다.

**결론**
- 알약 정보 필드와 bbox, area는 원본 값을 그대로 둔다. bbox와 area의 오류는 2절에서 고친다.
- 번호 3개, 클래스 표기 4개, iscrowd를 1-3에서 Kaggle 규칙대로 바꾼다.
    - 클래스는 dl_idx가 아니라 drug_N(K-코드)으로 정한다. 원본 dl_idx를 쓰면 모든 클래스가 하나씩 어긋난다. (위 표의 dl_idx = K-코드 - 1)

### 1-3. Kaggle 형식으로 바꾼 뒤 다시 비교
- 1차 정제 후 JSON(1-1)에서 Kaggle과 형식이 다른 키 8개만 Kaggle 규칙대로 바꾼다. 값이 맞는지 따로 분석하지 않아도 규칙으로 정해지는 키이다.
    - `images.id`: 이미지 파일명 순서로 1501부터 준다. 같은 이미지의 JSON끼리 같다.
    - `annotations.id`: 같은 이미지 순서로, 이미지 안에서는 JSON 경로 순서로 5692부터 준다.
        - 대회 train과 test가 image id 1~1500을 나눠 쓰고(test는 파일명 숫자), 대회 train의 annotation id 최댓값이 5691이다.
    - `annotations.image_id`: 그 이미지의 `images.id`를 넣는다.
    - `annotations.category_id`, `categories.id`, `images.dl_idx`: drug_N의 숫자로 바꾼다. `dl_idx`는 Kaggle처럼 문자열로 둔다.
    - `categories.name`: 제품명(`images.dl_name`)으로 바꾼다.
    - `annotations.iscrowd`: Kaggle처럼 0으로 바꾼다. 키가 없던 JSON 1개(1-2)에는 Kaggle 키 순서에 맞춰 넣는다.
- bbox, area를 비롯한 나머지 키는 원본 값 그대로 둔다.
    - bbox 오류, area 재계산, copy 중복 파일, `-오류` 폴더는 2절에서 다룬다.
- 읽을 수 없는 JSON 1개(1-2)는 키를 바꿀 수 없다. 빼지 않고 내용이 없는 채로 두고 2절에서 다룬다.
- 바꾼 결과는 파일로 저장하지 않고 메모리에만 둔다. 파일 저장은 2절에서 정제를 마친 뒤에 한다.
- 같은 검사를 Kaggle train, AI Hub 바꾸기 전, AI Hub 바꾼 뒤에 똑같이 적용해 비교한다.
    - `그 밖의 키` 행은 키 8개를 뺀 나머지 값이 바꾸기 전과 같은지 센다.

In [ ]:
KAGGLE_ANN_KEYS = kaggle_structure[2]  # Kaggle annotations의 키 순서 (1-2)
# images.id는 이미지 파일명 순서로 1501부터, annotations.id는 같은 순서(이미지 안에서는 JSON 경로 순서)로 5692부터 준다
image_ids = {fn: image_id for image_id, fn in enumerate(sorted({j['file_name'] for j in valid}), start=1501)}
annotation_ids = {j['path']: ann_id for ann_id, j in enumerate(sorted(valid, key=lambda j: (j['file_name'], j['path'])), start=5692)}


def to_kaggle_keys(j):
    """JSON 하나에서 Kaggle과 형식이 다른 키 8개만 바꾼 새 내용을 만든다. 나머지 키는 원본 값 그대로 둔다."""
    image, ann, category = j['data']['images'][0], j['data']['annotations'][0], j['data']['categories'][0]
    code = int(image['drug_N'][2:])  # K-코드 숫자 (K-001900 → 1900)
    new_image = {**image, 'dl_idx': str(code), 'id': image_ids[j['file_name']]}
    new_ann = {**ann, 'iscrowd': 0, 'category_id': code, 'id': annotation_ids[j['path']], 'image_id': new_image['id']}
    new_ann = {**{key: new_ann[key] for key in KAGGLE_ANN_KEYS if key in new_ann}, **new_ann}  # iscrowd가 없던 JSON도 Kaggle 키 순서로 맞춘다
    new_category = {**category, 'id': code, 'name': image['dl_name']}
    return {**j['data'], 'images': [new_image], 'annotations': [new_ann], 'categories': [new_category]}


# 1차 정제 후 JSON 하나가 한 줄 (aihub_jsons_1st와 같은 구조). 읽을 수 없는 JSON은 바꿀 수 없어 data가 None인 채로 둔다
aihub_kaggle_jsons = [{**j, 'data': None if j['data'] is None else to_kaggle_keys(j)} for j in aihub_jsons_1st]
aihub_kaggle = {}  # 이미지 파일명 → 대회 train_records(load_annotations)와 같은 구조. 1-4부터 이미지 단위로 볼 때 쓴다
for j in aihub_kaggle_jsons:
    if j['data'] is not None:
        image = j['data']['images'][0]
        record = aihub_kaggle.setdefault(j['file_name'], {'image': image, 'annotations': []})
        record['annotations'].append({**j['data']['annotations'][0], 'image': image, 'json_path': str(j['path'])})

TARGET_KEYS = [('images', 'id'), ('images', 'dl_idx'), ('annotations', 'id'), ('annotations', 'image_id'),
               ('annotations', 'category_id'), ('annotations', 'iscrowd'), ('categories', 'id'), ('categories', 'name')]


def field(d, part, key):
    """JSON 내용 d의 part(images, annotations, categories) 첫 항목에서 key 값을 꺼낸다. 키가 없으면 '(키 없음)'이다."""
    return d[part][0].get(key, '(키 없음)')


def other_values(d):
    """키 8개를 뺀 나머지 값을 모은다. 바꾸기 전과 바꾼 뒤가 같은지 볼 때 쓴다."""
    values = {('최상위', 'type'): d['type']}
    for part in ['images', 'annotations', 'categories']:
        values.update({(part, key): v for key, v in d[part][0].items() if (part, key) not in TARGET_KEYS})
    return values


def format_checks(docs):
    """Kaggle, 바꾸기 전, 바꾼 뒤에 똑같이 적용하는 검사. 항목 → 표에 적을 값."""
    n = len(docs)
    files = {field(d, 'images', 'file_name') for d in docs}
    ids, ann_ids = [field(d, 'images', 'id') for d in docs], [field(d, 'annotations', 'id') for d in docs]
    codes = [int(field(d, 'images', 'drug_N')[2:]) for d in docs]
    category_ids = [field(d, 'annotations', 'category_id') for d in docs]
    iscrowd = collections.Counter(field(d, 'annotations', 'iscrowd') for d in docs)
    return {
        'JSON 파일 (이미지)': f'{n}개 ({len(files)}장)',
        'images.id': f'{min(ids)}~{max(ids)} ({len(set(ids))}종 / 이미지 {len(files)}장)',
        'images.dl_idx': f"= K-코드 {sum(int(field(d, 'images', 'dl_idx')) == c for d, c in zip(docs, codes))}/{n}",
        'annotations.id': f'{min(ann_ids)}~{max(ann_ids)} ({len(set(ann_ids))}종 / annotation {n}개)',
        'annotations.image_id': f"= images.id {sum(field(d, 'annotations', 'image_id') == field(d, 'images', 'id') for d in docs)}/{n}",
        'annotations.category_id': f'= drug_N 숫자 {sum(a == c for a, c in zip(category_ids, codes))}/{n} ({len(set(category_ids))}종)',
        'annotations.iscrowd': ', '.join(f'{v} ({count})' for v, count in iscrowd.most_common()),
        'categories.id': f"= category_id {sum(field(d, 'categories', 'id') == a for d, a in zip(docs, category_ids))}/{n}",
        'categories.name': f"= dl_name {sum(field(d, 'categories', 'name') == field(d, 'images', 'dl_name') for d in docs)}/{n}",
        'JSON 키 구성': f'Kaggle과 같음 {sum(key_structure(d) == kaggle_structure for d in docs)}/{n}',
    }


kaggle_docs = [json.loads(p.read_text(encoding='utf-8')) for p in kaggle_json_paths]
before_docs = [j['data'] for j in valid]  # 1-2의 읽을 수 있는 JSON
after_docs = [j['data'] for j in aihub_kaggle_jsons if j['data'] is not None]  # before_docs와 같은 순서
checks = [format_checks(docs) for docs in (kaggle_docs, before_docs, after_docs)]
rows = [[item] + [c[item] for c in checks] for item in checks[0]]
rows[0][3] += f'. 읽을 수 없는 JSON {len(broken)}개는 바꾸지 못했다'
rows.append(['그 밖의 키 (bbox, area 등)', '-', '-',
             f'원본과 같음 {sum(other_values(b) == other_values(a) for b, a in zip(before_docs, after_docs))}/{len(after_docs)}'])
show_table(['항목', 'Kaggle train', 'AI Hub 바꾸기 전', 'AI Hub 바꾼 뒤'], rows)

**인사이트**
- 여기서 AIhub에서 Kaggle형식으로 바꾼 라벨은 annotation 하나마다 JSON 하나를 쓰는 형식이다. 학습에 쓸 수 없는 JSON도 일단 빼지 않고 형식이 다른 키 8개만 바꿨다.
    - 1차 정제 후 읽을 수 있는 JSON 45909개를 모두 바꿨다. 읽을 수 없는 JSON 1개는 바꾸지 못하고 그대로 두었다.
    - 바꾼 키 8개는 모두 Kaggle 규칙과 같아졌다(45909/45909). `iscrowd`가 빠졌던 1개도 채워서 JSON 키 구성이 모두 Kaggle과 같다.
    - bbox, area 등 나머지 값은 원본과 같다(45909/45909). `-오류` 폴더 JSON, copy 파일, bbox 오류도 그대로 남아 있다.
- 2절에서 bbox, area 오류를 고친 뒤 파일로 저장한다. 키 형식은 여기서 이미 맞췄다. 

**결론**
- AI Hub 라벨은 1-3의 규칙(키 8개)으로 바꿔 쓴다. 이후 분석은 바꾼 라벨(`aihub_kaggle`)로 한다.
    - images.id는 1501부터, annotations.id는 5692부터 매겨 대회 train·test의 번호와 겹치지 않는다.
    - category_id는 drug_N으로 만들어 116종이 되고, 대회 train과 같은 알약은 같은 번호로 묶인다.
- 학습 데이터를 만드는 코드도 같은 규칙으로 바꾼 뒤, 2-5처럼 정제 목록을 적용한다.

- 1-2의 같은 AI Hub JSON을 바꾸기 전과 바꾼 뒤로 나누어 같은 Kaggle JSON 옆에 놓는다.
    - `바꿈` 열은 바꾸기 전과 바꾼 뒤의 값이 다른 키이다.
    - `Kaggle과 비교` 열은 바꾼 뒤의 값이 Kaggle과 같은지이다.
    - 형식을 다 맞췄다면, Kaggle과 다른 키는 사진이 달라서 생기는 값(파일명, bbox, area)과 이미지마다 새로 매기는 id만 남아야 한다.

In [ ]:
converted_example = next(j['data'] for j in aihub_kaggle_jsons if j['path'] == aihub_json_path)  # 1-2 예시 JSON을 바꾼 뒤
rows = [['파일', '-', short(kaggle_json_path.name, 40), short(aihub_json_path.name, 40), short(aihub_json_path.name, 40), '-', '-'],
        ['최상위', '(키 목록)', ', '.join(kaggle_json), ', '.join(aihub_json), ', '.join(converted_example), '',
         '같음' if list(kaggle_json) == list(converted_example) else '**다름**']]
for part in ['images', 'annotations', 'categories']:
    for key, kaggle_value in kaggle_json[part][0].items():
        before, after = aihub_json[part][0].get(key, '(키 없음)'), converted_example[part][0].get(key, '(키 없음)')
        rows.append([part, key, short(kaggle_value), short(before), short(after), '**바꿈**' if before != after else '',
                     '같음' if after == kaggle_value else '**다름**'])
n_keys = len(rows) - 2
n_changed, n_diff = sum(row[5] == '**바꿈**' for row in rows[2:]), sum(row[6] == '**다름**' for row in rows[2:])
rows.append([f'**바꾼 키 {n_changed}/{n_keys}개, 바꾼 뒤에도 Kaggle과 다른 키 {n_diff}/{n_keys}개**', '', '', '', '', '', ''])
show_table(['위치', '키', 'Kaggle train', 'AI Hub 바꾸기 전', 'AI Hub 바꾼 뒤', '바꿈', 'Kaggle과 비교'], rows)

**인사이트**
- 바꾼 뒤 Kaggle과 다른 키는 61개 중 7개이다.
    - 사진이 달라서 다른 값: file_name, imgfile, bbox, area
    - 새로 매긴 번호: images.id(2848), annotations.id(11080), annotations.image_id(2848)
- 클래스 표기 4개(dl_idx, category_id, categories.id, categories.name)는 Kaggle과 같아졌다.
    - category_id가 1900이 되어, 같은 알약이 대회 train과 같은 번호로 묶인다.
- 이 JSON은 iscrowd가 원래 0이라 값이 바뀐 키는 7개이다.

**결론**
- 1-3의 변환 규칙을 확정한다. 남은 차이는 사진마다 다른 값과 새로 매긴 번호뿐이다.
- 6-1에서 대회 train과 합친 라벨로 mAP를 계산해, 같은 알약이 같은 번호로 묶이는지 다시 확인한다.

### 1-4. 촬영 조건 비교
- 메타데이터의 촬영 조건 필드를 이미지 단위로 세어 대회 train과 AI Hub를 비교한다.
    - back_color(배경), light_color(조명), camera_la(카메라 위도, 70°/75°/90°), camera_lo(카메라 경도), drug_dir(알약 앞면/뒷면), drug_S(알약 상태), size, width×height
    - 알약마다 JSON이 따로 있으므로, 이미지 하나의 값은 그 이미지 JSON들의 값을 모은 것이다. 알약마다 값이 다르면 `섞임`으로 센다.

In [ ]:
CONDITION_FIELDS = ['back_color', 'light_color', 'camera_la', 'camera_lo', 'drug_dir', 'drug_S', 'size']


def condition_counts(records):
    """촬영 조건 필드마다 이미지 단위 값의 개수를 센다. 같은 이미지의 알약끼리 값이 다르면 '섞임'으로 센다."""
    counts = {field: collections.Counter() for field in CONDITION_FIELDS + ['width×height']}
    for record in records.values():
        for field in CONDITION_FIELDS:
            values = {a['image'][field] for a in record['annotations']} or {record['image'][field]}
            counts[field][values.pop() if len(values) == 1 else '섞임'] += 1
        counts['width×height'][f"{record['image']['width']}×{record['image']['height']}"] += 1
    return counts


def count_text(counter, total):
    """값별 이미지 수를 많은 순서로 '값 n장 (비율)'로 적는다. 셋이 넘으면 나머지는 합친다."""
    items = counter.most_common()
    text = [f'{v} {n:,}장 ({n / total:.1%})' for v, n in items[:3]]
    if len(items) > 3:
        text.append(f'그 외 {sum(n for _, n in items[3:]):,}장')
    return ', '.join(text)


train_conditions = condition_counts(train_records)
aihub_conditions = condition_counts(aihub_kaggle)
show_table(['필드', f'대회 train ({len(train_records)}장)', f'AI Hub ({len(aihub_kaggle):,}장)'],
           [[field, count_text(train_conditions[field], len(train_records)), count_text(aihub_conditions[field], len(aihub_kaggle))]
            for field in CONDITION_FIELDS + ['width×height']])

**인사이트**
- 촬영 조건 필드는 대회 train과 AI Hub가 거의 같다.
    - 배경(연회색), 조명(주백색), 카메라 경도(0), size(200), 해상도(976×1280)가 같다.
- 촬영 각도는 AI Hub가 70°, 75°, 90° 3,999장씩 정확히 3분의 1이다. 대회 train은 90° 36.6%, 75° 32.3%, 70° 31.0%이다.
    - AI Hub는 조합마다 세 각도를 모두 찍었다. (4-2)
- AI Hub의 "검은색 배경" 1장은 메타데이터만 틀렸다.
    - `K-012247-016551-024850-029451_0_0_0_2_90_000_200.png`이다. 사진을 열어 보면 같은 조합의 75° 사진과 같은 연회색 배경이다.
    - 파일명의 배경 자리도 다른 사진(`_0_2_`)과 달리 `_0_0_`이다.
- 알약 뒷면 사진은 AI Hub 6장(0.1%), 대회 train 4장(1.7%)이다.

**결론**
- 촬영 조건이 같으므로 AI Hub 사진을 대회 사진과 그대로 섞어 학습한다.
- "검은색 배경" 1장도 그대로 쓴다. 사진은 다른 사진과 같은 조건이다.

- 이미지마다 R, G, B 채널의 평균 밝기(0~255)를 대회 train, test, AI Hub로 나눠 분포를 비교한다. (0번에서 계산한 값)
    - 세 묶음의 이미지 수가 크게 달라서(232장, 842장, 약 12,000장) 넓이가 1인 밀도로 그린다.
    - 00 1-2에서 대회 train과 test의 채널 평균 차이는 3~7%였다.

In [ ]:
groups = {'대회 train': 'train', 'test': 'test', 'AI Hub': 'aihub'}
group_means = {name: stats['mean_rgb'][[k.startswith(prefix + '/') and not k.endswith('_index.png') for k in stats['keys']]]
               for name, prefix in groups.items()}
fig, axes = plt.subplots(1, 3, figsize=(20, 5), layout='constrained', sharey=True)
bins = np.linspace(min(m.min() for m in group_means.values()) - 2, max(m.max() for m in group_means.values()) + 2, 60)
for c, (ax, channel) in enumerate(zip(axes, ['R', 'G', 'B'])):
    for (name, means), color in zip(group_means.items(), ['tab:blue', 'tab:orange', 'tab:green']):
        values = means[:, c]
        ax.hist(values, bins=bins, density=True, histtype='step', linewidth=2, color=color,
                label=f'{name} {len(values):,}장 (중앙값 {np.median(values):.1f})')
        ax.axvline(np.median(values), color=color, linestyle='--', linewidth=1)
    ax.set_xlabel(f'{channel} 채널 평균 밝기 (0~255)')
    ax.set_title(f'{channel} 채널')
    ax.legend(fontsize=9)
axes[0].set_ylabel('밀도 (넓이 합 = 1)')
fig.suptitle('이미지별 채널 평균 밝기 분포 (점선 = 중앙값, 분포가 겹칠수록 촬영 조건이 비슷함)', fontsize=14)
save_fig(fig, '1-4_channel_means')
plt.show()

**인사이트**
- AI Hub의 채널별 중앙값(R 111.2, G 126.8, B 146.5)은 대회 train과 test 사이에 있다.
    - test가 가장 어둡다: R 106.4, G 115.4 (대회 train 114.4, 129.4)
    - B 채널은 세 묶음이 거의 같다: 144.5~147.4
- AI Hub 분포(초록)는 대회 train(파랑)과 test(주황)의 범위를 모두 덮는다. test에만 있는 밝기 구간이 없다.
- G 채널은 세 묶음 모두 봉우리가 둘이다. 촬영 회차마다 조명 밝기가 달랐던 것으로 보인다. (아래 표의 zip별 차이)

**결론**
- AI Hub를 섞어도 학습 데이터의 밝기 범위가 test를 덮으므로, 밝기를 맞추는 전처리는 하지 않는다.
- 색·밝기 증강은 00에서 정한 ultralytics 기본값(`hsv_h=0.015`, `hsv_s=0.7`, `hsv_v=0.4`)을 그대로 쓴다.

- 묶음별(대회 train, test, AI Hub 이미지 zip 8개) 채널 평균 밝기의 중앙값을 표로 비교한다.
    - 밝기는 R, G, B 평균의 합을 3으로 나눈 값(0~255)이다. 대회 test와의 차이를 비율로 함께 적는다.

In [ ]:
def brightness_row(name, means):
    med = np.median(means, axis=0)
    gray = means.mean(axis=1)
    return [name, f'{len(means):,}장', *[f'{v:.1f}' for v in med], f'{np.median(gray):.1f}',
            f'{(np.median(gray) / np.median(test_gray) - 1):+.1%}', f'{np.percentile(gray, 10):.1f} ~ {np.percentile(gray, 90):.1f}']


test_gray = group_means['test'].mean(axis=1)
rows = [brightness_row('대회 train', group_means['대회 train']), brightness_row('test', group_means['test'])]
for image_zip in PAIR.values():
    idx = [stat_index[f'aihub/{fn}'] for fn, z in sorted(image_zip_of.items()) if z == image_zip and not fn.endswith('_index.png')]
    if idx:
        rows.append(brightness_row(f'AI Hub {image_zip}', stats['mean_rgb'][idx]))
show_table(['묶음', '이미지', 'R 중앙값', 'G 중앙값', 'B 중앙값', '밝기 중앙값', 'test 대비', '밝기 10%~90% 범위'], rows)

**인사이트**
- train 뿐만 아니라 AI Hub 데이터들에 비해서 test 데이터가 어두운 편이다. 

**결론**
- 이미지 자체가 까맣게 칠해지거나 하얗게 번진 건 없으므로 문제 없다고 판단한다. 

### 1-5. 이미지 육안 확인
- AI Hub 이미지 zip 8개에서 1장씩, 라벨 오류가 없는 이미지를 무작위로 골라 1-3 라벨의 box를 겹쳐 그린다. (seed 고정)
    - 라벨 오류가 없는 이미지: 2절에서 다룰 문제(읽을 수 없는 JSON, `-오류` 폴더, copy 파일, bbox 오류, 라벨 누락)가 하나도 없는 이미지이다. (1-2)
    - box는 원본 bbox 그대로이다. 1-3은 bbox를 바꾸지 않는다.
    - box 위 글자는 K-코드와 메타데이터의 각인이다. 알약에 찍힌 글자와 대조하면 box가 맞는 알약을 감싸는지 알 수 있다.
- box가 알약을 빈틈없이 감싸는지, 대회 사진(00 1-3)과 같은 장면인지 본다.

In [ ]:
jsons_of = collections.defaultdict(list)  # 이미지 파일명 → 그 이미지의 JSON (1-3에서 키를 바꾼 것)
for j in aihub_kaggle_jsons:
    jsons_of[j['file_name']].append(j)


def has_label_problem(fn):
    """2절에서 다룰 라벨 문제(읽을 수 없는 JSON, -오류 폴더, copy 파일, bbox 오류, 라벨 누락)가 있는 이미지인지 본다."""
    js = jsons_of[fn]
    if any(j['data'] is None or j['pill_folder'].endswith('-오류') or ' copy' in j['path'].name for j in js):
        return True
    boxes = [j['data']['annotations'][0]['bbox'] for j in js]
    if any(not isinstance(b, list) or len(b) != 4 or b[2] <= 0 or b[3] <= 0 for b in boxes):
        return True
    return len({j['data']['annotations'][0]['category_id'] for j in js}) < len(pill_codes(fn))


rng = np.random.default_rng(SEED)
look_files = []
for image_zip in PAIR.values():
    pool = sorted(fn for fn, z in image_zip_of.items() if z == image_zip and fn in aihub_kaggle and not has_label_problem(fn))
    if pool:  # 아직 받지 않은 zip은 건너뛴다
        look_files.append(pool[rng.integers(len(pool))])

fig, axes = plt.subplots(2, 4, figsize=(24, 18), layout='constrained')
for ax, fn in zip(axes.flat, look_files):
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for ann in aihub_kaggle[fn]['annotations']:
        x, y, w, h = ann['bbox']
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2.5))
        ax.text(x, y - 8, pill_label(ann), fontsize=9, va='bottom', bbox=dict(facecolor='lime', alpha=0.85, pad=1.5, linewidth=0))
    ax.set_title(f'{image_zip_of[fn]}\n{fn}', fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('AI Hub 이미지 zip마다 무작위 1장 (연두 box = 1-3 라벨의 box, 글자 = K-코드(각인))', fontsize=14)
save_fig(fig, '1-5_aihub_samples')
plt.show()

**인사이트**
- 8장 모두 box가 알약 하나를 빈틈없이 감싼다.
    - box 위 각인이 사진 속 알약의 각인과 맞는다. (예: TS_3의 K-023223 V100, TS_6의 K-011220 Bayer, TS_8의 K-015710 Entelon150)
- 장면은 대회 사진(00 1-3)과 같다. 연회색 바닥에 알약 3~4개가 서로 떨어져 놓여 있다.
- TS_1 사진은 다른 zip보다 밝고, 조명 원 밖의 모서리가 하얗게 보인다. (아래 1-4 표의 TS_1 +27.0%)
- TS_6의 K-031885(81)처럼 파란 알약은 바닥과 색이 비슷하다.

**결론**
- 라벨 오류가 없는 AI Hub box는 대회 box처럼 알약을 감싸므로 그대로 학습에 쓴다. 여백은 3-1에서 수치로 비교한다.
- 라벨 오류가 있는 이미지는 2절에서 고친다.

### 1-6. 누수 점검
- AI Hub 사진 중에 대회 train, test와 같은 사진이 있는지 세 가지 방법으로 찾는다.
    - 파일 이름: 대회 train 이미지, train 조합, 라벨이 누락된 train 8장(00 2-1)의 이름이 AI Hub에 있는지 센다.
    - 파일 내용: MD5 해시(파일 내용이 1바이트라도 다르면 값이 바뀜)가 같은 파일을 찾는다.
    - 축소 그림: 대회 측이 다시 저장해 파일 내용이 달라도 같은 사진이면, 32×40 회색 축소 그림이 거의 같다.
        - 두 축소 그림의 RMS 차이(픽셀 밝기 차이 제곱의 평균에 제곱근, 0~255)를 계산한다. 0이면 똑같고, 같은 사진을 다시 저장하면 1~2 정도이다.
        - 사진마다 가장 가까운 다른 사진을 찾고, RMS 차이가 작은 순서로 아래 그림에서 눈으로 확인한다.
- AI Hub끼리도 다른 조합에 같은 사진이 있는지 본다. 같은 조합의 각도별 사진은 원래 비슷하므로 뺀다.
    - 다른 조합에 같은 사진이 있으면 학습과 검증에 같은 사진이 들어갈 수 있다.
- `_index.png`는 90° 사진에 글자를 적은 것이라(1-1) 모든 비교에서 뺀다.

In [ ]:
def thumbs(prefix):
    """0번에서 계산한 회색 축소 그림을 묶음별로 꺼낸다. _index.png는 뺀다. (파일명 목록, float32 행렬)"""
    idx = [i for i, k in enumerate(stats['keys']) if k.startswith(prefix + '/') and not k.endswith('_index.png')]
    return [stats['keys'][i].split('/', 1)[1] for i in idx], stats['thumb'][idx].reshape(len(idx), -1).astype(np.float32)


def nearest(a, b, groups=None, chunk=2000):
    """a의 축소 그림마다 b에서 RMS 차이가 가장 작은 그림의 번호와 차이를 찾는다. groups를 주면(a와 b가 같을 때) 같은 묶음은 뺀다."""
    b_sq = (b ** 2).sum(axis=1)
    best_idx, best_rms = np.zeros(len(a), int), np.zeros(len(a))
    for start in range(0, len(a), chunk):
        block = a[start:start + chunk]
        d2 = (block ** 2).sum(axis=1)[:, None] + b_sq[None, :] - 2 * block @ b.T  # 제곱 거리 = |a|² + |b|² - 2a·b
        if groups is not None:
            d2[groups[start:start + chunk, None] == groups[None, :]] = np.inf
        best_idx[start:start + chunk] = d2.argmin(axis=1)
        best_rms[start:start + chunk] = np.sqrt(np.maximum(d2.min(axis=1), 0) / a.shape[1])
    return best_idx, best_rms


train_names, train_thumbs = thumbs('train')
test_names, test_thumbs = thumbs('test')
aihub_names, aihub_thumbs = thumbs('aihub')
near = {
    '대회 train': nearest(train_thumbs, aihub_thumbs),
    'test': nearest(test_thumbs, aihub_thumbs),
    'AI Hub 다른 조합 사진': nearest(aihub_thumbs, aihub_thumbs, groups=np.array([combo_name(fn) for fn in aihub_names])),
}
md5_of = collections.defaultdict(list)
for key, digest in zip(stats['keys'], stats['md5']):
    md5_of[digest].append(key)
same_md5 = {group: sum(any(k.startswith('aihub/') for k in keys) and any(k.startswith(group + '/') for k in keys)
                       for keys in md5_of.values()) for group in ['train', 'test']}
aihub_dup_md5 = sum(sum(k.startswith('aihub/') and not k.endswith('_index.png') for k in keys) > 1 for keys in md5_of.values())
missing_train = [fn for fn in train_files if len({a['category_id'] for a in train_records[fn]['annotations']}) < len(pill_codes(fn))]
aihub_photo_names = set(aihub_photos_1st)  # 1차 정제 후 촬영 PNG (1-1)
aihub_combos = {combo_name(fn) for fn in aihub_photo_names}
NEAR_RMS = 2  # 이 값보다 작으면 같은 사진 후보로 센다 (같은 사진에 글자만 적은 _index.png와 90° 사진이 0.6 정도, 1-1)

rows = [
    ['파일 이름: 대회 train 이미지', f'{len(set(train_files) & aihub_photo_names)}/{len(train_files)}'],
    ['파일 이름: 대회 train 조합', f'{len(set(train_combos) & aihub_combos)}/{len(train_combos)}'],
    ['파일 이름: 라벨이 누락된 train 이미지 (00 2-1)', f'{len(set(missing_train) & aihub_photo_names)}/{len(missing_train)}'],
    ['파일 내용(MD5)이 같은 사진: AI Hub와 대회 train', same_md5['train']],
    ['파일 내용(MD5)이 같은 사진: AI Hub와 test', same_md5['test']],
    ['파일 내용(MD5)이 같은 사진: AI Hub끼리', aihub_dup_md5],
]
for name, (_, rms) in near.items():
    rows.append([f'축소 그림: {name}에서 가장 가까운 AI Hub 사진의 RMS 차이',
                 f'최소 {rms.min():.1f}, 중앙값 {np.median(rms):.1f}, {NEAR_RMS} 미만 {(rms < NEAR_RMS).sum()}장'])
show_table(['확인 항목', '결과'], rows)

**인사이트**
- Kaggle 제공 train과 test 각각에 대해서 완전히 겹치는 이미지는 한 장도 없었다. 
- AIHub 데이터 내부에서도 같은 이미지는 없었다. 

**결론**
- 대회 train, test와 같은 사진이 없으므로, 누수 때문에 빼는 AI Hub 사진은 없다.
    - 가장 가까운 사진도 RMS 1.9~2.0이다. 아래 그림처럼 알약 하나만 바꿔 같은 자리에서 찍은 다른 사진이다.
- 대회 train의 이미지와 조합이 AI Hub 허용 zip에 하나도 없다. 대회 데이터는 금지된 TL_2·TS_2에서 나온 것으로 보인다.
    - 대회 train의 라벨 누락 8장(00 2-1)은 AI Hub로 복구할 수 없다.
    - 하지만 이미 라벨을 만들어서 넣었다. ㄱㅊ음
- **RMS 0.4인 AI Hub 쌍은 같은 사진이 다른 조합 이름으로 두 번 들어간 것이다. (아래 그림의 AI Hub끼리 4쌍)**
    - 조합 `K-012247-016548-033009-044199`와 `K-012247-016551-033009-044199`, 조합 `K-004543-016548-023203-024850`과 `K-016548-016551-023203-024850`의 각도별 사진 3쌍씩, 모두 6쌍이다.
    - 2-5 정제 뒤에는 두 사진의 클래스도 같다. (2-2의 `-오류` 폴더, 2-3의 클래스 바꿈)
    - **조합 단위로 나누면 같은 사진이 train과 val에 갈라질 수 있다. 두 조합을 한 묶음으로 나누도록 분할 담당 팀원에게 알린다.**
- RMS 1.2~2.0인 나머지 AI Hub 쌍은 알약 하나만 바꿔 찍은 다른 사진이다. 그대로 쓴다.

- 축소 그림이 가장 가까운 쌍을 묶음마다 4쌍씩, RMS 차이가 작은 순서로 나란히 그린다.
    - 윗줄은 대회 train 또는 test 사진(AI Hub끼리는 앞 사진), 아랫줄은 가장 가까운 AI Hub 사진이다.
- 같은 사진인지(알약 배치와 위치가 같은지), 비슷한 장면의 다른 사진인지 본다.

In [ ]:
pair_sources = [('대회 train', train_names, TRAIN_IMG_DIR), ('test', test_names, TEST_IMG_DIR), ('AI Hub 다른 조합 사진', aihub_names, AIHUB_IMG_DIR)]
fig, axes = plt.subplots(2, 12, figsize=(30, 7.5), layout='constrained')
col = 0
for name, names, folder in pair_sources:
    idx, rms = near[name]
    shown = []
    for i in np.argsort(rms):
        pair = frozenset((names[i], aihub_names[idx[i]]))
        if name == 'AI Hub 다른 조합 사진' and pair in shown:  # 같은 쌍이 순서만 바뀌어 두 번 나오는 것을 뺀다
            continue
        shown.append(pair)
        axes[0, col].imshow(Image.open(folder / names[i]))
        axes[0, col].set_title(f'{name}\n{names[i][:34]}', fontsize=8)
        axes[1, col].imshow(Image.open(AIHUB_IMG_DIR / aihub_names[idx[i]]))
        axes[1, col].set_title(f'AI Hub (RMS 차이 {rms[i]:.1f})\n{aihub_names[idx[i]][:34]}', fontsize=8)
        col += 1
        if len(shown) == 4:
            break
for ax in axes.flat:
    ax.axis('off')
fig.suptitle('축소 그림이 가장 가까운 사진 쌍 (윗줄 = 찾는 사진, 아랫줄 = 가장 가까운 AI Hub 사진, RMS 차이가 작은 순서)', fontsize=14)
save_fig(fig, '1-6_nearest_pairs')
plt.show()

**인사이트**
- 거의 같은 알약 조합인데, 하나의 조합만 다른 경우에 RMS가 낮았다.

**결론**
- **RMS 0.4인 AI Hub 쌍은 같은 사진이 다른 조합 이름으로 두 번 들어간 것이다. (아래 그림의 AI Hub끼리 4쌍)**
    - 조합 `K-012247-016548-033009-044199`와 `K-012247-016551-033009-044199`, 조합 `K-004543-016548-023203-024850`과 `K-016548-016551-023203-024850`의 각도별 사진 3쌍씩, 모두 6쌍이다.
    - 2-5 정제 뒤에는 두 사진의 클래스도 같다. (2-2의 `-오류` 폴더, 2-3의 클래스 바꿈)
    - **조합 단위로 나누면 같은 사진이 train과 val에 갈라질 수 있다. 두 조합을 한 묶음으로 나누도록 분할 담당 팀원에게 알린다.**

## 2. 라벨 품질
- 00 2절과 같은 항목을 1-3에서 키를 바꾼 AI Hub 라벨로 본다.
    - `aihub_kaggle_jsons`: JSON 하나가 한 줄이다. 읽을 수 없는 JSON도 내용 없이 들어 있다.
    - `aihub_kaggle`: 이미지 파일명마다 읽을 수 있는 JSON의 annotation을 모은 것이다. (대회 `train_records`와 같은 구조)
    - 1-3은 bbox를 바꾸지 않았다. copy 파일, `-오류` 폴더, bbox 오류도 그대로 들어 있다.
- 목적: 고치거나 뺄 라벨과 이미지를 정한다.
    - 평가 지표 mAP@[0.75:0.95]는 IoU 0.75 미만인 box를 오답으로 처리한다. 라벨 box가 조금만 틀려도 학습과 평가가 모두 흔들린다.

### 2-1. 라벨 누락과 클래스 불일치
- 1차 정제 후 촬영 PNG(1-1)마다 파일명에 적힌 K-코드와, 그 이미지 파일명을 가리키는 JSON의 category_id를 대조한다.
    - 파일명의 K-코드 하나가 사진 속 알약 하나, 곧 객체(annotation) 하나이다.
    - 쓸 수 있는 bbox는 값이 4개이고 너비와 높이가 0보다 큰 bbox이다. `-오류` 폴더의 JSON과 읽을 수 없는 JSON은 쓸 수 없다.
- 라벨 누락: 파일명에는 있는데 쓸 수 있는 bbox가 없는 알약이다. 빠진 알약마다 그 알약의 JSON을 보고 원인을 나눈다.
    - JSON 없음: 알약 폴더(`<조합>_json/K-<코드>/`)에 이 이미지 파일명의 JSON이 없다. 이유에 따라 셋으로 나눈다.
        - 촬영 PNG만 있는 이미지: 조합의 JSON 폴더가 통째로 없다. (1-1)
        - copy 파일이 있는 알약 폴더: 폴더의 JSON 3개 중 둘 이상이 같은 각도의 이미지 파일명을 가리켜서, 이 각도를 가리키는 JSON이 없다. (1-2, 아래 copy 배정에서 맞는 각도에 다시 배정한다)
        - 그 밖: 위 두 경우가 아닌데 이 각도의 JSON이 없다.
    - bbox 빈 값: bbox가 `[]`이다.
    - 숫자가 붙은 bbox: bbox 값이 4개가 아니다. (예: `[121125, 333, 475]`)
    - `-오류` 폴더: JSON이 `K-<코드>-오류` 폴더에 있다. 이 폴더의 JSON은 bbox도 `[]`이지만 이 원인으로 센다. (1-2)
    - 읽을 수 없는 JSON: JSON 문법이 깨져 읽을 수 없다. `-오류` 폴더에 있지만 이 원인으로 센다. (1-2)
- 클래스 불일치: 이미지 한 장 기준으로 센다.
    - 파일명에 없는 K-코드: 파일명에 없는 K-코드의 JSON이 그 이미지 파일명을 가리킨다.
    - **한 이미지에 같은 K-코드의 box가 파일명에 있는 횟수보다 보다 많음**: 같은 K-코드의 쓸 수 있는 bbox가 파일명에 적힌 수보다 많다.
        - copy 파일이 있는 알약 폴더에서 생긴다. 폴더 단위로는 JSON이 3개로 맞지만, 둘 이상이 같은 각도를 가리켜서 '그 각도의 이미지'에는 box가 남고 다른 각도의 이미지에는 JSON(박스)이 없다.
        - 예: `K-001900-003544-004543-016548_json/K-016548/`의 `_70`, `_75`, `_75 copy` JSON 3개는 사진 3장에 1개, 2개, 0개(90°)로 나뉜다. 75° 사진의 남는 box 1개는 이 행에, 90° 사진의 빠진 box 1개는 "JSON 없음: copy 파일이 있는 알약 폴더" 행에 들어간다.
- 표 읽는 법
    - `객체 수`는 라벨 누락 행에서는 빠진 알약 수이고, 클래스 불일치 행에서는 남는 box 수이다.
    - `확인할 파일 경로`에는 레포 최상위 기준 경로를 한 줄에 하나씩 적는다. 행마다 목록의 처음, 가운데, 끝에서 3개를 고른다.
        - JSON 없음 행은 있어야 할 JSON의 경로이다. 그 경로에 파일이 없다.
        - 한 이미지에 같은 K-코드의 box가 많은 행은 예시 이미지 2장마다 그 K-코드의 JSON을 모두 적는다.

In [ ]:
def usable_bbox(bbox):
    """학습에 쓸 수 있는 bbox인지 본다. 값이 4개이고 너비와 높이가 0보다 커야 한다."""
    return isinstance(bbox, list) and len(bbox) == 4 and bbox[2] > 0 and bbox[3] > 0


def bbox_problem(j):
    """JSON 하나의 bbox를 쓸 수 없는 원인을 돌려준다. 쓸 수 있으면 None이다. 원인이 둘 이상이면 앞의 것으로 센다."""
    if j['data'] is None:
        return '읽을 수 없는 JSON'
    if j['pill_folder'].endswith('-오류'):
        return '-오류 폴더'
    bbox = j['data']['annotations'][0]['bbox']
    if usable_bbox(bbox):
        return None
    if bbox == []:
        return 'bbox 빈 값'
    if not isinstance(bbox, list) or len(bbox) != 4:
        return '숫자가 붙은 bbox'
    return 'bbox 너비·높이 0 이하'


def json_code(j):
    """JSON의 K-코드 숫자. 읽을 수 없는 JSON은 알약 폴더 이름(`K-016551-오류`)에서 읽는다."""
    return int(j['pill_folder'][2:8]) if j['data'] is None else j['data']['annotations'][0]['category_id']


missing_labels = []  # (이미지, 빠진 K-코드, 원인, JSON 경로, bbox). JSON 없음이면 경로와 bbox가 None이다
extra_labels = []  # (이미지, 파일명에 없는 K-코드, JSON 경로)
surplus_labels = []  # (이미지, K-코드, 남는 box 수, 그 K-코드의 쓸 수 있는 JSON 경로)
for fn in aihub_photos_1st:
    named = collections.Counter(pill_codes(fn))
    jsons_by_code = collections.defaultdict(list)
    for j in jsons_of.get(fn, []):  # jsons_of: 1-5에서 만든 이미지 파일명 → 1-3의 JSON
        jsons_by_code[json_code(j)].append(j)
    for code in sorted(set(named) | set(jsons_by_code)):
        js = jsons_by_code[code]
        if code not in named:
            extra_labels += [(fn, code, j['path']) for j in js]
            continue
        usable = [j for j in js if bbox_problem(j) is None]
        unusable = [j for j in js if bbox_problem(j) is not None]
        for k in range(named[code] - len(usable)):  # 쓸 수 없는 JSON의 원인을 먼저 붙이고, JSON이 모자라면 JSON 없음이다
            if k < len(unusable):
                j = unusable[k]
                bbox = None if j['data'] is None else j['data']['annotations'][0]['bbox']
                missing_labels.append((fn, code, bbox_problem(j), j['path'], bbox))
            else:
                missing_labels.append((fn, code, 'JSON 없음', None, None))
        if len(usable) > named[code]:
            surplus_labels.append((fn, code, len(usable) - named[code], [j['path'] for j in usable]))

copy_pill_folders = {(j['combo_folder'], j['pill_folder']) for j in aihub_kaggle_jsons if ' copy' in j['path'].name}
MISSING_GROUPS = ['JSON 없음: 촬영 PNG만 있는 이미지 (1-1)', 'JSON 없음: copy 파일이 있는 알약 폴더 (1-2)', 'JSON 없음: 그 밖',
                  'bbox 빈 값', '숫자가 붙은 bbox', '-오류 폴더', '읽을 수 없는 JSON', 'bbox 너비·높이 0 이하']


def missing_group(m):
    """라벨 누락 하나가 들어갈 표의 행. JSON 없음은 이유에 따라 셋으로 나눈다."""
    fn, code, cause = m[:3]
    if cause != 'JSON 없음':
        return cause
    if not jsons_of.get(fn):
        return MISSING_GROUPS[0]
    if (f'{combo_name(fn)}_json', f'K-{code:06d}') in copy_pill_folders:
        return MISSING_GROUPS[1]
    return MISSING_GROUPS[2]


def missing_path(m):
    """라벨 누락의 JSON 경로. JSON 없음이면 있어야 할 경로이다."""
    fn, code, _, path, _ = m
    return path or AIHUB_RAW_ANN_DIR / f'{combo_name(fn)}_json' / f'K-{code:06d}' / f"{fn[:-len('.png')]}.json"


def repo_path(path):
    """표에 적을 레포 최상위 기준 경로."""
    return f"`{unicodedata.normalize('NFC', path.relative_to(PROJECT_ROOT).as_posix())}`"


def spread(items, n=3):
    """목록의 처음, 가운데, 끝에서 고르게 n개를 고른다."""
    return [items[i] for i in sorted(set(np.linspace(0, len(items) - 1, min(n, len(items))).round().astype(int)))]


rows = []
for group in MISSING_GROUPS:
    items = [m for m in missing_labels if missing_group(m) == group]
    if items:
        rows.append([group, len(items), len({m[0] for m in items}), len({m[1] for m in items}),
                     '<br>'.join(repo_path(missing_path(m)) for m in spread(items))])
missing_files = sorted({m[0] for m in missing_labels})
per_image = collections.Counter(collections.Counter(m[0] for m in missing_labels).values())
rows.append(['**라벨 누락 합계**', f'**{len(missing_labels)}**', f'**{len(missing_files)}장**', f"**{len({m[1] for m in missing_labels})}**", ''])
rows.append(['이미지당 빠진 알약 수', '', ', '.join(f'{k}개 {v}장' for k, v in sorted(per_image.items())), '', ''])
rows.append(['클래스 불일치: 파일명에 없는 K-코드', len(extra_labels), len({e[0] for e in extra_labels}), len({e[1] for e in extra_labels}),
             '<br>'.join(repo_path(e[2]) for e in spread(extra_labels))])
rows.append(['클래스 불일치: 한 이미지에 같은 K-코드의 box가 파일명보다 많음 (copy 파일이 있는 알약 폴더)', sum(s[2] for s in surplus_labels), len({s[0] for s in surplus_labels}),
             len({s[1] for s in surplus_labels}), '<br>'.join(repo_path(p) for s in spread(surplus_labels, n=2) for p in s[3])])
show_table(['원인', '객체 수', '이미지 수', '클래스 수', '확인할 파일 경로'], rows)

**인사이트**
- 라벨 누락은 객체 234개, 이미지 175장(촬영 PNG 12,009장의 1.5%)이다.
    - bbox 빈 값이 110개로 가장 많다. 다음은 숫자가 붙은 bbox 45개, 촬영 PNG만 있는 이미지 42개이다.
    - 빠진 알약은 클래스 85개에 흩어져 있다.
- 이미지당 빠진 알약은 1개인 경우가 147장으로 대부분이다.
- 파일명에 없는 K-코드의 box는 없다. JSON의 drug_N과 파일명의 K-코드는 서로 맞는다.
- 한 이미지에 같은 K-코드의 box가 많은 경우는 23개(18장)이고, 모두 copy 파일이 있는 알약 폴더에서 생긴다.
    - 같은 폴더에서 "JSON 없음" 23개가 함께 생긴다. 남는 box와 빠진 box의 수가 같다.

**결론**
- 라벨이 빠진 알약을 학습에서 빼지 않고 채운다. 빠진 알약이 사진에 있는지는 아래 그림에서 확인한다.
- copy 파일은 지우지 않고, 아래 copy 배정에서 맞는 각도에 다시 배정한다.

- 위 표의 행마다 예시 이미지를 최대 2장씩 그린다. 예시가 없는 행(파일명에 없는 K-코드)은 건너뛴다.
    - 연두 box는 그 이미지 파일명을 가리키는 JSON 중 쓸 수 있는 bbox이다.
    - 주황 box는 한 이미지에 파일명보다 많이 붙은 K-코드의 box이다. box 위에 그 box를 가져온 JSON 이름(각도와 copy 부분)을 함께 적는다.
    - 제목 셋째 줄부터는 라벨 누락 행에서는 빠진 알약의 K-코드와 각인이다. 각인은 같은 클래스의 다른 AI Hub annotation에서 가져온다.
        - 한 이미지에 같은 K-코드의 box가 많은 행에서는 그 K-코드의 box 수와 파일명에 적힌 수이다.
    - 한 이미지에 같은 K-코드의 box가 많은 행은 파일명 순서로 첫 이미지와, 남는 box가 가장 많은 이미지 중 첫 이미지를 그린다.
- 빠진 알약이 사진에 실제로 있는지, 어느 알약인지 본다.
- 주황 box가 사진 속 알약 수보다 많은지, 남는 box가 어느 알약이나 자리에 걸치는지 본다.

In [ ]:
aihub_example = {}  # K-코드 → 바꾼 AI Hub 라벨에서 그 클래스의 annotation 하나 (각인 표시용)
for fn in sorted(aihub_kaggle):
    for ann in aihub_kaggle[fn]['annotations']:
        aihub_example.setdefault(ann['category_id'], ann)


def missing_text(fn):
    """그 이미지에서 빠진 알약의 K-코드와 각인을 적는다. 그림 제목이 옆 그림과 겹치지 않게 두 개씩 줄을 나눈다."""
    labels = [pill_label(aihub_example[m[1]]) if m[1] in aihub_example else f'K-{m[1]:06d}' for m in missing_labels if m[0] == fn]
    return '\n'.join(', '.join(labels[i:i + 2]) for i in range(0, len(labels), 2))


surplus_of = collections.defaultdict(dict)  # 이미지 → {box가 파일명보다 많은 K-코드: (box 수, 파일명에 적힌 수)}
for fn, code, extra, paths in surplus_labels:
    surplus_of[fn][code] = (len(paths), len(paths) - extra)


def json_part(ann):
    """box를 가져온 JSON 이름에서 각도와 copy 부분('75 copy')만 꺼낸다."""
    return Path(ann['json_path']).stem.split('_0_2_0_2_')[1].replace('_000_200', '')


SURPLUS_GROUP = '클래스 불일치: 한 이미지에 같은 K-코드의 box가 파일명보다 많음'
examples = []
for group in MISSING_GROUPS:
    files = sorted({m[0] for m in missing_labels if missing_group(m) == group})
    examples += [(group, fn) for fn in files[:2]]
surplus_files = sorted(surplus_of)
if surplus_files:  # 파일명 순서로 첫 이미지와, 남는 box가 가장 많은 이미지 중 첫 이미지를 그린다
    most = max(surplus_files, key=lambda f: sum(n - named for n, named in surplus_of[f].values()))
    examples += [(SURPLUS_GROUP, fn) for fn in dict.fromkeys([surplus_files[0], most])]
n_cols = 5
fig, axes = plt.subplots(int(np.ceil(len(examples) / n_cols)), n_cols, figsize=(30, 9.5 * np.ceil(len(examples) / n_cols)),
                         layout='constrained', squeeze=False)
for ax, (group, fn) in zip(axes.flat, examples):
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for ann in aihub_kaggle.get(fn, {'annotations': []})['annotations']:
        if not usable_bbox(ann['bbox']):
            continue  # 쓸 수 없는 bbox는 그릴 수 없다 (위 표의 라벨 누락)
        x, y, w, h = ann['bbox']
        surplus = ann['category_id'] in surplus_of.get(fn, {})  # 남는 box가 있는 K-코드는 주황으로 그리고 JSON 이름을 붙인다
        color = 'orange' if surplus else 'lime'
        text = f'{pill_label(ann)} [{json_part(ann)}]' if surplus else pill_label(ann)
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2.5))
        ax.text(x, y - 8, text, fontsize=8, va='bottom', bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0))
    if group == SURPLUS_GROUP:
        detail = 'box가 남는 알약: ' + ', '.join(f'K-{code:06d} box {n}개 (파일명 {named}개)' for code, (n, named) in surplus_of[fn].items())
    else:
        detail = f'bbox 없음: {missing_text(fn)}'
    ax.set_title(f'[{group}]\n{fn}\n{detail}', fontsize=9)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
    if not ax.images:
        ax.axis('off')
fig.suptitle('2-1 표의 행별 예시 (연두 box = 쓸 수 있는 bbox, 주황 box = 한 이미지에 파일명보다 많이 붙은 K-코드의 box와 그 JSON 이름, '
             '제목 셋째 줄부터 = 빠진 알약 또는 box가 남는 알약)', fontsize=14)
save_fig(fig, '2-1_missing_examples')
plt.show()

**인사이트**
- JSON 없음, bbox 빈 값, 숫자가 붙은 bbox 행은 빠진 알약이 사진에 그대로 있다. box만 없다.
    - 예: 촬영 PNG만 있는 `K-001900-010224-016551-029345` 사진에는 알약 4개가 모두 있다.
- `-오류` 폴더와 읽을 수 없는 JSON 행은 파일명의 알약이 사진에 없고, 다른 알약이 box 없이 놓여 있다.
    - `-오류` 폴더의 K-023319(마크/P) 대신 흰 D|CT 알약이 있다.
    - 읽을 수 없는 JSON의 K-016551(DGTH/800) 대신 노란 GB 알약이 있다.
- copy 파일 행은 한 알약에 주황 box 2~3개가 조금씩 어긋나 겹친다. (`75`, `75 copy`, `75 copy 2`)
    - 어긋난 box는 다른 각도 사진의 알약 자리로 보인다. (아래 copy 배정 그림에서 확인)

**결론**
- 사진에 있는 빠진 알약은 사진에서 box를 찾아 그려 넣는다. (2-2)
- `-오류` 폴더와 읽을 수 없는 JSON의 알약은 사진 속 실제 클래스로 붙인다. (2-2의 `-오류` 폴더 그림)
- copy 파일의 box는 맞는 각도의 사진에 하나씩 다시 배정한다. (아래 copy 배정)

- 같은 K-코드의 JSON끼리 알약 정보가 같은지 확인한다.
    - 1-3에서 키를 바꾼 읽을 수 있는 JSON 전체(`aihub_kaggle`)로 본다. copy 파일, `-오류` 폴더, bbox 오류가 있는 JSON도 넣는다.
    - 알약 정보 필드: dl_name(제품명), print_front·print_back(각인), drug_shape(모양), color_class1(색), leng_long·leng_short·thick(크기)
    - 같은 알약이면 사진이 달라도 이 값들이 같아야 한다. 다르면 다른 알약의 JSON이 섞였을 수 있다.
- 대회 train과 같은 클래스는 대회 JSON의 값과도 비교한다.

In [ ]:
PILL_FIELDS = ['dl_name', 'print_front', 'print_back', 'drug_shape', 'color_class1', 'leng_long', 'leng_short', 'thick']
aihub_field_values = collections.defaultdict(lambda: collections.defaultdict(collections.Counter))  # K-코드 → 필드 → 값 개수
for record in aihub_kaggle.values():
    for ann in record['annotations']:
        for field in PILL_FIELDS:
            aihub_field_values[ann['category_id']][field][str(ann['image'].get(field))] += 1
kaggle_field_values = {code: {field: str(ann['image'].get(field)) for field in PILL_FIELDS} for code, ann in class_example.items()}

rows = []
for code in sorted(aihub_field_values):
    for field in PILL_FIELDS:
        values = aihub_field_values[code][field]
        kaggle_value = kaggle_field_values.get(code, {}).get(field)
        if len(values) > 1 or (kaggle_value is not None and kaggle_value not in values):
            rows.append([f'K-{code:06d}', field, ', '.join(f'{v} ({n})' for v, n in values.most_common()), kaggle_value or '-'])
n_classes = len(aihub_field_values)
n_bad = len({r[0] for r in rows})
rows.append([f'**{n_classes}개 클래스 중 값이 하나가 아닌 클래스 {n_bad}개**', '', '', ''])
show_table(['K-코드', '필드', 'AI Hub 값 (annotation 수)', '대회 train 값'], rows)

**인사이트**
- K-코드가 같으면 알약 정보는 같다. 
- 휴 다행...

**결론**
- 같은 K-코드의 JSON은 같은 알약이고 대회 train의 값과도 같으므로, drug_N(K-코드)을 클래스로 그대로 쓴다.
- 새로 그려 넣는 box(촬영 PNG만 있는 이미지 등)의 알약 메타데이터는 같은 클래스의 다른 AI Hub annotation에서 가져온다. (2-5의 `drawn_annotation`)

- 2-1 표의 두 행은 copy 파일이 있는 알약 폴더 21개에서 생긴다. (1-2의 copy 중복 파일 24개)
    - "JSON 없음: copy 파일이 있는 알약 폴더" 23개와 "한 이미지에 같은 K-코드의 box가 파일명보다 많음" 23개이다.
    - 폴더마다 JSON은 3개이지만, 그중 둘 이상이 파일 이름과 JSON 안의 `file_name`, `camera_la`에 같은 각도를 적었다.
    - 1-3은 JSON 파일 이름의 각도대로 사진에 넣었다. 그래서 그 각도의 사진에는 그 K-코드의 box가 2~3개 붙고, 다른 각도의 사진에는 box가 없다.
- 세 JSON의 box를 70°, 75°, 90° 사진에 모두 그려 보고, 사진마다 맞는 JSON을 하나씩 정해 `COPY_ANGLE_FILES`에 적는다.
    - 70° 사진은 75°·90° 사진과 알약 배치가 다르다. 75°와 90° 사진은 배치가 같고 알약 자리만 조금 다르다.
    - 파일 이름의 각도와 맞는 각도는 다를 수 있고, copy가 붙은 쪽이 항상 틀린 것도 아니다. (예: 아래 표 1번 폴더의 `75`는 90° 사진의 box이다)
- 사진마다 그 K-코드의 annotation을 맞는 JSON 하나로 바꿔 `aihub_kaggle`을 고친다.
    - 그대로: 1-3이 넣은 box가 맞는 JSON 하나뿐이다.
    - 남는 box 지움: 맞는 JSON의 box와 다른 각도의 box가 함께 붙어 있다. 다른 각도의 box를 지운다.
    - 바꿈: 1-3이 넣은 box가 다른 각도의 box 하나이다. 맞는 JSON의 box로 바꾼다.
    - 채움: 1-3이 넣은 box가 없다. 맞는 JSON의 box를 넣는다.
    - 옮긴 JSON의 이미지 필드(파일명, 촬영 각도, `images.id` 등)는 그 사진의 값으로 바꾼다. 알약 필드, bbox, area, `annotations.id`는 그 JSON의 값 그대로 둔다.
        - JSON 3개가 사진 3장에 하나씩 가므로 `annotations.id`는 겹치지 않는다.
- 2-1의 라벨 누락(`missing_labels`)과 남는 box(`surplus_labels`)에서 고친 알약을 뺀다. 2-2부터는 고친 라벨로 본다.
- 표는 행마다 폴더 하나이다. 왼쪽 세 칸은 1-3이 각 사진에 넣은 그 K-코드의 JSON, 오른쪽 세 칸은 고친 뒤의 JSON과 처리이다.
    - box가 없거나 2개 이상인 칸은 굵게 적는다.
    - 합계 행은 2-1 표의 두 행이 사진 각도별로 몇 개였고, 고친 뒤 몇 개인지이다.

In [ ]:
# 폴더마다 JSON 3개의 box를 세 각도 사진에 모두 그려 보고 정한 배정이다 (아래 그림으로 확인)
COPY_ANGLE_FILES = {  # (조합, 알약 폴더) → 70°, 75°, 90° 사진에 맞는 JSON의 파일 이름 (각도와 copy 부분)
    ('K-001900-003544-004543-016548', 'K-016548'): ('70', '75 copy', '75'),
    ('K-001900-003544-004543-016551', 'K-016551'): ('70', '75 copy', '75'),
    ('K-001900-003544-006563-016551', 'K-001900'): ('70', '75 copy', '75'),
    ('K-001900-003544-016548-021026', 'K-021026'): ('75 copy 2', '75 copy', '75'),
    ('K-001900-003544-016548-027926', 'K-027926'): ('70', '75 copy 2', '75 copy'),
    ('K-001900-003544-016548-029345', 'K-016548'): ('75 copy', '75', '90'),
    ('K-001900-003544-016548-029345', 'K-029345'): ('70', '75 copy', '75'),
    ('K-001900-003544-016548-031705', 'K-001900'): ('70', '75 copy', '75'),
    ('K-001900-003544-016548-033009', 'K-033009'): ('70', '75 copy', '75'),
    ('K-001900-003544-016551-029451', 'K-001900'): ('75', '90 copy', '90'),
    ('K-001900-003544-016551-029451', 'K-029451'): ('70', '90 copy', '90'),
    ('K-001900-003544-016551-031705', 'K-031705'): ('70 copy 2', '70 copy', '70'),
    ('K-001900-004543-016551-019607', 'K-001900'): ('75 copy', '75', '90'),
    ('K-001900-004543-016551-033208', 'K-004543'): ('70', '75 copy', '75'),
    ('K-001900-006563-016548-018110', 'K-018110'): ('70', '75 copy', '75'),
    ('K-001900-006563-016548-019607', 'K-006563'): ('75 copy', '75', '90'),
    ('K-001900-006563-016548-019607', 'K-019607'): ('70', '90 copy', '90'),
    ('K-001900-006563-016548-024850', 'K-024850'): ('70', '90 copy', '90'),
    ('K-001900-010224-016548-029345', 'K-010224'): ('70', '90 copy', '90'),
    ('K-001900-010224-016548-029345', 'K-016548'): ('70', '90 copy', '90'),
    ('K-001900-010224-016548-033009', 'K-033009'): ('70', '75 copy', '75'),
}
ANGLES = ['70', '75', '90']


def raw_json_path(combo, folder, name):
    """'75 copy' 같은 파일 이름 부분으로 알약 폴더 안 원본 JSON의 경로를 만든다."""
    angle, _, copy = name.partition(' ')
    return AIHUB_RAW_ANN_DIR / f'{combo}_json' / folder / f"{combo}_0_2_0_2_{angle}_000_200{' ' + copy if copy else ''}.json"


def copy_action(before_names, name):
    """1-3이 이 사진에 넣은 JSON 이름 목록과 맞는 JSON 이름으로 처리를 정한다."""
    if before_names == [name]:
        return '그대로'
    if not before_names:
        return '채움'
    extra = f'남는 box {len(before_names) - 1}개 지움'
    if name in before_names:
        return extra
    return '바꿈' + (f', {extra}' if len(before_names) > 1 else '')


assert {(f'{combo}_json', folder) for combo, folder in COPY_ANGLE_FILES} == copy_pill_folders, \
    'copy 파일이 있는 알약 폴더(2-1의 copy_pill_folders)와 배정 표의 폴더가 다르다'
copy_fixes = []  # 사진마다 한 줄. 1-3이 넣은 box(before)와 다시 배정한 box(name, box), 처리
copy_json_of = {}  # (이미지, K-코드) → 다시 배정한 원본 JSON 경로 (2-2의 raw_annotation이 읽는다)
for n, ((combo, folder), names) in enumerate(COPY_ANGLE_FILES.items(), 1):
    name_of = {raw_json_path(combo, folder, name): name for name in names}  # 원본 JSON 경로 → 파일 이름 부분
    folder_jsons = {j['path']: j for j in aihub_kaggle_jsons if (j['combo_folder'], j['pill_folder']) == (f'{combo}_json', folder)}
    assert set(name_of) == set(folder_jsons), f'{combo} {folder}의 JSON이 배정 표와 다르다'
    code = int(folder[2:])
    for angle, name in zip(ANGLES, names):
        fn = f'{combo}_0_2_0_2_{angle}_000_200.png'
        record = aihub_kaggle[fn]
        # 1-3은 JSON 파일 이름의 각도대로 넣었다 (jsons_of: 1-5의 이미지 파일명 → 1-3의 JSON). 노트북을 다시 실행해도 같은 값이다
        before = [(name_of[j['path']], j['data']['annotations'][0]['bbox']) for j in jsons_of[fn] if j['path'] in name_of]
        right = folder_jsons[raw_json_path(combo, folder, name)]
        # 파일명, 촬영 각도, images.id 같은 이미지 필드는 이 사진의 값으로 바꾸고, annotation id는 1-3에서 이 JSON에 준 값을 그대로 쓴다
        image = {**right['data']['images'][0], **{key: record['image'][key] for key in IMAGE_FIELDS}}
        ann = {**right['data']['annotations'][0], 'image_id': record['image']['id'], 'image': image, 'json_path': str(right['path'])}
        kept = [a for a in record['annotations'] if Path(a['json_path']) not in name_of]  # 이 폴더의 JSON은 모두 빼고 맞는 하나만 넣는다
        record['annotations'] = sorted(kept + [ann], key=lambda a: a['category_id'])  # 1-3처럼 K-코드 폴더 순서로 둔다
        assert sum(a['category_id'] == code for a in record['annotations']) == pill_codes(fn).count(code), f'{fn} K-{code:06d}의 box 수가 파일명과 다르다'
        copy_json_of[(fn, code)] = right['path']
        copy_fixes.append({'n': n, 'combo': combo, 'folder': folder, 'angle': angle, 'file_name': fn, 'code': code,
                           'before': before, 'name': name, 'box': ann['bbox'],
                           'action': copy_action([b for b, _ in before], name)})
annotation_ids = [a['id'] for record in aihub_kaggle.values() for a in record['annotations']]
assert len(annotation_ids) == len(set(annotation_ids)), '다시 배정한 뒤 annotations.id가 겹친다'

# 2-1 표의 두 행이 모두 풀렸는지 확인하고, 2-2부터는 남은 라벨 문제만 센다
copy_filled = {(f['file_name'], f['code']) for f in copy_fixes if not f['before']}
copy_trimmed = {(f['file_name'], f['code']) for f in copy_fixes if len(f['before']) > 1}
assert {(m[0], m[1]) for m in missing_labels if missing_group(m) == MISSING_GROUPS[1]} <= copy_filled, \
    '2-1의 "JSON 없음: copy 파일이 있는 알약 폴더" 중 채우지 못한 알약이 있다'
assert {(s[0], s[1]) for s in surplus_labels} <= copy_trimmed, '2-1의 "box가 파일명보다 많음" 중 남는 box를 지우지 못한 이미지가 있다'
missing_labels = [m for m in missing_labels if (m[0], m[1]) not in copy_filled]
surplus_labels = [s for s in surplus_labels if (s[0], s[1]) not in copy_trimmed]


def before_text(before):
    """고치기 전 칸: 1-3이 이 사진에 넣은 그 K-코드의 JSON. box가 없거나 많으면 굵게 적는다."""
    names = ', '.join(f'`{name}`' for name, _ in before)
    if len(before) == 1:
        return names
    return f'**box {len(before)}개**: {names}' if before else '**box 없음**'


rows = []
for n, ((combo, folder), names) in enumerate(COPY_ANGLE_FILES.items(), 1):
    fixes = [f for f in copy_fixes if f['n'] == n]
    rows.append([n, f'`{combo}`<br>{folder}', *[before_text(f['before']) for f in fixes],
                 *[f"`{f['name']}` ({f['action']})" for f in fixes]])
by_angle = [[f for f in copy_fixes if f['angle'] == angle] for angle in ANGLES]
rows.append(['**합계**', '**사진에 box 없음**<br>(2-1의 JSON 없음: copy 파일이 있는 알약 폴더)',
             *[f"**{sum(not f['before'] for f in fixes)}개**" for fixes in by_angle], *['**0개**'] * 3])
rows.append(['', '**남는 box**<br>(2-1의 한 이미지에 같은 K-코드의 box가 파일명보다 많음)',
             *[f"**{sum(max(0, len(f['before']) - 1) for f in fixes)}개**" for fixes in by_angle], *['**0개**'] * 3])
action_kinds = [[('남는 box 지움' if f['action'].startswith('남는') else f['action']) for f in fixes] for fixes in by_angle]
rows.append(['', '**처리**', '', '', '',
             *['<br>'.join(f'{action} {count}장' for action, count in collections.Counter(kinds).most_common()) for kinds in action_kinds]])
show_table(['번호', '조합<br>알약 폴더', '고치기 전 70° 사진', '고치기 전 75° 사진', '고치기 전 90° 사진',
            '고친 뒤 70° 사진', '고친 뒤 75° 사진', '고친 뒤 90° 사진'], rows)

**인사이트**
- copy 파일이 있는 알약 폴더 21개는 JSON 3개 중 2~3개가 같은 각도를 가리킨다.
    - 겹친 각도는 75°가 14개 폴더로 가장 많고, 90° 6개, 70° 1개이다.
    - 4번, 12번 폴더는 JSON 3개가 모두 같은 각도이다.
- 파일 이름의 각도와 실제로 맞는 사진의 각도가 다른 JSON이 있다.
    - 1번 폴더는 `75` JSON이 90° 사진의 box이고, `75 copy`가 75° 사진의 box이다.
    - copy가 붙은 쪽이 항상 틀린 것도 아니다.
- 고친 뒤에는 사진마다 box가 하나씩이다. 빠진 box 23개와 남는 box 23개가 모두 0개가 된다.

**결론**
- copy 파일은 지우지 않고 `COPY_ANGLE_FILES`의 배정대로 맞는 각도의 사진에 넣는다. 파일 이름의 각도는 믿지 않는다.
- 이 배정은 정제 목록에 2-1 줄(45줄)로 적어, 학습 데이터 코드도 같은 배정을 쓰게 한다. (2-4)

- 위 표의 고치기 전과 고친 뒤를 사진 전체에 그려 나란히 놓는다. 행마다 폴더 하나(번호 = 위 표의 번호)이다.
    - 왼쪽(붉은 바탕) 세 칸은 고치기 전, 1-3이 넣은 라벨 그대로이다. 주황 box가 그 K-코드의 box이다.
    - 오른쪽(초록 바탕) 세 칸은 고친 뒤, 셀 62에서 고친 `aihub_kaggle`이다. 연두 box가 다시 배정한 box이다.
    - box 위나 아래의 [ ] 안은 box를 가져온 JSON 이름(각도와 copy 부분)이다.
    - 흰 점선 box는 같은 사진의 다른 알약 box이다. 남는 box가 다른 알약 위에 놓였는지 볼 때 쓴다.
- 칸 제목은 사진 각도와 그 K-코드의 box 수이다. 고치기 전에 box가 없거나 2개 이상이면 빨간 글씨이고, 고친 뒤에는 처리를 함께 적는다.
- 고치기 전에 box가 남거나 없던 사진에서, 고친 뒤 box 하나가 그 알약을 감싸는지 본다.

In [ ]:
def draw_copy_panel(ax, fn, other_boxes, boxes, color):
    """사진 전체에 다른 알약의 box(흰 점선)와 그 K-코드의 box(color, JSON 이름)를 그린다."""
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for code, (x, y, w, h) in other_boxes:
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='white', linewidth=1.2, linestyle='--'))
        ax.text(x + 4, y + 4, f'K-{code:06d}', fontsize=6, color='black', va='top', clip_on=True,
                bbox=dict(facecolor='white', alpha=0.7, pad=0.8, linewidth=0))
    for k, (name, (x, y, w, h)) in enumerate(boxes):
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2.5))
        # box가 여럿이면 이름표가 겹치지 않게 위 왼쪽, 아래 왼쪽, 위 오른쪽 순서로 단다
        tx, ha = (x + w, 'right') if k == 2 else (x, 'left')
        ty, va = (y + h + 6, 'top') if k == 1 else (y - 6, 'bottom')
        ax.text(tx, ty, f'[{name}]', fontsize=8, ha=ha, va=va, clip_on=True, bbox=dict(facecolor=color, alpha=0.85, pad=1, linewidth=0))
    ax.set_xticks([])
    ax.set_yticks([])


n_rows = len(COPY_ANGLE_FILES)
fig = plt.figure(figsize=(20, 4.6 * n_rows + 1), layout='constrained')
before_fig, after_fig = fig.subfigures(1, 2, wspace=0.02)
before_axes, after_axes = before_fig.subplots(n_rows, 3), after_fig.subplots(n_rows, 3)
for fix in copy_fixes:
    row, col = fix['n'] - 1, ANGLES.index(fix['angle'])
    fn, code = fix['file_name'], fix['code']
    # 고치기 전: 1-3이 넣은 그대로 (jsons_of, 1-5). 다른 알약의 box도 1-3 그대로 그린다
    before_others = [(json_code(j), j['data']['annotations'][0]['bbox']) for j in jsons_of[fn]
                     if json_code(j) != code and bbox_problem(j) is None]
    ax = before_axes[row, col]
    draw_copy_panel(ax, fn, before_others, fix['before'], 'orange')
    count = f"box {len(fix['before'])}개" if fix['before'] else 'box 없음'
    ax.set_title(f"{fix['angle']}° 사진: {count}", fontsize=10, color='black' if len(fix['before']) == 1 else 'red')
    # 고친 뒤: 다시 배정한 aihub_kaggle (셀 62)
    after_others = [(a['category_id'], a['bbox']) for a in aihub_kaggle[fn]['annotations']
                    if a['category_id'] != code and usable_bbox(a['bbox'])]
    ax = after_axes[row, col]
    draw_copy_panel(ax, fn, after_others, [(fix['name'], fix['box'])], 'lime')
    ax.set_title(f"{fix['angle']}° 사진: box 1개 ({fix['action']})", fontsize=10)
for n, ((combo, folder), _) in enumerate(COPY_ANGLE_FILES.items(), 1):
    for axes in (before_axes, after_axes):
        axes[n - 1, 0].set_ylabel(f'{n}. {folder}\n{combo}', fontsize=9)
before_fig.set_facecolor('#fdecea')
after_fig.set_facecolor('#e8f5e9')
before_fig.suptitle('고치기 전: 1-3의 라벨 (JSON 파일 이름의 각도대로 넣음)\n주황 = 그 K-코드의 box, [ ] = box를 가져온 JSON', fontsize=13)
after_fig.suptitle('고친 뒤: 맞는 각도에 다시 배정한 라벨 (셀 62)\n연두 = 그 K-코드의 box, [ ] = box를 가져온 JSON', fontsize=13)
fig.suptitle('copy 파일이 있는 알약 폴더 21개의 box 고치기 전과 고친 뒤 (행 = 위 표의 폴더 번호, 흰 점선 = 다른 알약의 box)', fontsize=15)
save_fig(fig, '2-1_copy_reassign')
plt.show()

**인사이트**
- 고치기 전 box가 2~3개 붙은 사진에서는 box들이 같은 알약 위에서 조금씩 어긋나 있다.
    - 어긋난 box는 다른 각도 사진의 알약 자리와 맞는다. 75°와 90° 사진은 배치가 같고 알약 자리만 조금 다르기 때문이다.
- 고친 뒤에는 21개 폴더 모두 세 각도 사진에서 box 하나가 그 알약을 감싼다.

**결론**
- 위 표의 배정을 그대로 쓴다. 2-2부터는 다시 배정한 라벨로 본다.

### 2-2. 좌표 오류와 고치는 방법
- 00 2-2와 같은 오류를 1-3의 `aihub_kaggle`에서 찾는다. bbox와 area는 원본 JSON 값 그대로이다.
    - 이미지 밖: bbox가 이미지(976×1280) 밖으로 나간다.
    - area 불일치: area가 bbox 너비×높이와 다르다. area 값의 오타일 수도 있고, bbox 값의 오타일 수도 있다.
    - 다른 클래스와 box가 똑같음: 같은 이미지 안에서 두 클래스의 box 좌표가 똑같다. 한 알약의 box를 다른 클래스에도 복사한 오류이다.
    - copy 파일을 다시 배정한 알약은 셀 62에서 고친 값(다시 배정한 JSON의 bbox와 area)을 쓴다.
- 이 세 검사는 쓸 수 있는 bbox(값 4개, 너비·높이가 0보다 큼, 2-1의 `usable_bbox`)에만 한다.
    - 쓸 수 없는 bbox는 2-1의 라벨 누락이므로 아래에서 함께 센다.
- 2-1의 라벨 누락도 2-1 표의 행 이름 그대로 같은 표에 센다. 고치는 방법이 같기 때문이다.
    - copy 파일을 다시 배정해 채운 알약은 셀 62에서 빠졌다.
    - 촬영 PNG만 있는 이미지(1-1)는 JSON이 하나도 없지만 사진은 있으므로, 사진에서 알약을 찾아 채운다.

In [ ]:
def raw_annotation(fn, code):
    """1-3의 aihub_kaggle에서 그 알약의 원본 (bbox, area)를 꺼낸다. annotation이 없으면 (None, None)이다.

    aihub_kaggle의 bbox와 area는 원본 JSON 값 그대로이고, copy 파일을 다시 배정한 알약(2-1)은 다시 배정한 JSON의 값이다.
    파일명에 같은 K-코드가 두 번 나오는 이미지가 없어서 (이미지, K-코드)마다 annotation은 하나이다.
    """
    anns = [a for a in aihub_kaggle.get(fn, {'annotations': []})['annotations'] if a['category_id'] == code]
    return (anns[0]['bbox'], anns[0]['area']) if anns else (None, None)


problems = collections.defaultdict(list)  # 이미지 → [(K-코드, 문제, 원본 bbox, 원본 area)]
for fn, record in sorted(aihub_kaggle.items()):
    anns = [a for a in record['annotations'] if usable_bbox(a['bbox'])]  # 쓸 수 없는 bbox는 아래에서 2-1의 라벨 누락으로 센다
    same_box = collections.Counter(tuple(a['bbox']) for a in anns)
    for ann in anns:
        x, y, w, h = ann['bbox']
        kinds = []
        if same_box[tuple(ann['bbox'])] > 1:
            kinds.append('다른 클래스와 box가 똑같음')
        if x < 0 or y < 0 or x + w > IMG_W or y + h > IMG_H:
            kinds.append('이미지 밖')
        if ann['area'] != w * h:
            kinds.append('area 불일치')
        if kinds:
            problems[fn].append((ann['category_id'], ' + '.join(kinds), ann['bbox'], ann['area']))
for m in missing_labels:  # 2-1의 라벨 누락 (copy 파일을 다시 배정해 채운 알약은 셀 62에서 빠졌다)
    fn, code = m[:2]
    problems[fn].append((code, f'라벨 누락: {missing_group(m)}', *raw_annotation(fn, code)))

kind_count = collections.Counter(kind for items in problems.values() for _, kind, _, _ in items)
rows = []
for kind, n in kind_count.most_common():
    files = sorted(fn for fn, items in problems.items() if any(k == kind for _, k, _, _ in items))
    example = next(item for item in problems[files[0]] if item[1] == kind)
    rows.append([kind, n, len(files), f'`{files[0]}` K-{example[0]:06d} bbox {example[2]}, area {example[3]}'])
rows.append(['**합계**', f'**{sum(kind_count.values())}**', f'**{len(problems)}장**', ''])
show_table(['문제', 'annotation 수', '이미지 수', '예시 (이미지, 원본 bbox, 원본 area)'], rows)

In [ ]:
problem_examples = []  # 위 표의 행마다 (문제, 예시 이미지, K-코드, 원본 bbox, 원본 area). 표와 같이 파일명 순서로 첫 이미지이다
for kind, _ in kind_count.most_common():
    files = sorted(fn for fn, items in problems.items() if any(k == kind for _, k, _, _ in items))
    problem_examples.append((kind, files[0], *next((code, bbox, area) for code, k, bbox, area in problems[files[0]] if k == kind)))
n_cols = 4
n_rows = int(np.ceil(len(problem_examples) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, 9.5 * n_rows), layout='constrained', squeeze=False)
for ax, (kind, fn, code, bbox, area) in zip(axes.flat, problem_examples):
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    anns_of_box = collections.defaultdict(list)  # box → 그 box를 쓰는 annotation. box가 똑같은 클래스는 이름을 한 번에 적는다
    for ann in aihub_kaggle.get(fn, {'annotations': []})['annotations']:
        if usable_bbox(ann['bbox']):  # 쓸 수 없는 bbox(라벨 누락)는 그릴 수 없다
            anns_of_box[tuple(ann['bbox'])].append(ann)
    for (x, y, w, h), anns in anns_of_box.items():
        target = any(a['category_id'] == code for a in anns)  # 예시 알약의 box는 빨간 점선으로 그린다 (이미지 밖으로 나간 부분은 잘려 보인다)
        color = 'red' if target else 'lime'
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2.5, linestyle='--' if target else '-'))
        ax.text(x, y - 8, '\n'.join(map(pill_label, anns)), fontsize=8, va='bottom', color='white' if target else 'black',
                bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0))
    pill_text = pill_label(aihub_example[code]) if code in aihub_example else f'K-{code:06d}'
    detail = f'{pill_text}: 원본 bbox {bbox}, area {area}'
    if usable_bbox(bbox):  # 좌표 문제는 area, 이미지 크기와 견줄 값을 함께 적는다
        x, y, w, h = bbox
        detail += f'\n너비×높이 {w * h}, 오른쪽 끝(x+너비) {x + w}, 아래 끝(y+높이) {y + h}'
    ax.set_title(f'[{kind}]\n{fn}\n{detail}', fontsize=9)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
    if not ax.images:
        ax.axis('off')
fig.suptitle(f'2-2 표의 문제별 예시 (표의 예시 이미지, 빨간 점선 = 예시 알약의 원본 bbox, 연두 = 같은 이미지의 다른 쓸 수 있는 bbox, '
             f'라벨 누락은 그릴 bbox가 없다, 이미지 {IMG_W}×{IMG_H})', fontsize=14)
save_fig(fig, '2-2_problem_examples')
plt.show()

**인사이트**
- 2-1의 라벨 누락까지 합치면 문제가 있는 annotation은 838개, 이미지 691장이다.
- area 불일치가 427개(414장)로 가장 많다.
    - 예시의 K-002483은 bbox 너비가 210인데 area 10248은 21×488이다. box는 알약을 잘 감싸므로 area 값만 틀렸다.
- 다른 클래스와 box가 똑같은 annotation은 134개(70장)이다.
    - 예시에서 K-005002와 K-012778이 ELC 캡슐의 box 하나를 같이 쓰고, K-012778(빨간 YH 알약)에는 box가 없다.
- 이미지 밖 box는 60개(이미지 밖 33개, 이미지 밖 + area 불일치 27개)이고, 숫자 오타가 원인으로 보인다.
    - `[77, 158, 3131, 498]`은 너비 313에 1이 하나 더 붙었다.
    - `[644, 851, 190, 498]`은 box가 HT EDT 캡슐이 아니라 아래 TYLENOL 쪽에 있다. y 값이 틀린 것으로 보인다.
- 숫자가 붙은 bbox `[121125, 333, 475]`는 `[121, 125, 333, 475]`에서 쉼표 하나가 빠진 값이다.

**결론**
- 좌표 오류는 대부분 숫자 하나가 빠지거나, 더해지거나, 쉼표가 빠진 오타이다. 오타를 고친 후보 중 사진의 알약과 맞는 값으로 고친다. (아래 처리 규칙)
- 원본 bbox가 없거나 오타로 고칠 수 없는 box는 사진에서 알약을 찾아 다시 그린다. (아래 OpenCV 방법)
- area는 bbox를 고친 뒤 너비×높이로 다시 계산한다. (00과 같은 방법)

- 문제가 있는 box는 사진에서 알약을 직접 찾아 고친다. 00 2-1은 영역을 눈으로 정했지만, 여기는 이미지가 많아 OpenCV로 알약 자리까지 찾는다.
    - 바닥 추정: 라벨 box 자리를 뺀 나머지 픽셀을 크게 흐려, 위치마다 바닥의 밝기(R+G+B)와 파랑 비율 B/(R+G+B)를 구한다. 사진 위쪽이 아래쪽보다 밝기 때문이다.
    - 알약 픽셀: 00 2-1의 기준(바닥보다 파랑 비율이 0.04 이상 낮거나, 20% 이상 밝음)에, 바닥보다 15% 이상 어둡고 더 푸르지 않은 픽셀을 더한다.
        - 검은 캡슐은 바닥보다 어둡지만 색 비율은 바닥과 같아서 00의 기준으로는 찾지 못한다. 그림자는 바닥보다 어둡고 더 푸르러서 빠진다.
    - 알약 덩어리: 1/4로 줄인 사진에서 알약 픽셀이 붙은 덩어리(`cv2.connectedComponentsWithStats`)를 찾는다. 작은 잡음은 뺀다.
        - 조명 밖의 밝은 모서리는 밝기 기준에 걸리므로, 밝기로만 찾은 픽셀 중 사진 테두리와 이어진 덩어리를 먼저 지운다. 테두리 가까이 놓인 색 있는 알약은 남는다.
    - 알약 범위: 덩어리 주변을 원본 해상도로 다시 보고, 00 2-1처럼 알약 픽셀이 충분한 행과 열의 범위를 잰다.
    - box: 알약 범위의 네 변에 클래스별 여백(라벨 box 변과 알약 범위 사이 거리)의 중앙값을 더한다.
- 클래스 기준값: 라벨에 문제가 없는 이미지 3,000장(seed 고정)에서 라벨 box마다 알약 범위를 재서, 클래스마다 여백, 알약 범위의 긴 변·짧은 변, 가운데 색(Lab)의 중앙값을 구한다.
    - Lab은 사람 눈이 느끼는 색 차이에 가깝게 만든 색 공간이다. 찾은 알약이 어느 클래스인지 크기와 색으로 정할 때 쓴다.
- 검증: 기준값을 만들 때 쓰지 않은 다른 이미지 1,000장으로 확인한다.
    - box 하나 지우기: 라벨 box 하나를 지우고 사진에서 다시 찾아, 크기와 색이 그 클래스에 가장 가까운 덩어리로 만든 box와 라벨 box의 IoU를 본다.
    - box 둘 지우기: 라벨 box 둘을 지우고 다시 찾아, 크기와 색으로 두 클래스를 맞는 알약에 배정하는지 본다. (box가 똑같은 오류와 알약 여러 개가 빠진 이미지에 쓴다)
- 계산에 몇 분 걸려 결과를 `runs/eda_raewoo/cache/2-2_pill_finder.json`에 저장한다.
    - 고른 이미지와 그 box가 같을 때만 저장한 결과를 쓴다.

In [ ]:
SCALE = 4  # 알약 덩어리를 찾을 때 사진을 줄이는 배율
BLUE_DROP = 0.04  # 바닥보다 파랑 비율이 이만큼 낮으면 알약 픽셀 (00 2-1)
BRIGHT_RISE = 0.2  # 바닥보다 이 비율 이상 밝으면 알약 픽셀 (흰 알약, 00 2-1)
DARK_DROP, DARK_BLUE = 0.15, 0.01  # 바닥보다 15% 이상 어둡고 파랑 비율이 바닥+0.01보다 낮으면 알약 픽셀 (검은 캡슐)
MIN_SHARE = 0.1  # 알약 픽셀이 가장 많은 행(열)의 이 비율 이상인 행(열)만 알약 범위에 넣는다 (00 2-1)
REFINE_PAD = 24  # 덩어리 주변을 원본 해상도로 다시 볼 때 넓히는 폭 (픽셀)
MIN_COMPONENT = 300  # 줄인 사진에서 이보다 작은 덩어리(원본 약 4,800픽셀)는 알약으로 보지 않는다
N_REF, N_VAL = 3000, 1000  # 클래스 기준값과 검증에 쓰는 이미지 수


def read_rgb(path):
    return cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)


def box_area_mask(shape, boxes, scale=1, pad=0):
    """xywh box 자리를 True로 칠한 마스크를 만든다."""
    mask = np.zeros(shape, bool)
    for x, y, w, h in boxes:
        mask[max(0, int(y) // scale - pad):int(y + h) // scale + pad, max(0, int(x) // scale - pad):int(x + w) // scale + pad] = True
    return mask


def background(rgb, known_boxes):
    """위치마다 바닥의 밝기(R+G+B)와 파랑 비율을 추정한다. 라벨 box 자리와 바닥과 크게 다른 픽셀은 빼고 흐리게 평균낸다."""
    small = cv2.resize(rgb, (IMG_W // SCALE, IMG_H // SCALE), interpolation=cv2.INTER_AREA).astype(np.float32)
    total = small.sum(axis=2) + 1
    blue = small[..., 2] / total
    known = box_area_mask(total.shape, known_boxes, SCALE, 4)
    floor = (~known) & (np.abs(blue - np.median(blue[~known])) < 0.03) & (np.abs(total / np.median(total[~known]) - 1) < 0.15)
    weight = np.maximum(cv2.GaussianBlur(floor.astype(np.float32), (0, 0), 25), 1e-3)
    bg_total = cv2.GaussianBlur(np.where(floor, total, 0).astype(np.float32), (0, 0), 25) / weight
    bg_blue = cv2.GaussianBlur(np.where(floor, blue, 0).astype(np.float32), (0, 0), 25) / weight
    return cv2.resize(bg_total, (IMG_W, IMG_H)), cv2.resize(bg_blue, (IMG_W, IMG_H))


def pill_pixels(rgb, bg_total, bg_blue, split=False):
    """바닥과 다른 픽셀(알약)을 True로 표시한다. split이면 (색으로 찾은 픽셀, 밝기로만 찾은 픽셀)을 따로 돌려준다."""
    rgb = rgb.astype(np.float32)
    total = rgb.sum(axis=2) + 1
    blue = rgb[..., 2] / total
    colored = (blue < bg_blue - BLUE_DROP) | ((total < bg_total * (1 - DARK_DROP)) & (blue < bg_blue + DARK_BLUE))
    bright = (total > bg_total * (1 + BRIGHT_RISE)) & ~colored
    return (colored, bright) if split else (colored | bright)


def corner_zone(bright):
    """밝기로만 찾은 픽셀 중 사진 테두리와 이어진 덩어리(조명 밖의 밝은 모서리)를 찾는다."""
    _, labels = cv2.connectedComponents(bright.astype(np.uint8), connectivity=8)
    edge = np.unique(np.concatenate([labels[0], labels[-1], labels[:, 0], labels[:, -1]]))
    return np.isin(labels, edge[edge > 0])


def refine_extent(rgb, box, bg_total, bg_blue, known_boxes, corner=None):
    """box 주변을 원본 해상도로 보고 가장 큰 알약 덩어리의 범위 [x0, y0, x1, y1]를 찾는다. 다른 라벨 box 자리와 밝은 모서리는 뺀다."""
    x, y, w, h = box
    x0, y0 = max(0, int(x) - REFINE_PAD), max(0, int(y) - REFINE_PAD)
    x1, y1 = min(IMG_W, int(x + w) + REFINE_PAD), min(IMG_H, int(y + h) + REFINE_PAD)
    if x1 - x0 < 8 or y1 - y0 < 8:
        return None  # box가 이미지 밖에 있다
    mask = pill_pixels(rgb[y0:y1, x0:x1], bg_total[y0:y1, x0:x1], bg_blue[y0:y1, x0:x1])
    if corner is not None:
        mask &= ~corner[y0:y1, x0:x1]
    for kx, ky, kw, kh in known_boxes:
        mask[max(0, int(ky) - y0):max(0, int(ky + kh) - y0), max(0, int(kx) - x0):max(0, int(kx + kw) - x0)] = False
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    n, labels, comp_stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return None
    comp = labels == 1 + int(np.argmax(comp_stats[1:, 4]))
    rows = np.nonzero(comp.sum(axis=1) >= MIN_SHARE * comp.sum(axis=1).max())[0]
    cols = np.nonzero(comp.sum(axis=0) >= MIN_SHARE * comp.sum(axis=0).max())[0]
    return [x0 + int(cols[0]), y0 + int(rows[0]), x0 + int(cols[-1]) + 1, y0 + int(rows[-1]) + 1]


def find_pills(rgb, known_boxes):
    """라벨 box 밖에서 알약을 찾아 알약 범위 [x0, y0, x1, y1] 목록을 큰 덩어리부터 돌려준다."""
    bg_total, bg_blue = background(rgb, known_boxes)
    small = cv2.resize(rgb, (IMG_W // SCALE, IMG_H // SCALE), interpolation=cv2.INTER_AREA)
    size = small.shape[1::-1]
    colored, bright = pill_pixels(small, cv2.resize(bg_total, size), cv2.resize(bg_blue, size), split=True)
    corner = cv2.dilate(corner_zone(bright).astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)
    mask = (colored | bright) & ~corner & ~box_area_mask(colored.shape, known_boxes, SCALE, 4)
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))
    _, _, comp_stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    corner_full = cv2.resize(corner.astype(np.uint8), (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST).astype(bool)
    extents = []
    for x, y, w, h, area in sorted(comp_stats[1:], key=lambda s: -s[4]):
        if area < MIN_COMPONENT:
            continue
        extent = refine_extent(rgb, [x * SCALE, y * SCALE, w * SCALE, h * SCALE], bg_total, bg_blue, known_boxes, corner_full)
        if extent:
            extents.append(extent)
    return extents


def extent_color(rgb, extent):
    """알약 범위 가운데 절반의 평균 색을 Lab으로 구한다."""
    x0, y0, x1, y1 = extent
    patch = rgb[y0 + (y1 - y0) // 4:y1 - (y1 - y0) // 4, x0 + (x1 - x0) // 4:x1 - (x1 - x0) // 4]
    return cv2.cvtColor(patch, cv2.COLOR_RGB2LAB).reshape(-1, 3).astype(float).mean(axis=0)


def appearance_cost(extent, lab, ref):
    """알약 범위의 긴 변·짧은 변과 Lab 색이 클래스 기준값과 얼마나 다른지 잰다. 0이면 같다. (색 차이 30이 크기 e배 차이와 같은 무게)"""
    long_side, short_side = sorted([extent[2] - extent[0], extent[3] - extent[1]], reverse=True)
    return (abs(np.log(long_side / ref['long'])) + abs(np.log(short_side / ref['short']))
            + np.linalg.norm(np.asarray(lab) - np.asarray(ref['lab'])) / 30)


def with_margin(extent, margin):
    """알약 범위에 여백(왼, 위, 오른, 아래)을 더해 xywh box를 만든다. 이미지 밖으로 나가지 않게 자른다."""
    x0, y0, x1, y1 = extent
    x, y = max(0, round(x0 - margin[0])), max(0, round(y0 - margin[1]))
    return [int(x), int(y), int(min(IMG_W, round(x1 + margin[2])) - x), int(min(IMG_H, round(y1 + margin[3])) - y)]


finder_cache = CACHE_DIR / '2-2_pill_finder.json'
clean_pool = sorted(fn for fn in aihub_kaggle if fn not in problems and fn in set(aihub_files))  # 라벨에 문제가 없는 이미지
order = np.random.default_rng(SEED).permutation(len(clean_pool))
ref_files, val_files = [clean_pool[i] for i in order[:N_REF]], [clean_pool[i] for i in order[N_REF:N_REF + N_VAL]]
finder_key = [[fn, [a['bbox'] for a in aihub_kaggle[fn]['annotations']]] for fn in ref_files + val_files]  # 고른 이미지와 그 box
if finder_cache.exists() and json.loads(finder_cache.read_text()).get('key') == finder_key:
    finder = json.loads(finder_cache.read_text())
else:
    measures = collections.defaultdict(list)  # K-코드 → [여백 4개, 긴 변, 짧은 변, Lab 3개]
    for fn in ref_files:
        rgb = read_rgb(AIHUB_IMG_DIR / fn)
        boxes = [a['bbox'] for a in aihub_kaggle[fn]['annotations']]
        bg_total, bg_blue = background(rgb, boxes)
        for k, ann in enumerate(aihub_kaggle[fn]['annotations']):
            extent = refine_extent(rgb, ann['bbox'], bg_total, bg_blue, boxes[:k] + boxes[k + 1:])
            if extent:
                x, y, w, h = ann['bbox']
                measures[ann['category_id']].append([extent[0] - x, extent[1] - y, x + w - extent[2], y + h - extent[3],
                                                     *sorted([extent[2] - extent[0], extent[3] - extent[1]], reverse=True),
                                                     *extent_color(rgb, extent)])
    global_margin = np.median([m[:4] for ms in measures.values() for m in ms], axis=0)
    refs = {str(c): {'margin': (np.median(np.array(ms)[:, :4], axis=0) if len(ms) >= 3 else global_margin).tolist(),
                     'long': float(np.median(np.array(ms)[:, 4])), 'short': float(np.median(np.array(ms)[:, 5])),
                     'lab': np.median(np.array(ms)[:, 6:9], axis=0).tolist(), 'n': len(ms)} for c, ms in measures.items()}
    hide_one, hide_two = [], []  # (K-코드, IoU), (맞게 배정했는지)
    rng = np.random.default_rng(SEED)
    for fn in val_files:
        rgb = read_rgb(AIHUB_IMG_DIR / fn)
        anns = [a for a in aihub_kaggle[fn]['annotations']]
        boxes = [a['bbox'] for a in anns]
        for k, ann in enumerate(anns):
            ref = refs.get(str(ann['category_id']))
            extents = find_pills(rgb, boxes[:k] + boxes[k + 1:]) if ref else []
            if not ref:
                continue
            if not extents:
                hide_one.append((ann['category_id'], 0.0))
                continue
            best = min(extents, key=lambda e: appearance_cost(e, extent_color(rgb, e), ref))
            hide_one.append((ann['category_id'], box_iou(with_margin(best, ref['margin']), ann['bbox'])))
        pair = [k for k in rng.permutation(len(anns))[:2] if str(anns[k]['category_id']) in refs]
        if len(pair) == 2:
            extents = find_pills(rgb, [b for k, b in enumerate(boxes) if k not in pair])
            if len(extents) < 2:
                hide_two += [False, False]
                continue
            cost = np.array([[appearance_cost(e, extent_color(rgb, e), refs[str(anns[k]['category_id'])]) for e in extents] for k in pair])
            for i, j in zip(*scipy.optimize.linear_sum_assignment(cost)):
                ref = refs[str(anns[pair[i]]['category_id'])]
                hide_two.append(box_iou(with_margin(extents[j], ref['margin']), anns[pair[i]]['bbox']) > 0.5)
    finder = {'key': finder_key, 'refs': refs, 'global_margin': global_margin.tolist(),
              'hide_one': hide_one, 'hide_two': [bool(v) for v in hide_two]}
    finder_cache.write_text(json.dumps(finder))
class_refs = {int(c): ref for c, ref in finder['refs'].items()}
hide_one_iou = np.array([v for _, v in finder['hide_one']])

fig, axes = plt.subplots(1, 2, figsize=(20, 5.5), layout='constrained', gridspec_kw={'width_ratios': [1, 2]})
axes[0].hist(hide_one_iou, bins=np.linspace(0, 1, 51), color='tab:blue')
for t in [0.75, 0.95]:
    axes[0].axvline(t, color='gray', linestyle='--', linewidth=1)
axes[0].set_yscale('log')
axes[0].set_xlabel('다시 찾은 box와 라벨 box의 IoU')
axes[0].set_ylabel('box 수 (로그 눈금)')
axes[0].set_title(f'box 하나 지우고 다시 찾기 {len(hide_one_iou):,}개: 중앙값 {np.median(hide_one_iou):.3f}, '
                  f'0.75 미만 {(hide_one_iou < 0.75).mean():.1%}, 0.95 이상 {(hide_one_iou >= 0.95).mean():.1%}', fontsize=11)
by_class = collections.defaultdict(list)
for c, v in finder['hide_one']:
    by_class[c].append(v)
order = sorted(by_class, key=lambda c: np.median(by_class[c]))
axes[1].bar(np.arange(len(order)), [np.median(by_class[c]) for c in order], color='tab:blue')
axes[1].axhline(0.95, color='gray', linestyle='--', linewidth=1)
axes[1].axhline(0.75, color='gray', linestyle='--', linewidth=1)
axes[1].set_xticks(np.arange(len(order)), [f'K-{c:06d}' for c in order], rotation=90, fontsize=6)
axes[1].set_ylim(0.5, 1)
axes[1].set_ylabel('IoU 중앙값')
axes[1].set_title(f'클래스별 IoU 중앙값 ({len(order)}개 클래스, 낮은 순서)', fontsize=11)
fig.suptitle(f'OpenCV로 빠진 box를 찾는 방법의 검증 (이미지 {N_VAL:,}장, 회색 점선 = IoU 0.75와 0.95), '
             f"box 둘 지우고 클래스 배정: {sum(finder['hide_two'])}/{len(finder['hide_two'])} 맞음", fontsize=13)
save_fig(fig, '2-2_finder_validation')
plt.show()

In [ ]:
refind_samples = []  # 위 셀의 box 하나 지우기 검증 box (이미지, annotation 순번, K-코드). hide_one과 같은 순서로 돈다
for fn in val_files:
    for k, ann in enumerate(aihub_kaggle[fn]['annotations']):
        if str(ann['category_id']) in finder['refs']:
            refind_samples.append((fn, k, ann['category_id']))
assert [s[2] for s in refind_samples] == [c for c, _ in finder['hide_one']], '검증 box 순서가 위 셀의 hide_one과 다르다'
refind_samples = [(*s, iou) for s, (_, iou) in zip(refind_samples, finder['hide_one'])]
# IoU 중앙값이 0.85 미만인 클래스(2-2에서 사진으로 찾지 않는 클래스)마다 0.75 미만 사진을 IoU 순서로 고르게 4장 고른다
hard_classes = sorted((c for c in by_class if np.median(by_class[c]) < 0.85), key=lambda c: np.median(by_class[c]))
panels = []
for c in hard_classes:
    fails = sorted((s for s in refind_samples if s[2] == c and s[3] < 0.75), key=lambda s: (s[3], s[0]))
    panels += [fails[i] for i in np.unique(np.linspace(0, len(fails) - 1, min(4, len(fails))).round().astype(int))]
n_rows = int(np.ceil(len(panels) / 4))
fig, axes = plt.subplots(n_rows, 4, figsize=(20, 7.8 * n_rows), layout='constrained', squeeze=False)
for ax, (fn, k, code, iou) in zip(axes.flat, panels):
    rgb = read_rgb(AIHUB_IMG_DIR / fn)
    boxes = [a['bbox'] for a in aihub_kaggle[fn]['annotations']]
    extents = find_pills(rgb, boxes[:k] + boxes[k + 1:])  # 위 셀처럼 k번째 box를 지우고 다시 찾는다
    ax.imshow(rgb)
    for j, (x, y, w, h) in enumerate(boxes):
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime' if j == k else 'white', linewidth=2.5 if j == k else 1))
    for x0, y0, x1, y1 in extents:
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='yellow', linewidth=1.2, linestyle=':'))
    if extents:
        best = min(extents, key=lambda e: appearance_cost(e, extent_color(rgb, e), class_refs[code]))
        found = with_margin(best, class_refs[code]['margin'])
        ax.add_patch(plt.Rectangle(found[:2], found[2], found[3], fill=False, edgecolor='orangered', linewidth=2.5, linestyle='--'))
    assert abs((box_iou(found, boxes[k]) if extents else 0.0) - iou) < 1e-9, f'{fn}: 다시 찾은 IoU가 위 셀과 다르다'
    v = np.array(by_class[code])
    ax.set_title(f"{pill_label(aihub_example[code])} {aihub_example[code]['image']['dl_name']}\n"
                 f'클래스 IoU 중앙값 {np.median(v):.3f}, 0.75 미만 {(v < 0.75).sum()}/{len(v)}개\n{fn}\n'
                 f'이 사진 IoU {iou:.3f}, 찾은 알약 덩어리 후보 {len(extents)}개', fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
for ax in axes.flat[len(panels):]:
    ax.axis('off')
fig.suptitle(f'box 하나 지우고 다시 찾기의 IoU 중앙값이 0.85 미만인 클래스 {len(hard_classes)}개의 실패 사진 '
             '(한 줄에 한 클래스, IoU 0.75 미만 사진을 IoU 순서로 고르게 4장)\n'
             '연두 = 지운 라벨 box, 노랑 점선 = 사진에서 찾은 알약 덩어리 후보, 주황 점선 = 크기·색으로 고른 후보에 여백을 더한 box, '
             '흰 선 = 다른 라벨 box (찾을 때 빼는 자리)', fontsize=14)
save_fig(fig, '2-2_finder_failures')
plt.show()

**인사이트**
- box 하나를 지우고 다시 찾으면 라벨 box와의 IoU 중앙값이 0.958이다. 3,835개 중 60.6%가 0.95 이상이다.
- IoU 0.75 미만은 5.0%이고, 대부분 바닥과 색이 같은 파란 알약 4개 클래스에서 나온다.
    - K-010487 50/50개, K-012638 42/42개, K-031885 38/38개, K-041149 21/26개가 0.75 미만이다. (아래 실패 사진 제목)
    - 이 4개 클래스의 151개를 빼면 0.75 미만은 약 1%이다.
    - 실패 사진에서 파란 캡슐과 파란 알약은 알약 덩어리로 잡히지 않는다. 흰색과 파란색 반반인 K-041149는 흰 쪽만 잡힌다.
- 나머지 111개 클래스는 IoU 중앙값이 0.85 이상이다.
- box 둘을 지우고 크기·색으로 클래스를 배정하면 1,966개 중 1,816개(92.4%)를 맞힌다.

**결론**
- 사진에서 찾은 box를 빠진 box 채우기와 오타 후보 확인에 쓴다. 대부분의 클래스에서 평가 기준인 IoU 0.75를 넘는다.
- 4개 클래스(K-010487, K-012638, K-031885, K-041149)는 사진에서 찾은 box를 쓰지 않는다. 이 클래스의 box는 2-3 갤러리에서 눈으로 확인한다.
- 크기·색 배정은 8% 정도 틀리므로, 크기·색 차이 1.5 이하인 짝만 쓰고 결과를 문제 갤러리에서 눈으로 확인한다.

- 좌표 값은 멀쩡해도 box가 알약을 감싸지 않는 경우가 있다. (box가 바닥이나 옆 알약을 감쌈, 한쪽으로 밀림)
    - 이런 box는 2-2 표의 검사로는 찾을 수 없어서, 쓸 수 있는 모든 box를 사진과 대조한다.
- box 맞음: 라벨 box 주변에서 알약 범위를 재고(`refine_extent`), 클래스 여백을 더해 다시 만든 box와 라벨 box의 IoU이다.
    - 1이면 라벨 box가 알약을 클래스의 다른 box들과 같은 여백으로 감싼다. 0이면 box 안에 알약이 없거나 다른 자리에 있다.
    - 0.75 미만인 box를 문제("알약을 감싸지 않는 box")로 보고, 2-2 표의 문제와 함께 아래 규칙으로 고친다.
        - 0.75는 평가 지표의 가장 낮은 IoU 기준이다. 위 검증에서 box 하나 지우기의 IoU가 0.75 미만인 경우는 1~2%였다.
    - 알약 범위를 잘 재지 못하는 클래스는 이 검사에서 뺀다. 이 클래스는 갤러리(2-3)로 본다.
        - 위 검증에서 IoU 중앙값이 0.85 미만인 클래스
        - box 맞음 0.75 미만이 그 클래스 box의 30%를 넘는 클래스 (라벨이 그만큼 틀렸을 가능성보다 재는 방법이 실패했을 가능성이 크다)
- 여기서 잰 알약 범위와 색은 2-3의 크기·색 대조에도 쓴다. 결과는 `runs/eda_raewoo/cache/2-2_box_measures.json`에 저장한다.
    - 이미지의 box 목록이 같을 때만 저장한 값을 쓴다. 알약 범위를 잴 때 같은 사진의 다른 box 자리를 빼기 때문이다.

In [ ]:
FIT_THRESHOLD = 0.75


def measure_boxes(job):
    """이미지 하나의 box마다 (알약 범위, Lab 색)을 잰다. 알약 범위를 못 찾으면 None이다."""
    fn, boxes = job
    rgb = read_rgb(AIHUB_IMG_DIR / fn)
    bg_total, bg_blue = background(rgb, boxes)
    out = []
    for k, box in enumerate(boxes):
        extent = refine_extent(rgb, box, bg_total, bg_blue, boxes[:k] + boxes[k + 1:])
        out.append([fn, box, extent, extent_color(rgb, extent).tolist() if extent else None])
    return out


def measure_all(records):
    """쓸 수 있는 box마다 알약 범위와 색을 재서 (이미지, box) → (알약 범위, Lab 색)으로 돌려준다.
    같은 box 목록을 이미 잰 이미지는 캐시에서 읽는다."""
    cache_path = CACHE_DIR / '2-2_box_measures.json'
    cached = json.loads(cache_path.read_text()) if cache_path.exists() else {}  # '이미지 box 목록' → box마다 [알약 범위, Lab 색]
    jobs = {}  # 캐시 키 → (이미지, 쓸 수 있는 box 목록)
    for fn, r in sorted(records.items()):
        boxes = [a['bbox'] for a in r['annotations'] if usable_bbox(a['bbox'])]  # 쓸 수 없는 bbox는 2-2 표의 라벨 누락이다
        jobs[f'{fn} {json.dumps(boxes)}'] = (fn, boxes)
    todo = [key for key in jobs if key not in cached]
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
        for key, rows in zip(todo, pool.map(measure_boxes, [jobs[key] for key in todo])):
            cached[key] = [[extent, lab] for _, _, extent, lab in rows]
    if todo:
        cache_path.write_text(json.dumps(cached))
    return {(fn, tuple(box)): tuple(cached[key][k]) for key, (fn, boxes) in jobs.items() for k, box in enumerate(boxes)}


box_measures = measure_all({fn: r for fn, r in aihub_kaggle.items() if fn in set(aihub_files)})
no_ref = {'margin': finder['global_margin']}
box_fit = {}  # (이미지, K-코드) → box 맞음 IoU
for fn, record in aihub_kaggle.items():
    if fn not in set(aihub_files):
        continue
    for ann in record['annotations']:
        if not usable_bbox(ann['bbox']):
            continue  # 쓸 수 없는 bbox는 잴 수 없다 (2-2 표의 라벨 누락)
        extent, _ = box_measures[(fn, tuple(ann['bbox']))]
        margin = class_refs.get(ann['category_id'], no_ref)['margin']
        box_fit[(fn, ann['category_id'])] = box_iou(with_margin(extent, margin), ann['bbox']) if extent else 0.0
# 위 검증에서 다시 찾기가 잘 안 되는 클래스(IoU 중앙값 0.85 미만, 예: 바닥과 색이 같은 파란 알약)는 box 맞음을 믿을 수 없어 뺀다
validation_by_class = collections.defaultdict(list)
for c, v in finder['hide_one']:
    validation_by_class[c].append(v)
low_share = collections.defaultdict(list)
for (fn, code), v in box_fit.items():
    low_share[code].append(v < FIT_THRESHOLD)
# 한 클래스의 30% 넘게 box 맞음이 낮으면 라벨 오류가 아니라 알약 범위를 못 재는 클래스로 본다
unreliable_classes = sorted({c for c, v in validation_by_class.items() if np.median(v) < 0.85}
                            | {c for c, v in low_share.items() if len(v) >= 10 and np.mean(v) > 0.3})
already = {(fn, code) for fn, items in problems.items() for code, *_ in items}
low_fit = sorted(key for key, v in box_fit.items() if v < FIT_THRESHOLD and key not in already and key[1] not in unreliable_classes)
for fn, code in low_fit:
    raw_box, raw_area = raw_annotation(fn, code)
    problems[fn].append((code, '알약을 감싸지 않는 box', raw_box, raw_area))
kind_count = collections.Counter(kind for items in problems.values() for _, kind, _, _ in items)

fit_values = np.array(list(box_fit.values()))
fig, axes = plt.subplots(1, 2, figsize=(20, 5.5), layout='constrained', gridspec_kw={'width_ratios': [1, 1.6]})
axes[0].hist(fit_values, bins=np.linspace(0, 1, 51), color='tab:blue')
axes[0].axvline(FIT_THRESHOLD, color='red', linestyle='--', linewidth=1)
axes[0].set_yscale('log')
axes[0].set_xlabel('box 맞음 (다시 만든 box와 라벨 box의 IoU)')
axes[0].set_ylabel('box 수 (로그 눈금)')
axes[0].set_title(f'box {len(fit_values):,}개: 중앙값 {np.median(fit_values):.3f}, {FIT_THRESHOLD} 미만 {(fit_values < FIT_THRESHOLD).sum()}개', fontsize=11)
low_by_class = collections.Counter(code for (fn, code), v in box_fit.items() if v < FIT_THRESHOLD)
n_by_class = collections.Counter(code for fn, code in box_fit)
order = sorted(low_by_class, key=lambda c: -low_by_class[c] / n_by_class[c])[:40]
axes[1].bar([f'K-{c:06d}' for c in order], [low_by_class[c] / n_by_class[c] for c in order], color='tab:orange')
axes[1].tick_params(axis='x', rotation=90, labelsize=7)
axes[1].set_ylabel('box 맞음 0.75 미만 비율')
axes[1].set_title('클래스별 box 맞음 0.75 미만 비율 (높은 40개 클래스)', fontsize=11)
fig.suptitle(f'모든 AI Hub box의 box 맞음 (빨간 점선 = {FIT_THRESHOLD}, 이 아래는 "알약을 감싸지 않는 box"로 고친다), '
             f'믿을 수 없어 뺀 클래스: {", ".join(f"K-{c:06d}" for c in unreliable_classes) or "없음"}', fontsize=13)
save_fig(fig, '2-2_box_fit')
plt.show()

**인사이트**
- 쓸 수 있는 box 45,746개의 box 맞음 중앙값은 0.958이다. 0.75 미만은 2,829개(6.2%)이다.
- 0.75 미만 비율이 30%를 넘는 6개 클래스는 재는 방법이 실패한 것으로 보고 검사에서 뺐다.
    - K-005391, K-010487, K-012638, K-031885는 거의 100%, K-041149는 86%, K-027777은 32%이다.
    - K-027777을 뺀 5개는 바닥과 색이 비슷한 파란 계열 알약이다. (위 검증의 실패 사진)
- 나머지 클래스는 0.75 미만 비율이 5.5%(K-019881) 이하이다.
- 이 검사로 새로 찾은 문제 box는 313개이다. (아래 처리 표의 "알약을 감싸지 않는 box" 행)
    - 좌표 값이 정상 범위라 2-2 표의 검사로는 찾지 못한 box이다.

**결론**
- box 맞음 0.75 미만인 box를 "알약을 감싸지 않는 box"로 보고 2-2 표의 문제와 같은 규칙으로 고친다.
- 뺀 6개 클래스의 box는 2-3 crop 갤러리에서 눈으로 확인한다.

- 2-2 표의 문제와 "알약을 감싸지 않는 box"마다 아래 규칙으로 고친다.
    - 문제가 있는 annotation은 모두 라벨 box에서 빼고, 남은 정상 box 밖에서 알약을 찾는다.
    - 찾은 알약과 문제 annotation을 크기와 색으로 짝짓는다. (위 검증의 box 둘 지우기와 같은 방법)
        - 크기·색 차이(`appearance_cost`)가 1.5를 넘는 짝은 만들지 않는다. 정상 라벨 box의 99.5%가 자기 클래스와 1.5 이하이다.
        - 사진에서 보이지 않는 알약의 클래스가 남은 다른 알약을 차지하지 않게 하려는 것이다. (예: 파란 알약 클래스가 옆의 주황 알약에 짝지어짐)
    - 원본 bbox가 있는 문제(이미지 밖, area 불일치, 숫자가 붙은 bbox, 알약을 감싸지 않는 box)는 원본 값을 최대한 살린다.
        - 원본 bbox와 찾은 알약 box의 IoU가 0.8 이상이면 bbox는 그대로 둔다. area 값만 틀린 것이다.
        - 아니면 오타를 고친 후보(숫자 하나를 빼거나, 넣거나, 바꾼 값, 쉼표가 빠져 붙은 숫자를 나눈 값)를 만든다.
            - 너비×높이가 원본 area와 같은 후보 중 찾은 알약 box와 IoU가 0.8 이상인 후보가 있으면, 그중 IoU가 가장 높은 값으로 고친다. ("숫자 하나 고침")
            - 그런 후보가 없으면 모든 후보 중 IoU가 0.85 이상인 값으로 고친다. 후보가 많아 우연히 맞는 값을 피하려고 기준을 높인다.
        - 맞는 후보가 없으면 찾은 알약 box로 바꾼다.
    - 다른 클래스와 box가 똑같은 두 annotation은 크기와 색으로 어느 클래스가 그 box의 알약인지 정한다.
        - 그 box의 알약으로 정해진 클래스는 원본 box를 그대로 쓰고, 다른 클래스는 찾은 알약 box를 쓴다.
    - 원본 bbox가 없는 문제(JSON 없음, bbox 빈 값, `-오류` 폴더, 읽을 수 없는 JSON)는 찾은 알약 box를 넣는다.
        - `-오류` 폴더의 알약은 사진에 폴더 이름과 다른 알약이 놓였다. (셀 87)
        - 그래서 사진으로 확인한 실제 클래스(K-023319-오류 → K-029711, K-016551-오류 → K-016548)의 크기·색으로 짝짓고, 그 클래스로 붙인다.
    - 짝지을 알약을 찾지 못하면, 쓸 수 있는 원본 bbox(이미지 안에 있는 값 4개)는 그대로 둔다.
        - 쓸 수 있는 원본 bbox가 없으면, 오타를 고친 후보 중 너비×높이가 원본 area와 같은 후보가 하나뿐일 때 그 값으로 고친다. ("숫자 하나 고침 (area로 확인)")
        - box가 똑같은 쌍에서 다른 클래스가 사진의 다른 알약으로 옮겨졌으면, 찾지 못한 쪽이 원본 box의 알약이므로 원본 box를 그대로 둔다.
        - 사진으로 확인하지 못한 box이므로, box의 긴 변이 그 클래스 라벨 box 긴 변의 중앙값과 25% 넘게 다르면 쓰지 않는다.
            - 알약을 돌려도 box의 긴 변은 거의 같아서, 같은 클래스의 정상 라벨 box 중 25% 넘게 다른 box는 0.05%이다.
        - 그래도 없으면 그 이미지를 학습에서 뺀다.
- 사진으로 다시 찾을 수 없는 경우는 찾은 알약으로 쓰지 않는다. 처음 만든 갤러리에서 이런 box가 알약이 아닌 자리에 그려졌다.
    - 사진의 두 변에 닿는 덩어리(조명 밖의 밝은 모서리)는 알약으로 보지 않는다.
    - 위 검증에서 IoU 중앙값이 0.85 미만인 클래스(예: 바닥과 색이 같은 파란 캡슐)는 짝짓기에서 빼고 "찾지 못함"과 같은 규칙으로 처리한다.
- 마지막으로 문제 갤러리(셀 82)를 눈으로 보고, 위 규칙의 처리가 틀린 box를 고친다. ("눈으로 확인해 고침")
    - 고친 box와 이유는 셀 79의 `REVIEWED_BOXES`에 적는다.
        - 원본 bbox가 맞음: 짝지은 알약 box가 알약 그림자 쪽을 잘라 원본보다 좁거나, x나 y만 고친 값이 알약을 벗어남
        - 짝지은 알약 box가 맞음, 또는 한쪽 끝만 원본 bbox나 짝지은 알약 box의 선까지 옮김
        - 두 클래스의 라벨이 서로 바뀜 (예: K-010487 ↔ K-029711)
    - box 맞음을 믿을 수 없어 셀 76이 뺀 클래스는 틀린 box가 문제로 잡히지 않는다.
        - 갤러리에서 본 이런 box는 "눈으로 찾은 틀린 box"로 처리 결과에 더해 고친다. (예: 옆 알약을 감싼 K-010487 box)

In [ ]:
def digit_edits(value):
    """숫자 하나에서 숫자 한 개를 빼거나, 넣거나, 바꾼 값을 모두 만든다."""
    text = str(int(value))
    out = set()
    for i in range(len(text) + 1):
        if i < len(text):
            out.add(text[:i] + text[i + 1:])
            out.update(text[:i] + d + text[i + 1:] for d in '0123456789')
        out.update(text[:i] + d + text[i:] for d in '0123456789')
    return {int(t) for t in out if t and not (len(t) > 1 and t[0] == '0')}


def raw_candidates(raw):
    """원본 bbox 값에서 오타를 고친 [x, y, w, h] 후보를 만든다. 이미지 안에 들어가는 후보만 남긴다.

    - 값이 4개인 정수: x, y, w, h 중 하나에서 숫자 한 개를 빼거나, 넣거나, 바꾼 값
    - 값이 3개 이하: 소수점이나 붙은 숫자를 둘로 나눠 값 4개를 만든다. (쉼표가 빠진 것으로 본다. 예: 128.816 → 128, 816)
    """
    tokens = []
    for v in raw:
        tokens += str(v).split('.') if isinstance(v, float) else [str(v)]
    bases = []
    if len(tokens) == 4:
        bases.append([int(t) for t in tokens])
    elif len(tokens) == 3:
        bases += [[int(s) for s in tokens[:i] + [t[:cut], t[cut:]] + tokens[i + 1:]]
                  for i, t in enumerate(tokens) for cut in range(1, len(t))]
    cands = {tuple(b) for b in bases}
    if len(raw) == 4 and all(isinstance(v, int) for v in raw):
        cands |= {tuple(raw[:i] + [v] + raw[i + 1:]) for i in range(4) for v in digit_edits(raw[i])}
    return [list(c) for c in cands if c[2] >= 30 and c[3] >= 30 and c[0] >= 0 and c[1] >= 0
            and c[0] + c[2] <= IMG_W and c[1] + c[3] <= IMG_H]


def choose_box(raw_box, raw_area, found):
    """원본 bbox를 살릴 수 있으면 (box, 처리)를, 없으면 (찾은 box, '사진에서 다시 그림')을 돌려준다."""
    if raw_box and len(raw_box) == 4 and all(isinstance(v, int) for v in raw_box) and raw_box in raw_candidates(raw_box) \
            and box_iou(raw_box, found) >= 0.8:
        return raw_box, '그대로 둠'
    cands = raw_candidates(raw_box) if raw_box else []
    by_area = [c for c in cands if raw_area and c[2] * c[3] == raw_area]
    for pool, threshold in [(by_area, 0.8), (cands, 0.85)]:
        if pool:
            best = max(pool, key=lambda c: box_iou(c, found))
            if box_iou(best, found) >= threshold:
                return best, '숫자 하나 고침'
    return found, '사진에서 다시 그림'


CORNER_GAP = 10  # 알약 범위가 사진의 두 변에서 이 픽셀 안이면 모서리 덩어리로 본다
MAX_COST = 1.5  # 크기·색 차이가 이보다 큰 알약과는 짝짓지 않는다 (정상 라벨 box의 99.5%가 이 안)
LONG_TOLERANCE = 0.25  # 사진으로 확인하지 못한 box는 긴 변이 클래스 중앙값과 이 비율 안일 때만 쓴다
long_values = collections.defaultdict(list)  # K-코드 → 문제가 없는 이미지의 라벨 box 긴 변
for fn in clean_pool:
    for ann in aihub_kaggle[fn]['annotations']:
        long_values[ann['category_id']].append(max(ann['bbox'][2], ann['bbox'][3]))
class_long = {code: float(np.median(v)) for code, v in long_values.items()}  # K-코드 → 긴 변의 중앙값
unfindable_classes = {c for c, v in validation_by_class.items() if np.median(v) < 0.85}  # 위 검증에서 사진으로 다시 찾지 못하는 클래스
# -오류 폴더의 알약은 사진에 다른 알약이 놓였다. 사진으로 확인한 실제 클래스로 짝지어 붙인다 (셀 87)
ERROR_FOLDER_CLASS = {23319: 29711, 16551: 16548}  # -오류 폴더의 K-코드 → 사진 속 실제 클래스 (K-029711 D|CT, K-016548 GB 주황 원형)
error_folder_labels = {(fn, json_code(j)) for fn, js in jsons_of.items() for j in js if j['pill_folder'].endswith('-오류')}  # (이미지, K-코드)
assert {code for _, code in error_folder_labels} <= set(ERROR_FOLDER_CLASS), '실제 클래스를 정하지 않은 -오류 폴더가 있다'


def fix_candidates(rgb, known_boxes):
    """정상 box 밖에서 알약 범위를 찾고, 사진의 두 변에 닿는 모서리 덩어리(조명 밖의 밝은 모서리)는 뺀다."""
    out = []
    for extent in find_pills(rgb, known_boxes):
        x0, y0, x1, y1 = extent
        touches_x = x0 <= CORNER_GAP or x1 >= IMG_W - CORNER_GAP
        touches_y = y0 <= CORNER_GAP or y1 >= IMG_H - CORNER_GAP
        if not (touches_x and touches_y):
            out.append(extent)
    return out


def usable_raw(raw_box):
    """원본 bbox를 고치지 않고 쓸 수 있는지(이미지 안에 있는 정수 4개) 확인한다."""
    return bool(raw_box) and len(raw_box) == 4 and all(isinstance(v, int) for v in raw_box) and raw_box in raw_candidates(raw_box)


def plausible(box, code):
    """box의 긴 변이 그 클래스 라벨 box 긴 변의 중앙값과 LONG_TOLERANCE 안으로 비슷한지 본다. 중앙값이 없는 클래스는 통과시킨다."""
    return code not in class_long or abs(np.log(max(box[2], box[3]) / class_long[code])) <= np.log(1 + LONG_TOLERANCE)


def solve_image(fn):
    """이미지 하나의 문제 annotation을 고친다. 결과는 문제마다 dict 하나이다."""
    rgb = read_rgb(AIHUB_IMG_DIR / fn)
    items = problems[fn]
    labels = [ERROR_FOLDER_CLASS[code] if (fn, code) in error_folder_labels else code for code, *_ in items]  # 붙일 클래스
    bad = {code for code, *_ in items}
    good_boxes = [a['bbox'] for a in aihub_kaggle.get(fn, {'annotations': []})['annotations'] if a['category_id'] not in bad]
    extents = fix_candidates(rgb, good_boxes)
    labs = [extent_color(rgb, e) for e in extents]
    searchable = [i for i, label in enumerate(labels) if label not in unfindable_classes]  # 사진으로 찾을 수 있는 클래스만 짝짓는다
    appear = np.zeros((len(searchable), len(extents)))  # 크기·색 차이
    cost = np.zeros((len(searchable), len(extents)))  # 짝짓기 비용 = 크기·색 차이 - 원본 bbox 후보와의 IoU
    for row, i in enumerate(searchable):
        code, kind, raw_box, raw_area = items[i]
        ref = class_refs.get(labels[i], no_ref)
        for j, extent in enumerate(extents):
            appear[row, j] = appearance_cost(extent, labs[j], ref) if 'long' in ref else 0
            cost[row, j] = appear[row, j]
            if raw_box and '똑같음' not in kind:  # 원본 bbox가 가리키는 알약이면 짝지을 때 우선한다
                cost[row, j] -= max([box_iou(c, with_margin(extent, ref['margin'])) for c in raw_candidates(raw_box)], default=0)
    cost[appear > MAX_COST] = 1e6  # 크기·색이 너무 다른 짝은 만들지 않는다
    results = []
    pairs = zip(*scipy.optimize.linear_sum_assignment(cost)) if extents and searchable else []
    assigned = {searchable[row]: j for row, j in pairs if appear[row, j] <= MAX_COST}
    owner = {j: labels[i] for i, j in assigned.items()}  # 짝지어진 후보 → 그 K-코드 (갤러리에 근거로 적는다)
    for i, (code, kind, raw_box, raw_area) in enumerate(items):
        label = labels[i]
        result = {'file_name': fn, 'category_id': code, 'label_id': label, 'problem': kind, 'raw_box': raw_box, 'raw_area': raw_area,
                  'n_found': len(extents)}
        if i not in assigned:
            if i in searchable and extents:  # 갤러리에 적을 근거: 크기·색이 가장 비슷한 후보, 그 차이, 그 후보와 짝지어진 클래스
                row = searchable.index(i)
                j = int(np.argmin(appear[row]))
                result.update(nearest=float(appear[row, j]), nearest_owner=owner.get(j),
                              nearest_box=with_margin(extents[j], class_refs.get(label, no_ref)['margin']))
            area_matched = [c for c in raw_candidates(raw_box) if c[2] * c[3] == raw_area and plausible(c, label)] if raw_box and raw_area else []
            if usable_raw(raw_box) and plausible(raw_box, label) and '똑같음' not in kind:
                box, action = raw_box, '그대로 둠 (알약을 찾지 못함)'
            elif len(area_matched) == 1 and '똑같음' not in kind:
                box, action = area_matched[0], '숫자 하나 고침 (area로 확인)'
            else:
                box, action = None, '찾지 못함'
            results.append({**result, 'box': box, 'cost': None, 'action': action})
            continue
        found = with_margin(extents[assigned[i]], class_refs.get(label, no_ref)['margin'])
        if '똑같음' in kind:
            box, action = (raw_box, '그대로 둠') if box_iou(raw_box, found) >= 0.5 else (found, '사진에서 다시 그림')
        else:
            box, action = choose_box(raw_box, raw_area, found)
        row = searchable.index(i)
        results.append({**result, 'box': box, 'found': found, 'action': action, 'cost': float(cost[row, assigned[i]]),
                        'appear': float(appear[row, assigned[i]])})
    # box가 똑같은 쌍에서 다른 클래스가 다른 알약으로 옮겨졌으면, 찾지 못한 쪽이 원본 box의 알약이다
    moved = {tuple(r['raw_box']) for r in results if '똑같음' in r['problem'] and r['action'] == '사진에서 다시 그림'}
    for r in results:
        if ('똑같음' in r['problem'] and r['action'] == '찾지 못함' and tuple(r['raw_box']) in moved and usable_raw(r['raw_box'])
                and plausible(r['raw_box'], r['category_id'])):
            r['box'], r['action'] = r['raw_box'], '그대로 둠 (알약을 찾지 못함)'
    return results


fix_cache = CACHE_DIR / '2-2_box_fixes.json'  # 처리마다 붙일 클래스와 판단 근거(찾은 후보 수, 크기·색 차이)도 저장한다
# 문제 목록, 모든 이미지의 box, 위의 클래스 기준값(finder), -오류 폴더의 실제 클래스가 같을 때만 저장한 결과를 쓴다
problem_key = hashlib.sha1(json.dumps([sorted(problems.items()), [[fn, [a['bbox'] for a in r['annotations']]] for fn, r in sorted(aihub_kaggle.items())],
                                      finder, sorted(ERROR_FOLDER_CLASS.items())]).encode()).hexdigest()
if fix_cache.exists() and json.loads(fix_cache.read_text())['key'] == problem_key:
    box_fixes = json.loads(fix_cache.read_text())['fixes']
else:
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
        box_fixes = [r for results in pool.map(solve_image, sorted(problems)) for r in results]
    fix_cache.write_text(json.dumps({'key': problem_key, 'fixes': box_fixes}))

# 아래 문제 갤러리(셀 82)를 눈으로 확인해, 자동 처리가 틀린 box를 고친다 (주석 #번호 = 갤러리 번호)
# (이미지, K-코드) → (고친 box [x, y, w, h] 또는 '원본'(원본 bbox를 그대로 씀), 고친 이유)
REVIEWED_BOXES = {
    ('K-003832-005391-038162_0_2_0_2_90_000_200.png', 5391): ([134, 846, 196, 195], 'K-038162의 원본 bbox 자리가 이 클래스의 알약(PROSCAR)'),  # #41
    ('K-003832-005391-038162_0_2_0_2_90_000_200.png', 38162): ([625, 790, 239, 268], 'K-005391로 짝지은 알약이 이 클래스의 알약'),  # #41
    ('K-000573-001866-019552-038954_0_2_0_2_70_000_200.png', 38954): ([96, 148, 181, 453], '높이가 길어 아래 끝을 짝지은 알약 box까지 줄임'),  # #298
    ('K-000573-002483-005094-019552_0_2_0_2_90_000_200.png', 5094): ([109, 799, 281, 275], '너비가 모자라 오른쪽 끝을 짝지은 알약 box까지 늘림'),  # #300
    ('K-000573-002483-022627-025438_0_2_0_2_75_000_200.png', 2483): ([627, 92, 226, 469], '너비가 넓어 오른쪽 끝을 짝지은 알약 box까지 줄임'),  # #302
    ('K-000573-003743-005002-019552_0_2_0_2_90_000_200.png', 19552): ([70, 148, 310, 492], '높이가 모자라 아래 끝을 짝지은 알약 box까지 늘림'),  # #304
    ('K-001866-005002-005886-022627_0_2_0_2_75_000_200.png', 5002): ([644, 108, 240, 463], '높이가 길어 위 끝을 짝지은 알약 box까지 줄임'),  # #311
    ('K-001900-004543-016548-024850_0_2_0_2_70_000_200.png', 16548): ('원본', '원본 bbox가 맞음'),  # #316
    ('K-003832-033026-036637_0_2_0_2_90_000_200.png', 33026): ([78, 885, 285, 171], '너비가 모자라 오른쪽 끝을 짝지은 알약 box까지 늘림'),  # #375
    ('K-005391-013900-016232_0_2_0_2_90_000_200.png', 13900): ([656, 744, 151, 274], '너비가 넓어 오른쪽 끝을 짝지은 알약 box까지 줄임'),  # #417
    ('K-005391-016688-041768_0_2_0_2_75_000_200.png', 16688): ([118, 514, 273, 645], '너비가 넓어 오른쪽 끝을 짝지은 알약 box까지 줄임'),  # #419
    ('K-006835-019881-023319-041149_0_2_0_2_90_000_200.png', 19881): ('원본', '원본 bbox가 맞음 (짝지은 알약 box는 알약이 잘림)'),  # #448
    ('K-006835-019881-029711-041149_0_2_0_2_75_000_200.png', 19881): ([611, 717, 163, 249], '왼쪽 끝을 원본 bbox까지 늘림 (짝지은 알약 box는 알약이 잘림)'),  # #454
    ('K-011220-016232-025469-034597_0_2_0_2_70_000_200.png', 25469): ([584, 233, 352, 231], '오른쪽으로 옮김 (짝지은 알약 box는 알약 오른쪽이 잘리고 왼쪽 여백이 큼)'),  # #508
    ('K-016232-025469-031885-038723_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #633
    ('K-016235-020238-025367-027653_0_2_0_2_70_000_200.png', 20238): ([165, 860, 230, 234], '짝지은 알약 box가 맞음'),  # #641
    ('K-016235-025469-031885-035206_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #652
    ('K-003832-016688-024752_0_2_0_2_70_000_200.png', 24752): ('원본', '원본 bbox가 맞음'),  # #743
    ('K-003832-022074-024752_0_2_0_2_70_000_200.png', 24752): ([111, 281, 334, 184], '위 끝을 원본 bbox까지 늘림'),  # #744
    ('K-006835-019881-025200-041149_0_2_0_2_90_000_200.png', 19881): ([646, 798, 157, 246], 'x와 너비를 원본 bbox로 (짝지은 알약 box는 알약이 잘림)'),  # #774
    ('K-006835-019881-029711-041149_0_2_0_2_70_000_200.png', 19881): ('원본', '원본 bbox가 맞음 (고친 너비는 알약이 잘림)'),  # #777
    ('K-006835-019881-029711-041149_0_2_0_2_90_000_200.png', 19881): ([648, 809, 159, 242], 'x와 너비를 원본 bbox로 (짝지은 알약 box는 알약이 잘림)'),  # #778
    ('K-010224-016548-023203-027926_0_2_0_2_90_000_200.png', 16548): ([607, 150, 235, 225], '짝지은 알약 box가 맞음'),  # #800
    ('K-010224-016548-023203-027926_0_2_0_2_90_000_200.png', 27926): ([604, 681, 253, 475], '짝지은 알약 box가 맞음'),  # #800
    ('K-010224-016551-023203-033009_0_2_0_2_70_000_200.png', 23203): ('원본', '원본 bbox가 맞음'),  # #802
    ('K-010487-019881-026993-043233_0_2_0_2_75_000_200.png', 26993): ('원본', '원본 bbox가 맞음 (고친 너비는 알약이 잘림)'),  # #804
    ('K-010487-019881-026993-043233_0_2_0_2_75_000_200.png', 10487): ([555, 0, 240, 427], 'box가 옆 알약(K-026993) 위에 있어 오른쪽의 파란 캡슐로 옮김'),  # #804
    ('K-010487-026455-029711-044266_0_2_0_2_75_000_200.png', 29711): ([35, 0, 414, 468], 'K-010487과 라벨이 바뀜: K-010487의 원본 bbox 자리가 이 클래스의 알약(D|CT)'),  # #808
    ('K-010487-026455-029711-044266_0_2_0_2_75_000_200.png', 10487): ([543, 12, 279, 430], 'K-029711과 라벨이 바뀜: K-029711의 원본 bbox 자리가 이 클래스의 알약(파란 캡슐)'),  # #808
    ('K-011220-016232-022347-025469_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #809
    ('K-011220-016232-025367-025469_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #810
    ('K-011220-022347-025469-029667_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #816
    ('K-011220-022347-025469-035206_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #817
    ('K-011220-022347-025469-036637_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #818
    ('K-011220-025367-025469-029667_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #819
    ('K-011220-025469-028763-035206_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #821
    ('K-011220-025469-029667-034597_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #822
    ('K-011220-025469-030308-035206_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #823
    ('K-011220-025469-031885-035206_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #824
    ('K-012247-016551-024850-029451_0_2_0_2_70_000_200.png', 16551): ([50, 710, 291, 486], '왼쪽 끝을 알약까지 늘림 (짝지은 알약 box는 알약 왼쪽이 잘림)'),  # #827
    ('K-012638-019881-025200-041149_0_2_0_2_90_000_200.png', 19881): ([645, 798, 159, 246], 'x와 너비를 원본 bbox로 (짝지은 알약 box는 알약이 잘림)'),  # #846
    ('K-016232-016235-027653-031885_0_2_0_2_70_000_200.png', 27653): ('원본', '원본 bbox가 맞음'),  # #874
    ('K-016232-022347-025469-038723_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #875
    ('K-016232-025367-025469-038723_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #876
    ('K-016232-025367-025469-038723_0_2_0_2_75_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #877
    ('K-016232-025469-028763-038723_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #878
    ('K-016232-025469-030308-038723_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #879
    ('K-016232-025469-031885-038723_0_2_0_2_75_000_200.png', 25469): ([109, 637, 201, 366], '너비가 모자라 오른쪽 끝을 알약까지 늘림'),  # #880
    ('K-016262-025367-025469-038723_0_2_0_2_70_000_200.png', 25469): ([631, 257, 222, 354], '너비가 모자라 오른쪽 끝을 알약까지 늘림'),  # #894
    ('K-016262-025469-030308-038723_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #895
    ('K-016262-025469-034597-038723_0_2_0_2_70_000_200.png', 25469): ('원본', '원본 bbox가 맞음'),  # #896
    ('K-018357-024752-036637_0_2_0_2_70_000_200.png', 24752): ('원본', '원본 bbox가 맞음'),  # #905
}
REVIEW_PROBLEM = '눈으로 찾은 틀린 box'  # box 맞음을 믿을 수 없어 셀 76이 뺀 클래스라 문제로 잡히지 않았지만, 갤러리에서 틀린 것을 본 box
fix_at = {(f['file_name'], f['category_id']): f for f in box_fixes}  # (이미지, K-코드) → 처리 결과
for (fn, code), (box, reason) in REVIEWED_BOXES.items():
    if (fn, code) not in fix_at:  # 문제로 잡히지 않은 box: 원본 라벨 그대로를 자동 처리로 두고 처리 결과에 더한다
        raw_box, raw_area = raw_annotation(fn, code)
        fix_at[(fn, code)] = {'file_name': fn, 'category_id': code, 'label_id': code, 'problem': REVIEW_PROBLEM, 'raw_box': raw_box,
                              'raw_area': raw_area, 'n_found': None, 'box': raw_box, 'cost': None, 'action': '문제로 잡히지 않음'}
        box_fixes.append(fix_at[(fn, code)])
    fix = fix_at[(fn, code)]
    fix.update(auto_box=fix['box'], auto_action=fix['action'], review=reason,
               box=fix['raw_box'] if box == '원본' else box, action='눈으로 확인해 고침')
    assert usable_raw(fix['box']) and fix['box'] != fix['auto_box'], f'{fn} K-{code:06d}: 눈으로 고친 box가 이미지 밖이거나 자동 처리와 같다'

kinds = [k for k, _ in kind_count.most_common()] + [REVIEW_PROBLEM]
actions = ['그대로 둠', '그대로 둠 (알약을 찾지 못함)', '숫자 하나 고침', '숫자 하나 고침 (area로 확인)', '사진에서 다시 그림', '찾지 못함',
           '눈으로 확인해 고침']
table = collections.Counter((f['problem'], f['action']) for f in box_fixes)
rows = [[kind, *[table[(kind, a)] or '-' for a in actions]] for kind in kinds]
rows.append(['**합계**', *[f'**{sum(table[(k, a)] for k in kinds)}**' for a in actions]])
show_table(['문제', *actions], rows)

**인사이트**
- 문제 annotation 1,153개 중 793개(69%)는 원본 값을 살렸다.
    - 그대로 둠 481개(알약을 찾지 못해 그대로 둔 33개 포함), 숫자 하나 고침 312개(area로 확인한 7개 포함)
- area 불일치 427개 중 337개는 bbox를 그대로 두고 area만 다시 계산한다.
- 알약을 감싸지 않는 box 313개 중 156개는 숫자 하나를 고쳐 맞췄다. 좌표 오타가 box를 다른 자리로 밀어낸 경우이다.
- 사진에서 다시 그린 box는 287개이다.
    - 그중 174개는 원본 bbox를 쓸 수 없는 라벨 누락(bbox 빈 값 102개, JSON 없음 47개 등)이다.
- 찾지 못함 21개는 사진에서 알약을 찾지 못했고, 원본 bbox로도 고칠 수 없었다.
- 눈으로 확인해 고친 box는 52개이다. 아래 문제 갤러리에서 자동 처리가 틀린 것을 찾아 고쳤다.

**결론**
- 위 처리 결과를 그대로 정제 목록에 넣는다. (2-4)
- 찾지 못한 21개가 있는 이미지는 학습에서 뺀다. box가 없는 알약은 배경으로 학습되기 때문이다.

- 2-2에서 문제가 있는 이미지를 모두 사진 한 장씩 그려, 문제가 실제로 있는지와 고친 box가 문제를 해결하는지 눈으로 확인한다.
    - 빨간 점선: 원본 bbox (값이 4개일 때만 그린다. 이미지 밖으로 나간 부분은 잘려 보인다)
        - 고친 box와 다르면 오른쪽 아래에 "원본 K-코드"를 적는다.
    - 자홍 점선: "알약을 감싸지 않는 box"로 판단한 근거. 원본 bbox 주변에서 잰 알약 범위에 클래스 여백을 더한 box이다.
        - 셀 76의 box 맞음은 이 box와 원본 bbox의 IoU이다.
    - 주황 점선: 사진에서 찾아 짝지은 알약 box (고친 box와 다를 때만 그린다)
    - 노랑 점선: 짝지을 알약을 찾지 못한 문제에서 크기·색이 가장 비슷한 후보. 그 크기·색 차이를 적는다.
    - 하늘 점선: 눈으로 확인해 고친 문제(셀 79의 `REVIEWED_BOXES`)에서 고치기 전 자동 처리 box (원본 bbox와 같으면 그리지 않는다)
    - 파란 실선: 고친 box. 안쪽 위에 붙인 클래스의 K-코드와 각인을 적는다.
        - `-오류` 폴더의 알약은 사진 속 실제 클래스(셀 79)를 붙이므로 파일명의 K-코드와 다르다.
    - 연두 실선: 같은 이미지에서 문제가 없는 라벨 box. 안쪽 아래에 K-코드를 적는다.
- 제목은 번호와 파일명이다. 고치지 못해 이미지를 빼는 경우 제목을 빨간색으로 쓴다.
- 사진 아래에는 문제 annotation마다 두 줄을 적는다.
    - "K-코드 문제: 문제로 판단한 값": area와 너비×높이, 이미지 밖으로 나간 끝 좌표, box가 똑같은 클래스, box 맞음
    - "→ 처리: 그렇게 정한 근거": 위 규칙의 순서대로 적는다.
        - 짝지은 경우: 크기·색 차이, 원본 bbox나 고친 값과 짝지은 알약 box의 IoU
        - 짝짓지 못한 경우: 먼저 짝짓지 못한 이유(사진으로 찾을 수 없는 클래스, 후보 없음, 가장 비슷한 후보도 차이가 1.5 초과, 그 후보가 다른 클래스와 짝지어짐), 다음에 원본 bbox로 고치지 못한 이유
        - 눈으로 확인해 고친 경우: 고친 이유와 자동 처리 결과
    - 짝짓지 못한 이유는 셀 79가 처리할 때 함께 저장한다. (`runs/eda_raewoo/cache/2-2_box_fixes.json`)
- 이미지는 문제 순서(라벨 누락 → 이미지 밖 → 다른 클래스와 box가 똑같음 → area 불일치 → 알약을 감싸지 않는 box → 눈으로 찾은 틀린 box)로 놓는다.
    - 문제가 여러 개인 이미지는 앞 순서의 문제로 묶는다.
- 시트는 `runs/eda_raewoo/figures_01/2-2_problem_gallery/`에 8장씩 저장하고, 아래에는 첫 시트만 보인다.
    - 번호 → 파일명, K-코드, 문제, 처리, 원본 bbox, 고친 box, 눈으로 고치기 전 자동 처리와 box, 사진 아래 두 줄(`problem_reason`, `action_reason`)의 색인은 같은 폴더의 `index.csv`이다.
    - 이미지 수가 바뀌면 시트 수도 바뀌므로, 저장하기 전에 옛 시트를 지운다.
- 원본 bbox가 정말 틀렸는지, 파란 box가 그 클래스의 알약 하나를 빈틈없이 감싸는지, 사진의 알약이 모두 연두나 파란 box에 들어갔는지 본다.

In [ ]:
PROBLEM_GALLERY_DIR = FIG_DIR / '2-2_problem_gallery'
PROBLEM_ROWS, PROBLEM_COLS = 2, 4  # 시트 한 장에 이미지 8장
PROBLEM_ORDER = ['라벨 누락', '이미지 밖', '다른 클래스와 box가 똑같음', 'area 불일치', '알약을 감싸지 않는 box', REVIEW_PROBLEM]  # 시트에 놓는 순서
TEXT_WIDTH = 92  # 사진 아래 설명의 한 줄 글자 폭 (한글 2칸, 그 밖 1칸)


def problem_rank(problem):
    """문제 이름에 들어 있는 PROBLEM_ORDER 항목 중 가장 앞 순서를 돌려준다."""
    return min(i for i, name in enumerate(PROBLEM_ORDER) if name in problem)


def drawable(box):
    """사진에 그릴 수 있는 xywh box(숫자 4개, 너비와 높이가 양수)인지 확인한다."""
    return bool(box) and len(box) == 4 and all(isinstance(v, (int, float)) for v in box) and box[2] > 0 and box[3] > 0


def problem_reason(fix):
    """문제 annotation을 문제로 판단한 값을 적는다. (2-2 표의 검사, 셀 76의 box 맞음)"""
    fn, code, kind, raw, area = fix['file_name'], fix['category_id'], fix['problem'], fix['raw_box'], fix['raw_area']
    if kind.startswith('라벨 누락'):
        return '원본 bbox 없음' if raw is None else f'원본 bbox {raw}, area {area}'
    if kind == REVIEW_PROBLEM:
        return f'box 맞음 {box_fit[(fn, code)]:.2f}이지만 셀 76이 box 맞음을 믿을 수 없어 뺀 클래스라 문제로 잡히지 않음, 갤러리에서 틀린 box를 봄'
    x, y, w, h = raw
    reasons = []
    if '똑같음' in kind:
        partners = [f'K-{c:06d}' for c, _, b, _ in problems[fn] if b == raw and c != code]
        reasons.append(f"box {raw}가 {', '.join(partners)}의 box와 똑같음")
    if '이미지 밖' in kind:
        if x < 0 or y < 0:
            reasons.append(f'왼쪽 위 ({x}, {y})가 0보다 작음')
        if x + w > IMG_W:
            reasons.append(f'오른쪽 끝 x+너비 {x + w} > {IMG_W}')
        if y + h > IMG_H:
            reasons.append(f'아래 끝 y+높이 {y + h} > {IMG_H}')
    if 'area 불일치' in kind:
        reasons.append(f'area {area} ≠ 너비×높이 {w}×{h}={w * h}')
    if kind == '알약을 감싸지 않는 box':
        reasons.append(f'box 맞음 {box_fit[(fn, code)]:.2f} < {FIT_THRESHOLD} (원본 bbox와 자홍 점선의 IoU)')
    return ', '.join(reasons)


def changed_values(raw, box):
    """원본 bbox에서 고친 값만 '높이 488→48'처럼 적는다. 값이 4개가 아니면(붙은 숫자를 나눔) 앞뒤 값을 모두 적는다."""
    if len(raw) == 4:
        return ', '.join(f'{name} {a}→{b}' for name, a, b in zip(['x', 'y', '너비', '높이'], raw, box) if a != b)
    return f'{raw}→{box}'


def not_found_reason(fix):
    """사진에서 짝지을 알약을 찾지 못한 이유를 셀 79의 판단 순서대로 적는다."""
    code = fix['label_id']
    if code in unfindable_classes:
        return f'사진으로 찾을 수 없는 클래스 (위 검증 IoU 중앙값 {np.median(validation_by_class[code]):.2f} < 0.85)'
    if not fix['n_found']:
        return '사진에서 찾은 알약 후보 없음'
    owner = f" (그 후보는 K-{fix['nearest_owner']:06d} 클래스와 짝지어짐)" if fix['nearest_owner'] is not None else ''
    if fix['nearest'] > MAX_COST:
        return f"가장 비슷한 후보(노랑 점선)도 크기·색 차이 {fix['nearest']:.2f} > {MAX_COST}{owner}"
    if owner:
        return f"크기·색이 맞는 후보(노랑 점선, 차이 {fix['nearest']:.2f})를 다른 클래스가 가져감{owner}"
    return f"크기·색이 비슷한 후보(노랑 점선, 차이 {fix['nearest']:.2f})와 짝짓지 못함"


def fallback_failure(fix):
    """짝을 찾지 못했을 때 원본 bbox로도 고치지 못한 이유를 적는다."""
    code, kind, raw, area = fix['label_id'], fix['problem'], fix['raw_box'], fix['raw_area']
    if not raw:
        return '쓸 원본 bbox도 없음'
    if '똑같음' in kind:
        return '원본 box는 같은 box를 쓰는 다른 클래스의 알약'
    reasons = []
    if not usable_raw(raw):
        reasons.append('원본 bbox가 이미지 안의 값 4개가 아님')
    elif not plausible(raw, code):
        reasons.append(f'원본 긴 변이 클래스 중앙값과 25% 넘게 다름 ({max(raw[2], raw[3])} / {class_long[code]:.0f})')
    n_area = sum(1 for c in raw_candidates(raw) if area and c[2] * c[3] == area and plausible(c, code))
    reasons.append(f'너비×높이가 area와 같은 오타 후보 {n_area}개')
    return ', '.join(reasons)


def action_reason(fix):
    """처리 결과를 그렇게 정한 근거를 셀 79의 규칙 순서대로 적는다."""
    code, kind, raw, area, box, action = (fix[k] for k in ['label_id', 'problem', 'raw_box', 'raw_area', 'box', 'action'])
    if code != fix['category_id']:  # -오류 폴더: 사진 속 실제 클래스로 짝지어 붙였다
        return f'사진 속 실제 클래스(K-{code:06d})로 붙임 (셀 87); ' + action_reason({**fix, 'category_id': code})
    if action == '눈으로 확인해 고침':  # 갤러리를 눈으로 확인해 고쳤다 (셀 79의 REVIEWED_BOXES)
        auto = f"{fix['auto_action']} {fix['auto_box']}" if fix['auto_box'] else fix['auto_action']
        return f"{fix['review']} (자동 처리: {auto})"
    if 'found' in fix:  # 사진에서 찾은 알약과 짝지어졌다 (주황 점선 = 짝지은 알약 box)
        found = fix['found']
        match = f"크기·색 차이 {fix['appear']:.2f}" if code in class_refs else '크기·색 기준값이 없어 차이를 0으로 계산'
        if '똑같음' in kind:
            same = box_iou(raw, found)
            if action == '그대로 둠':
                return f'짝지은 알약({match})이 원본 box 자리(IoU {same:.2f} ≥ 0.5) → 원본 box는 이 클래스'
            return f'짝지은 알약({match})이 원본 box 밖(IoU {same:.2f} < 0.5) → 원본 box는 다른 클래스, 짝지은 알약 box로 바꿈'
        if action == '그대로 둠':
            note = ' → area 값만 틀림' if 'area 불일치' in kind else ''
            return f'짝지은 알약({match}) box와 원본 bbox의 IoU {box_iou(raw, found):.2f} ≥ 0.8{note}'
        if action == '숫자 하나 고침':
            area_text = '너비×높이 = area' if box[2] * box[3] == area else '너비×높이 ≠ area'
            return f'{changed_values(raw, box)} ({area_text}), 짝지은 알약({match}) box와 IoU {box_iou(box, found):.2f}'
        if not raw:
            return f'원본 bbox 없음 → 짝지은 알약({match}) box를 넣음'
        cands = raw_candidates(raw)
        if not cands:
            return f'원본 bbox로 만든 이미지 안의 오타 후보 없음 → 짝지은 알약({match}) box로 바꿈'
        best = max(box_iou(c, found) for c in cands)
        return f'원본과 오타 후보 중 짝지은 알약({match}) box와 맞는 값 없음(가장 높은 IoU {best:.2f}) → 짝지은 알약 box로 바꿈'
    why = not_found_reason(fix)
    if action == '그대로 둠 (알약을 찾지 못함)':
        if '똑같음' in kind:
            return f'{why}; 같은 box의 다른 클래스가 사진의 다른 알약으로 옮겨짐 → 원본 box는 이 클래스'
        median = f'{class_long[code]:.0f}' if code in class_long else '없음'
        return f'{why}; 원본 bbox가 이미지 안의 값 4개이고 긴 변이 클래스 중앙값과 25% 안 ({max(raw[2], raw[3])} / {median}) → 그대로 둠'
    if action == '숫자 하나 고침 (area로 확인)':
        return f'{why}; 너비×높이가 area와 같은 오타 후보가 하나뿐 ({changed_values(raw, box)}, area {area})'
    return f'{why}; {fallback_failure(fix)} → 이 이미지를 학습에서 뺌'


def wrap_text(prefix, text, indent):
    """prefix 뒤에 text를 이어 쓰고, 글자 폭(한글 2칸, 그 밖 1칸)이 TEXT_WIDTH를 넘으면 띄어쓰기에서 줄을 나눈다."""
    lines, line = [], prefix
    for word in text.split(' '):
        candidate = line + word if line.endswith(' ') else f'{line} {word}'
        if sum(2 if unicodedata.east_asian_width(ch) in 'WF' else 1 for ch in candidate) > TEXT_WIDTH and line.strip():
            lines.append(line.rstrip())
            line = indent + word
        else:
            line = candidate
    return lines + [line]


def problem_panel(ax, n, fn, fixes):
    """문제 이미지 한 장에 라벨 box, 원본 bbox, 판단 근거 box, 고친 box를 그리고, 사진 아래에 문제마다 근거와 처리를 적는다."""
    # 사진은 절반 크기로 줄여 그리고, extent로 원본 좌표계에 맞춰 box는 원본 좌표 그대로 그린다
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn).convert('RGB').reduce(2), extent=(0, IMG_W, IMG_H, 0))
    bad = {fix['category_id'] for fix in fixes}
    for ann in aihub_kaggle.get(fn, {'annotations': []})['annotations']:
        if ann['category_id'] not in bad:
            x, y, w, h = ann['bbox']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=1.5))
            ax.text(x + 4, y + h - 4, f"K-{ann['category_id']:06d}", fontsize=6, va='bottom',
                    bbox=dict(facecolor='lime', alpha=0.8, pad=1, linewidth=0), clip_on=True)
    lines = []
    for fix in fixes:
        code = fix['category_id']
        if drawable(fix['raw_box']):
            x, y, w, h = fix['raw_box']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='red', linewidth=2, linestyle='--'))
            # 원본 bbox가 고친 box와 다르고 사진에 걸쳐 있으면, 사진 안에 보이는 오른쪽 아래 모서리에 K-코드를 적는다
            if fix['raw_box'] != fix['box'] and x < IMG_W and y < IMG_H and x + w > 0 and y + h > 0:
                ax.text(min(x + w, IMG_W) - 4, min(y + h, IMG_H) - 4, f'원본 K-{code:06d}', fontsize=6, ha='right', va='bottom',
                        color='white', bbox=dict(facecolor='red', alpha=0.8, pad=1, linewidth=0), clip_on=True)
        if fix['problem'] == '알약을 감싸지 않는 box':  # box 맞음을 잰 box: 원본 bbox 주변에서 잰 알약 범위 + 클래스 여백 (셀 76)
            extent, _ = box_measures[(fn, tuple(fix['raw_box']))]
            if extent:
                x, y, w, h = with_margin(extent, class_refs.get(code, no_ref)['margin'])
                ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='magenta', linewidth=2, linestyle=':'))
        if fix.get('found') and fix['found'] != fix['box']:
            x, y, w, h = fix['found']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='orange', linewidth=2, linestyle=':'))
        if fix.get('nearest_box'):  # 짝짓지 못한 문제: 크기·색이 가장 비슷한 후보
            x, y, w, h = fix['nearest_box']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='yellow', linewidth=2, linestyle=':'))
            ax.text(x + 4, y + h - 4, f"K-{code:06d} 후보 차이 {fix['nearest']:.2f}", fontsize=6, va='bottom',
                    bbox=dict(facecolor='yellow', alpha=0.8, pad=1, linewidth=0), clip_on=True)
        if fix.get('auto_box') and fix['auto_box'] not in (fix['box'], fix['raw_box']):  # 눈으로 고치기 전 자동 처리 box
            x, y, w, h = fix['auto_box']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='cyan', linewidth=2, linestyle='--'))
        if fix['box']:
            x, y, w, h = fix['box']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='blue', linewidth=2.5))
            label_id = fix['label_id']  # 붙인 클래스 (-오류 폴더만 파일명의 K-코드와 다르다)
            label = pill_label(aihub_example[label_id]) if label_id in aihub_example else f'K-{label_id:06d}'
            ax.text(max(x, 0) + 4, max(y, 0) + 4, label, fontsize=7, va='top', color='white',
                    bbox=dict(facecolor='blue', alpha=0.8, pad=1, linewidth=0))
        lines += wrap_text(f"K-{code:06d} {fix['problem']}: ", problem_reason(fix), '    ')
        lines += wrap_text(f"  → {fix['action']}: ", action_reason(fix), '      ')
    ax.set_xlim(0, IMG_W)
    ax.set_ylim(IMG_H, 0)
    excluded = any(fix['action'] == '찾지 못함' for fix in fixes)
    ax.set_title(f'#{n} {fn}' + (' (이미지 제외)' if excluded else ''), fontsize=8, loc='left', color='red' if excluded else 'black')
    ax.set_xlabel('\n'.join(lines), fontsize=7, loc='left')
    ax.set_xticks([])
    ax.set_yticks([])


fixes_by_file = collections.defaultdict(list)  # 이미지 → 그 이미지의 문제 annotation 처리 결과
for fix in box_fixes:
    fixes_by_file[fix['file_name']].append(fix)
problem_files = sorted(fixes_by_file, key=lambda fn: (min(problem_rank(f['problem']) for f in fixes_by_file[fn]), fn))
per_sheet = PROBLEM_ROWS * PROBLEM_COLS
n_problem_sheets = int(np.ceil(len(problem_files) / per_sheet))
PROBLEM_GALLERY_DIR.mkdir(exist_ok=True)
for old_sheet in PROBLEM_GALLERY_DIR.glob('p*.jpg'):  # 이미지 수가 줄어 남는 옛 시트가 없게 지운다
    old_sheet.unlink()
problem_index = []
for page in range(n_problem_sheets):
    chunk = problem_files[page * per_sheet:(page + 1) * per_sheet]
    sheet_name = f'p{page + 1:03d}.jpg'
    fig, axes = plt.subplots(PROBLEM_ROWS, PROBLEM_COLS, figsize=(20, 17.5), layout='constrained')
    for k, (ax, fn) in enumerate(zip(axes.flat, chunk)):
        n = page * per_sheet + k + 1
        problem_panel(ax, n, fn, fixes_by_file[fn])
        problem_index += [{'sheet': sheet_name, 'number': n, 'file_name': fn, 'category_id': fix['category_id'], 'label_id': fix['label_id'],
                           'problem': fix['problem'], 'action': fix['action'], 'bbox_before': json.dumps(fix['raw_box']),
                           'bbox_after': json.dumps(fix['box']), 'found_box': json.dumps(fix.get('found')),
                           'auto_action': fix.get('auto_action', ''), 'bbox_auto': json.dumps(fix.get('auto_box')) if 'auto_box' in fix else '',
                           'problem_reason': problem_reason(fix), 'action_reason': action_reason(fix)}
                          for fix in fixes_by_file[fn]]
    for ax in axes.flat[len(chunk):]:
        ax.axis('off')
    groups = ', '.join(name for i, name in enumerate(PROBLEM_ORDER)
                       if any(min(problem_rank(f['problem']) for f in fixes_by_file[fn]) == i for fn in chunk))
    fig.suptitle(f'2-2 문제 이미지 {page * per_sheet + 1}~{page * per_sheet + len(chunk)}번째 / {len(problem_files)}장 '
                 f'(시트 {page + 1}/{n_problem_sheets}, {groups})\n'
                 '빨간 점선 = 원본 bbox, 자홍 점선 = box 맞음을 잰 알약 범위 + 여백, 주황 점선 = 짝지은 알약 box, '
                 '노랑 점선 = 짝짓지 못한 문제의 가장 비슷한 후보\n'
                 '하늘 점선 = 눈으로 고치기 전 자동 처리 box, 파란 실선 = 고친 box, 연두 = 문제없는 라벨 box, 빨간 제목 = 이미지 제외\n'
                 '사진 아래: 문제마다 "K-코드 문제: 문제로 판단한 값"과 "→ 처리: 그렇게 정한 근거"', fontsize=14)
    fig.savefig(PROBLEM_GALLERY_DIR / sheet_name, dpi=90, bbox_inches='tight')  # 사진 아래 설명이 긴 시트도 아래가 잘리지 않게 한다
    if page:
        plt.close(fig)
with open(PROBLEM_GALLERY_DIR / 'index.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['sheet', 'number', 'file_name', 'category_id', 'label_id', 'problem', 'action',
                                           'bbox_before', 'bbox_after', 'found_box', 'auto_action', 'bbox_auto',
                                           'problem_reason', 'action_reason'])
    writer.writeheader()
    writer.writerows(problem_index)
plt.show()

**인사이트**
- 문제가 있는 이미지는 905장이고, 시트 114장에 나눠 저장했다.
- 첫 시트(라벨 누락 8장)에서 고친 파란 box는 모두 빠졌던 알약 하나를 빈틈없이 감싼다.
    - 짝지은 크기·색 차이는 대부분 0.09~0.39이다. #3은 0.73과 1.31로 기준 1.5 안에 있지만 상대적으로 크다.
- 첫 시트의 사진에서 모든 알약이 연두나 파란 box 안에 들어 있다.

**결론**
- 시트를 눈으로 보고 찾은 틀린 처리 52개를 `REVIEWED_BOXES`에 적어 고친다. (위 처리 표의 "눈으로 확인해 고침")
- 고친 결과로 라벨 문제가 얼마나 남았는지는 아래 2-2 수정 뒤 확인 표에서 센다.

- `-오류` 폴더의 JSON이 왜 오류인지 확인한다.
    - 표: `-오류` 폴더 JSON마다 폴더 이름, drug_N, 제품명, bbox를 본다.
    - 그림: 이미지마다 고치기 전 라벨(연두)과 2-2에서 고친 뒤 라벨(고친 box는 파랑)을 나란히 그린다. 한 줄이 한 조합의 각도별 사진 3장이다.
        - 고친 뒤의 파란 box에는 붙인 클래스를 적는다. `-오류` 폴더의 알약은 셀 79가 사진 속 실제 클래스로 붙인 box이다.
- 그 알약이 사진에 있는지, 다른 알약이 놓였는지, 가려졌는지 본다.

In [ ]:
error_jsons = [j['path'] for j in aihub_jsons_1st if j['pill_folder'].endswith('-오류')]  # aihub_jsons_1st는 경로 순서이다
rows = []
for path in error_jsons:
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except json.JSONDecodeError:
        rows.append([f'`{path.name}`', unicodedata.normalize('NFC', path.parent.name), '(읽을 수 없음)', '', ''])
        continue
    image = data['images'][0]
    rows.append([f'`{path.name}`', unicodedata.normalize('NFC', path.parent.name), image['drug_N'], image['dl_name'],
                 [a.get('bbox') for a in data['annotations']]])
show_table(['JSON', '폴더', 'drug_N', '제품명', 'bbox'], rows)

**인사이트**
- `-오류` 폴더 JSON은 9개이다. K-023319 폴더 2개에 6개, K-016551 폴더 1개에 3개가 있다.
- 읽을 수 있는 8개는 모두 bbox가 `[[]]`로 비어 있다. 나머지 1개는 읽을 수 없는 JSON이다.
- drug_N과 제품명은 폴더 이름의 K-코드와 맞는다. JSON 안의 값으로는 오류 이유를 알 수 없다.

**결론**
- 그 알약이 사진에 있는지 아래 그림으로 확인하고 처리를 정한다.

In [ ]:
error_files = sorted({path.name.replace('.json', '.png') for path in error_jsons})  # 이름순이라 한 조합의 각도별 사진 3장이 이어진다
n_rows = int(np.ceil(len(error_files) / 3))
fig, axes = plt.subplots(n_rows, 6, figsize=(24, 6.6 * n_rows), layout='constrained', squeeze=False)
for k, fn in enumerate(error_files):  # fixes_by_file은 위 문제 갤러리 셀에서 만든다
    image = Image.open(AIHUB_IMG_DIR / fn)
    fixes = fixes_by_file[fn]
    fixed_codes = {fix['category_id'] for fix in fixes}
    # 고치기 전: 쓸 수 있는 원본 라벨 box (-오류 폴더의 bbox []는 그릴 수 없다). 고친 뒤: 문제없는 box는 그대로, 문제 box는 2-2의 처리 결과
    before = [(ann['category_id'], ann['bbox'], 'lime') for ann in aihub_kaggle.get(fn, {'annotations': []})['annotations'] if usable_bbox(ann['bbox'])]
    after = [(code, box, 'lime') for code, box, _ in before if code not in fixed_codes] + [(fix['label_id'], fix['box'], 'blue') for fix in fixes if fix['box']]
    changes = [f"K-{fix['category_id']:06d}" + (f" → K-{fix['label_id']:06d}" if fix['label_id'] != fix['category_id'] else '') + f": {fix['action']}"
               for fix in fixes]
    excluded = any(fix['action'] == '찾지 못함' for fix in fixes)
    for ax, boxes, title, color in [(axes[k // 3, 2 * (k % 3)], before, f'고치기 전: {fn}\nbbox 없음: {missing_text(fn)}', 'black'),
                                    (axes[k // 3, 2 * (k % 3) + 1], after, '고친 뒤\n' + '\n'.join(changes), 'red' if excluded else 'blue')]:
        ax.imshow(image)
        for code, (x, y, w, h), box_color in boxes:
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=box_color, linewidth=2.5))
            label = pill_label(aihub_example[code]) if code in aihub_example else f'K-{code:06d}'
            text_y, va = (y - 8, 'bottom') if y >= 40 else (y + 8, 'top')  # 사진 위 끝에 붙은 box는 이름을 안쪽에 적어 제목을 가리지 않는다
            ax.text(x, text_y, label, fontsize=7, va=va, color='white' if box_color == 'blue' else 'black',
                    bbox=dict(facecolor=box_color, alpha=0.85, pad=1.5, linewidth=0))
        ax.set_title(title, fontsize=8, color=color)
        ax.set_xticks([])
        ax.set_yticks([])
for ax in axes.flat[2 * len(error_files):]:
    ax.axis('off')
fig.suptitle('-오류 폴더 JSON이 있는 이미지의 고치기 전과 뒤 (한 줄 = 한 조합의 각도별 사진 3장, 두 칸씩 왼쪽 = 고치기 전, 오른쪽 = 고친 뒤)\n'
             '연두 = 원본 라벨 box, 파랑 = 2-2에서 고친 box와 붙인 클래스 (-오류 폴더의 알약은 사진 속 실제 클래스로 붙였다)', fontsize=14)
save_fig(fig, '2-2_error_folder')
plt.show()

**인사이트**
- 9장 모두 폴더 이름의 알약이 사진에 없고, 다른 알약이 box 없이 놓여 있다.
    - K-023319(디카맥스디플러스정, 연두 타원형) 자리의 6장에는 흰 D|CT 알약(K-029711)이 있다.
    - K-016551(동아가바펜틴정 800mg, DGTH) 자리의 3장에는 노란 GB 알약(K-016548)이 있다.
- AI Hub는 다른 알약이 놓여 찍힌 사진의 라벨을 `-오류` 폴더로 따로 모은 것으로 보인다.
- 고친 뒤에는 파란 box가 실제 알약을 감싸고, 실제 알약의 클래스가 붙는다.
    - 둘째 줄 70° 사진은 K-019881 box도 옆 캡슐 위에 있었는데, 숫자 하나를 고쳐 Z 알약으로 옮겼다.

**결론**
- `-오류` 폴더 알약은 이미지를 빼지 않고 사진 속 실제 클래스(K-023319 → K-029711, K-016551 → K-016548)로 붙여 쓴다.
- 이 9장은 파일명의 K-코드와 라벨의 클래스가 다르다. 클래스는 파일명이 아니라 정제 라벨에서 읽는다. (2-5)
- K-016551 `-오류` 조합의 사진 3장은 다른 조합과 같은 사진이다. 분할할 때 그 조합과 한 묶음으로 둔다. (1-6)

- 2-2의 처리를 적용한 라벨에 2-1 표(셀 53)의 검사를 다시 해서, 2-1에서 찾은 라벨 문제가 얼마나 풀렸는지 본다.
    - 검사는 셀 53과 같다. 이미지마다 파일명의 K-코드 수와 쓸 수 있는 bbox 수를 맞대어 본다.
    - box가 모자라면 라벨 누락이다. 파일명에 없는 K-코드가 있거나 box가 남으면 클래스 불일치이다.
- 2-2 수정 뒤 라벨은 문제 annotation을 2-2의 처리(`box_fixes`)대로 바꾼 것이다.
    - 처리에 box가 있으면(그대로 둠, 숫자 하나 고침, 사진에서 다시 그림, 눈으로 확인해 고침) 그 box를 쓴다. 원본 bbox가 없던 알약에는 그 box를 넣는다.
    - `-오류` 폴더의 알약은 사진 속 실제 클래스로 붙였다(셀 79). 파일명의 K-코드 자리를 채운 것으로 세고, 표의 마지막 행에 따로 적는다.
    - "찾지 못함"인 알약은 box가 없어서 라벨 누락으로 남는다.
        - 2-1의 라벨 누락이었던 알약은 원래 원인의 행에 남는다.
        - 2-2 표의 좌표 문제나 알약을 감싸지 않는 box였던 알약은 "2-2에서 고치지 못해 box를 뺀 알약" 행에 센다.
- 칸은 단계마다 "객체 수 / 이미지 수"이다. 2-1 표(셀 53) → copy 재배정 뒤(셀 62) → 2-2 수정 뒤 순서이다.
    - 마지막 열은 2-2 수정 뒤에도 남은 이미지이다. 괄호 안은 위 2-2 문제 갤러리의 시트와 번호이다.
- 2-2 수정 뒤에도 남은 이미지는 2-5에서 학습 데이터에서 뺀다.

In [ ]:
def label_issues(labels_of):
    """셀 53과 같은 기준으로 이미지마다 파일명의 K-코드와 라벨을 맞대어 본다.

    Args:
        labels_of (dict): 이미지 파일명 → [(K-코드, bbox를 쓸 수 없는 원인)]. 원인이 None이면 쓸 수 있는 box이다.

    Returns:
        tuple: (라벨 누락 [(이미지, K-코드, 원인)], 파일명에 없는 K-코드 [(이미지, K-코드)], 남는 box [(이미지, K-코드, 남는 수)])
    """
    missing, extra, surplus = [], [], []
    for fn in aihub_photos_1st:
        named = collections.Counter(pill_codes(fn))
        causes_of = collections.defaultdict(list)  # K-코드 → 그 K-코드의 라벨마다 bbox를 쓸 수 없는 원인
        for code, cause in labels_of.get(fn, []):
            causes_of[code].append(cause)
        for code in sorted(set(named) | set(causes_of)):
            causes = causes_of[code]
            if code not in named:
                extra += [(fn, code)] * len(causes)
                continue
            unusable = [cause for cause in causes if cause is not None]
            n_usable = len(causes) - len(unusable)
            # 쓸 수 없는 라벨의 원인을 먼저 붙이고, 라벨이 모자라면 JSON 없음이다 (셀 53)
            missing += [(fn, code, unusable[k] if k < len(unusable) else 'JSON 없음') for k in range(named[code] - n_usable)]
            if n_usable > named[code]:
                surplus.append((fn, code, n_usable - named[code]))
    return missing, extra, surplus


UNFIXED = '2-2에서 고치지 못해 box를 뺀 알약'
fix_of = {(f['file_name'], f['category_id']): f for f in box_fixes}  # (이미지, K-코드) → 2-2의 처리
assert len(fix_of) == len(box_fixes), '2-2의 처리에 같은 (이미지, K-코드)가 두 번 있다'
# 2-1 표(셀 53): 1-3이 넣은 그대로의 JSON (1-5의 jsons_of는 셀 62가 바꾸지 않았다)
labels_21 = {fn: [(json_code(j), bbox_problem(j)) for j in js] for fn, js in jsons_of.items()}
# 2-2 수정 뒤: 처리에 box가 있으면 그 box를 쓰고, 찾지 못한 알약은 원인을 남긴다
labels_22 = collections.defaultdict(list)
for fn, record in aihub_kaggle.items():
    for ann in record['annotations']:
        if (fn, ann['category_id']) not in fix_of:
            assert usable_bbox(ann['bbox']), f"{fn} K-{ann['category_id']:06d}: 쓸 수 없는 bbox가 2-2 표에 없다"
            labels_22[fn].append((ann['category_id'], None))
for (fn, code), fix in fix_of.items():
    if fix['box']:
        labels_22[fn].append((code, None if usable_bbox(fix['box']) else UNFIXED))
    elif fix['problem'].startswith('라벨 누락: '):
        if not fix['problem'].startswith('라벨 누락: JSON 없음'):  # JSON이 없던 알약은 라벨을 두지 않아 다시 JSON 없음으로 센다
            labels_22[fn].append((code, fix['problem'].removeprefix('라벨 누락: ')))
    else:
        labels_22[fn].append((code, UNFIXED))

missing_21, extra_21, surplus_21 = label_issues(labels_21)
assert sorted(m for m in missing_21 if m[:2] not in copy_filled) == sorted(m[:3] for m in missing_labels), \
    '셀 53과 같은 기준으로 다시 센 라벨 누락에서 copy로 채운 알약을 빼면 셀 62의 missing_labels와 같아야 한다'
states = {'2-1 표 (셀 53)': (missing_21, extra_21, surplus_21),
          'copy 재배정 뒤 (셀 62)': (missing_labels, extra_labels, [s[:3] for s in surplus_labels]),
          '2-2 수정 뒤': label_issues(labels_22)}
gallery_of = {row['file_name']: f"{row['sheet']} #{row['number']}" for row in problem_index}  # 이미지 → 2-2 문제 갤러리의 시트와 번호


def count_text(items, weight=lambda item: 1):
    """항목 목록을 '객체 수 / 이미지 수'로 적는다. 없으면 0이다."""
    return f'{sum(map(weight, items))} / {len({item[0] for item in items})}장' if items else '0'


def left_text(items):
    """2-2 수정 뒤에도 남은 이미지를 2-2 문제 갤러리의 시트, 번호와 함께 모두 적는다."""
    return '<br>'.join(f"`{fn}` ({gallery_of.get(fn, '갤러리에 없음')})" for fn in sorted({item[0] for item in items}))


after = states['2-2 수정 뒤']
rows = []
for group in MISSING_GROUPS + [UNFIXED]:
    per_state = [[m for m in missing if missing_group(m) == group] for missing, _, _ in states.values()]
    if any(per_state):
        rows.append([group, *map(count_text, per_state), left_text(per_state[-1])])
rows.append(['**라벨 누락 합계**', *[f'**{count_text(missing)}**' for missing, _, _ in states.values()], ''])
rows.append(['클래스 불일치: 파일명에 없는 K-코드', *[count_text(extra) for _, extra, _ in states.values()], left_text(after[1])])
rows.append(['클래스 불일치: 한 이미지에 같은 K-코드의 box가 파일명보다 많음',
             *[count_text(surplus, weight=lambda s: s[2]) for _, _, surplus in states.values()], left_text(after[2])])
# -오류 폴더: 파일명의 K-코드 자리를 사진 속 실제 클래스로 채웠다 (위 2-2 수정 뒤 열에서는 라벨이 있는 것으로 센다)
relabeled = [(f['file_name'], f['category_id'], f['label_id']) for f in box_fixes if f['box'] and f['label_id'] != f['category_id']]
pairs = collections.Counter((code, label) for _, code, label in relabeled)
rows.append(['참고: `-오류` 폴더 알약을 사진 속 실제 클래스로 붙임', '-', '-', count_text(relabeled),
             '<br>'.join(f'K-{code:06d} → K-{label:06d}: {n}개' for (code, label), n in sorted(pairs.items()))])
show_table(['원인', *states, '2-2 수정 뒤에 남은 이미지 (2-2 문제 갤러리 시트 #번호)'], rows)

**인사이트**
- 라벨 누락은 234개(175장) → copy 재배정 뒤 211개(153장) → 2-2 수정 뒤 21개(20장)로 줄었다.
- 남은 21개는 사진에서 알약을 찾지 못한 경우이다.
    - bbox 빈 값 8개(7장), 숫자가 붙은 bbox 1개(1장), 2-2에서 고치지 못해 box를 뺀 알약 12개(12장)
- 남은 20장 중 17장에 바닥과 색이 비슷한 파란 계열 클래스(K-005391, K-010487, K-012638, K-031885, K-041149)가 있다.
- 클래스 불일치(남는 box)는 0개가 되었다. `-오류` 폴더 알약 9개는 실제 클래스로 붙었다.

**결론**
- 남은 20장은 2-5에서 학습 데이터에서 뺀다. 촬영 PNG 12,009장의 0.17%이다.

### 2-3. 크기·색 대조와 클래스별 crop 갤러리
- 2-2의 수정을 적용한 라벨(`fixed_records`)에서, box 안의 알약이 그 클래스가 맞는지 두 가지로 본다.
    - 크기·색 대조: 이미지마다 box 안의 알약 범위를 재서(2-2의 `refine_extent`), 크기와 색이 2-2의 클래스 기준값과 가장 잘 맞도록 클래스를 다시 배정한다.
        - 다시 배정한 클래스가 라벨과 다르면, 라벨이 서로 바뀌었거나 box가 다른 자리에 있는 이미지 후보이다.
        - 배정 방법은 2-2의 box 둘 지우기 검증과 같다. (클래스 배정 정확도는 2-2 그림 제목)
    - crop 갤러리: 모든 box를 클래스별로 잘라 늘어놓고 눈으로 본다. (아래)
- 2-2 수정 뒤 라벨은 2-2의 처리(`box_fixes`)로 문제 box를 바꾸거나 채운 것이다.
    - 눈으로 확인해 고친 box와, `-오류` 폴더 알약을 사진 속 실제 클래스로 붙인 것도 들어간다.
    - 2-2에서 사진에서 알약을 찾지 못한 이미지(처리가 "찾지 못함")는 뺀다.
- 알약 범위와 색은 2-2의 box 맞음 검사에서 잰 값을 쓴다.

In [ ]:
def drawn_annotation(fn, code, box):
    """원본 annotation이 없던 알약의 annotation을 만든다. 알약 메타데이터는 같은 클래스의 다른 annotation에서,
    이미지 필드(파일명, 촬영 각도 등)는 그 이미지의 것으로 채운다. (00 2-5와 같은 방법)"""
    example = aihub_example[code]
    image_fields = (aihub_kaggle[fn]['image'] if fn in aihub_kaggle
                    else {'file_name': fn, 'imgfile': fn, 'camera_la': int(fn.split('_')[5]), 'id': None})
    image = {**example['image'], **{k: image_fields[k] for k in IMAGE_FIELDS if k in image_fields}}
    return {**example, 'id': None, 'image_id': image.get('id'), 'json_path': None, 'bbox': list(box), 'image': image}


def fixed_annotation(fn, ann, fix):
    """2-2의 처리를 annotation에 적용한다. 원본 annotation이 없거나 사진 속 클래스가 다르면(-오류 폴더) 그 클래스의 메타데이터로 만든다."""
    box = list(fix['box'])
    if ann is None or fix['label_id'] != ann['category_id']:
        return drawn_annotation(fn, fix['label_id'], box)
    return {**ann, 'bbox': box, 'area': box[2] * box[3]}


fixed_records = {}  # 2-2 수정 뒤 라벨: 이미지 → {'image', 'annotations'}
fix_lookup = collections.defaultdict(dict)  # 이미지 → K-코드 → 2-2의 처리
for fix in box_fixes:
    fix_lookup[fix['file_name']][fix['category_id']] = fix
unfound_files = {f['file_name'] for f in box_fixes if f['action'] == '찾지 못함'}
for fn in aihub_files:
    if fn in unfound_files:
        continue
    fixes = fix_lookup[fn]
    anns = aihub_kaggle.get(fn, {'annotations': []})['annotations']
    # 1-3의 annotation 순서를 지켜야 2-2에서 잰 알약 범위(캐시)를 다시 쓴다. 원본 annotation이 없던 알약은 끝에 붙인다
    fixed = [fixed_annotation(fn, a, fixes[a['category_id']]) if a['category_id'] in fixes else a for a in anns]
    fixed += [fixed_annotation(fn, None, fix) for code, fix in fixes.items() if code not in {a['category_id'] for a in anns}]
    assert all(usable_bbox(a['bbox']) for a in fixed), f'{fn}: 2-2 수정 뒤에도 쓸 수 없는 bbox가 있다'
    fixed_records[fn] = {'image': fixed[0]['image'], 'annotations': fixed}


fixed_measures = measure_all(fixed_records)  # 2-2에서 잰 값을 다시 쓰고, 고친 box만 새로 잰다


def appearance_check(fn):
    """크기·색으로 다시 배정한 클래스 순서와 (라벨 그대로의 비용, 가장 맞는 배정의 비용)을 구한다. 대조할 수 없으면 None이다."""
    anns = fixed_records[fn]['annotations']
    codes = [a['category_id'] for a in anns]
    measures = [fixed_measures[(fn, tuple(a['bbox']))] for a in anns]
    if len(anns) < 2 or any(c not in class_refs for c in codes) or any(m[0] is None for m in measures):
        return None
    cost = np.array([[appearance_cost(extent, lab, class_refs[c]) for c in codes] for extent, lab in measures])
    _, order = scipy.optimize.linear_sum_assignment(cost)
    return {'order': [int(j) for j in order], 'label_cost': float(np.trace(cost)), 'best_cost': float(cost[np.arange(len(codes)), order].sum())}


appearance = {fn: appearance_check(fn) for fn in sorted(fixed_records)}
checked = {fn: r for fn, r in appearance.items() if r}
mismatched = sorted((fn for fn, r in checked.items() if r['order'] != list(range(len(r['order'])))),
                    key=lambda fn: -(checked[fn]['label_cost'] - checked[fn]['best_cost']))
show_table(['항목', '이미지 수'], [
    ['크기·색 대조를 한 이미지', f'{len(checked):,}장'],
    ['대조하지 못한 이미지 (알약 1개 이하, 기준값이 없는 클래스, 알약 범위를 못 찾은 box)', f'{len(appearance) - len(checked):,}장'],
    ['다시 배정한 클래스가 라벨과 다른 이미지', f'{len(mismatched):,}장 ({len(mismatched) / len(checked):.2%})'],
])

**인사이트**
- 크기·색으로 클래스를 다시 배정한 이미지 10,589장 중 70장(0.66%)이 라벨과 다르다.
- 1,400장은 대조하지 못했다. 알약이 1개 이하이거나, 기준값이 없는 클래스이거나, 알약 범위를 재지 못한 box가 있는 이미지이다.
    - 이 이미지는 아래 crop 갤러리로만 확인한다.

**결론**
- 라벨과 다른 70장을 아래 시트로 모두 보고, 라벨이 정말 틀린 이미지만 정제 목록에 넣는다. (2-4의 `LABEL_FINDINGS`)

- 다시 배정한 클래스가 라벨과 다른 이미지를 모두 그려 시트로 저장한다. 라벨 비용과 가장 맞는 배정의 비용 차이가 큰 순서이다.
    - 연두 box가 라벨이다. box 옆 글자는 알약 메타데이터의 K-코드, 각인, 색, 모양이다.
        - 크기·색으로 다른 클래스가 배정된 box는 빨간 글자로 `before`(라벨)와 `after`(크기·색으로 다시 배정한 클래스)를 함께 적는다. K-코드를 몰라도 각인, 색, 모양을 사진과 맞대어 볼 수 있다.
        - K-016548(가바토파정 100mg)은 AI Hub와 대회 데이터 모두 각인, 색, 모양 메타데이터가 비어 있어 "메타데이터 없음"으로 적는다.
        - 글자는 box 위, 아래, 안쪽 위 중 다른 글자와 box를 가리지 않는 자리에 적는다.
    - 시트는 `runs/eda_raewoo/figures_01/2-3_appearance_sheets/`에 8장씩 저장하고, 아래에는 첫 시트만 보인다.
- 라벨이 서로 바뀐 것인지, box가 다른 자리에 있는 것인지, 비슷한 알약이라 다시 배정이 틀린 것인지 본다.

In [ ]:
def pill_text(ann):
    """K-코드, 각인, 색, 모양을 한 줄로 적는다. (메타데이터 print_front/print_back, color_class1/2, drug_shape)"""
    info = ann['image']
    marks = '/'.join(info[key] for key in ('print_front', 'print_back') if info.get(key))
    colors = '/'.join(info[key] for key in ('color_class1', 'color_class2') if info.get(key))
    if not (marks or colors or info.get('drug_shape')):  # K-016548은 AI Hub와 대회 데이터 모두 메타데이터가 비어 있다
        return f"K-{ann['category_id']:06d}, 메타데이터 없음 ({info.get('dl_name')})"
    return f"K-{ann['category_id']:06d}, {marks or '각인 없음'}, {colors}, {info.get('drug_shape')}"


def place_label(ax, box, text, color, taken, others):
    """글자를 box 위, 아래, 안쪽 위 중 사진 안이면서 다른 글자, 다른 box와 겹치지 않는 첫 자리에 적는다.
    그런 자리가 없으면 글자끼리만 겹치지 않는 자리, 그것도 없으면 첫 자리에 적는다."""
    renderer = ax.figure.canvas.get_renderer()
    frame = ax.get_window_extent(renderer)
    x, y, w, h = box
    near = 'right' if x + w / 2 > IMG_W / 2 else 'left'  # 오른쪽 알약은 글자를 box 오른쪽 끝에 맞춘다
    far = 'left' if near == 'right' else 'right'
    spots = [(row, ha) for row in [(y - 6, 'bottom'), (y + h + 6, 'top'), (y + 6, 'top')] for ha in (near, far)]
    tried = []
    for (text_y, va), ha in spots:
        label = ax.text(x if ha == 'left' else x + w, text_y, text, fontsize=9, va=va, ha=ha, color=color,
                        bbox=dict(facecolor='white', alpha=0.85, pad=1.5, linewidth=0))
        extent = label.get_window_extent(renderer).padded(3)
        inside = frame.x0 <= extent.x0 and extent.x1 <= frame.x1 and frame.y0 <= extent.y0 and extent.y1 <= frame.y1
        free = not any(extent.overlaps(t) for t in taken)
        tried.append((label, extent, inside and free, inside and free and not any(extent.overlaps(o) for o in others)))
        label.set_visible(False)
    label, extent, _, _ = next((t for t in tried if t[3]), None) or next((t for t in tried if t[2]), tried[0])
    for other, *_ in tried:
        if other is not label:
            other.remove()
    label.set_visible(True)
    taken.append(extent)


APPEAR_DIR = FIG_DIR / '2-3_appearance_sheets'
APPEAR_DIR.mkdir(exist_ok=True)
for old in APPEAR_DIR.glob('*.jpg'):
    old.unlink()
for start in range(0, len(mismatched), 8):
    chunk = mismatched[start:start + 8]
    fig, axes = plt.subplots(2, 4, figsize=(24, 17), layout='constrained')
    patches = {}
    for ax, fn in zip(axes.flat, chunk):
        ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
        for i, ann in enumerate(fixed_records[fn]['annotations']):
            x, y, w, h = ann['bbox']
            patches[(fn, i)] = ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2))
        n = mismatched.index(fn) + 1
        ax.set_title(f"#{n} {fn}\n비용: 라벨 {checked[fn]['label_cost']:.2f} → 다시 배정 {checked[fn]['best_cost']:.2f}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
    for ax in axes.flat[len(chunk):]:
        ax.axis('off')
    fig.suptitle(f'크기·색으로 다시 배정한 클래스가 라벨과 다른 이미지 {start + 1}~{start + len(chunk)} / {len(mismatched)}장 '
                 '(연두 = 라벨 box, 글자 = K-코드, 각인, 색, 모양, 빨간 글자 = 다르게 배정된 box의 before 라벨 → after 크기·색으로 다시 배정한 클래스)',
                 fontsize=14)
    fig.draw_without_rendering()  # 배치를 먼저 정하고 고정해야 글자 자리를 잴 수 있다
    fig.set_layout_engine('none')
    renderer = fig.canvas.get_renderer()
    for ax, fn in zip(axes.flat, chunk):
        anns = fixed_records[fn]['annotations']
        order = checked[fn]['order']
        taken = []
        for i in sorted(range(len(anns)), key=lambda i: order[i] == i):  # 다르게 배정된 box(긴 글자)부터 자리를 잡는다
            changed = order[i] != i
            text = f"before: {pill_text(anns[i])}\n→ after: {pill_text(anns[order[i]])}" if changed else pill_text(anns[i])
            others = [patches[(fn, k)].get_window_extent(renderer) for k in range(len(anns)) if k != i]
            place_label(ax, anns[i]['bbox'], text, 'red' if changed else 'black', taken, others)
    fig.savefig(APPEAR_DIR / f'2-3_appearance_p{start // 8 + 1:02d}.jpg', dpi=90)
    if start:
        plt.close(fig)
plt.show()

**인사이트**
- 첫 시트 8장 중 5장은 같은 사진 안에서 두 알약의 라벨이 서로 바뀌었다.
    - #1, #2: K-019881(Z분할선Z, 하양 장방형)과 K-006835(CKD, 주황 원형)
    - #4: K-029667과 K-030308, #5: K-016262와 K-024752, #6: K-044266(BTS 330)과 K-013004(Igatan F)
- #3, #7, #8(조합 `K-016548-016551-023203-024850`)은 라벨이 서로 바뀐 것이 아니라 파일명의 알약이 사진에 없다.
    - 사진에 K-016551(DGTH)은 없고 Airtal 알약(K-004543)이 있다. 크기·색으로 다시 배정한 결과도 틀렸다.
    - 이 조합의 사진은 조합 `K-004543-016548-023203-024850`과 같은 사진이다. (1-6)
- 라벨이 서로 바뀐 #1, #2는 비용이 8.0~8.2에서 0.7~1.1로 크게 줄었다.

**결론**
- 시트에서 확인한 오류는 2-4의 `LABEL_FINDINGS`에 적어 고친다.
    - 서로 바뀐 라벨은 맞바꾸고, 파일명에 없는 알약은 그 알약의 클래스로 바꾼다.
- 크기·색 배정 결과를 그대로 쓰지 않고, 사진으로 확인한 것만 고친다. #3처럼 배정도 틀릴 수 있다.

- crop 갤러리: 2-2의 수정을 적용한 라벨(`fixed_records`)로, 모든 bbox를 클래스별로 잘라 시트에 늘어놓는다.
    - crop은 box 중심에서 긴 변보다 10% 큰 정사각형을 잘라 96×96으로 줄인 것이다. (00 2-3과 같은 `crop_square`) 노란 선이 라벨 box이다.
    - 클래스마다 box 넓이 순서로 놓는다. 크기가 튀는 box(헐거운 box, 옆 알약까지 감싼 box)는 시트의 처음이나 끝에 모인다.
    - 대회 train에도 있는 클래스는 첫 시트 첫 줄(빨간 테두리)에 대회 train crop을 놓는다. drug_N으로 매긴 클래스가 대회의 같은 알약인지 대조한다.
- 시트 한 장은 10×10 = 100칸이다. 시트는 `runs/eda_raewoo/figures_01/2-3_gallery/`에 `K-<코드>_p<쪽>.jpg`로 저장한다.
    - 칸마다 클래스 안 번호를 적는다. 번호로 이미지와 box를 찾는 색인은 같은 폴더의 `index.csv`이다.
- 1·2차 확인은 Claude Code가 시트를 모두 보고 하고, 애매한 칸은 사람이 확인한다. (00 5-2와 같은 분담)

In [ ]:
GALLERY_DIR = FIG_DIR / '2-3_gallery'
GALLERY_DIR.mkdir(exist_ok=True)
THUMB, COLS, ROWS = 96, 10, 10
gallery_items = collections.defaultdict(list)  # K-코드 → [(이미지, bbox)] box 넓이 순서
for fn, record in fixed_records.items():
    for ann in record['annotations']:
        gallery_items[ann['category_id']].append((fn, ann['bbox']))
for code in gallery_items:
    gallery_items[code].sort(key=lambda item: (item[1][2] * item[1][3], item[0]))
gallery_key = hashlib.md5(json.dumps(sorted(gallery_items.items())).encode()).hexdigest()
index_path = GALLERY_DIR / 'index.csv'
font_path = font_manager.findfont(plt.rcParams['font.family'][0])


def thumb_bytes(job):
    """이미지 하나에서 crop을 모두 잘라 JPEG 바이트로 돌려준다. (메모리를 아끼려고 압축해 둔다)"""
    folder, fn, boxes = job
    image = Image.open(folder / fn).convert('RGB')
    out = {}
    for box in boxes:
        x, y, w, h = box
        side = max(w, h) * 1.1  # crop_square와 같은 정사각형 (긴 변 + 10%)
        scale = THUMB / side
        thumb = crop_square(image, box, size=THUMB)
        left, top = THUMB / 2 - w * scale / 2, THUMB / 2 - h * scale / 2  # box는 정사각형 가운데에 온다
        ImageDraw.Draw(thumb).rectangle([left, top, left + w * scale, top + h * scale], outline=(255, 230, 0), width=1)
        buffer = io.BytesIO()
        thumb.save(buffer, format='JPEG', quality=90)
        out[(fn, tuple(box))] = buffer.getvalue()
    return out


def class_header(code, n_items, page, n_pages):
    info = (aihub_example.get(code) or class_example[code])['image']
    marks = '/'.join(m for m in (info.get('print_front'), info.get('print_back')) if m) or '각인 없음'
    return (f"K-{code:06d} {info['dl_name'][:28]} | 각인 {marks} | {info.get('color_class1')} {info.get('drug_shape')} | "
            f"AI Hub {n_items}개 | {page}/{n_pages}쪽")


cached = index_path.exists() and (GALLERY_DIR / 'key.txt').exists() and (GALLERY_DIR / 'key.txt').read_text() == gallery_key
if not cached:
    jobs = collections.defaultdict(list)
    for code, items in gallery_items.items():
        for fn, box in items:
            jobs[(AIHUB_IMG_DIR, fn)].append(box)
    for code in set(gallery_items) & set(train_classes):  # 대회 train crop (정제 라벨)
        for fn, record in clean_records.items():
            for ann in record['annotations']:
                if ann['category_id'] == code:
                    jobs[(TRAIN_IMG_DIR, fn)].append(ann['bbox'])
    thumbs_jpeg = {}
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
        for out in pool.map(thumb_bytes, [(folder, fn, boxes) for (folder, fn), boxes in jobs.items()]):
            thumbs_jpeg.update(out)
    font = ImageFont.truetype(font_path, 13)
    small_font = ImageFont.truetype(font_path, 11)
    index_rows = []
    for old in GALLERY_DIR.glob('K-*.jpg'):
        old.unlink()
    for code, items in sorted(gallery_items.items()):
        kaggle = [(fn, a['bbox']) for fn, r in sorted(clean_records.items()) for a in r['annotations'] if a['category_id'] == code][:COLS]
        first_capacity = COLS * (ROWS - 1) if kaggle else COLS * ROWS
        pages = [items[:first_capacity]] + [items[i:i + COLS * ROWS] for i in range(first_capacity, len(items), COLS * ROWS)]
        for p, page_items in enumerate(pages, start=1):
            sheet = Image.new('RGB', (COLS * THUMB, 24 + ROWS * (THUMB + 14)), 'white')
            draw = ImageDraw.Draw(sheet)
            draw.text((4, 4), class_header(code, len(items), p, len(pages)), fill='black', font=font)
            first_row = kaggle if p == 1 else []  # 첫 시트 첫 줄은 대회 train crop
            start_no = 0 if p == 1 else first_capacity + (p - 2) * COLS * ROWS
            for col, (fn, box) in enumerate(first_row):
                x0, y0 = col * THUMB, 24
                sheet.paste(Image.open(io.BytesIO(thumbs_jpeg[(fn, tuple(box))])), (x0, y0))
                draw.rectangle([x0, y0, x0 + THUMB - 1, y0 + THUMB - 1], outline='red', width=3)
                draw.text((x0 + 2, y0 + THUMB), '대회', fill='red', font=small_font)
            for k, (fn, box) in enumerate(page_items):
                row, col = divmod(k + (COLS if first_row else 0), COLS)
                x0, y0 = col * THUMB, 24 + row * (THUMB + 14)
                sheet.paste(Image.open(io.BytesIO(thumbs_jpeg[(fn, tuple(box))])), (x0, y0))
                number = start_no + k + 1
                draw.text((x0 + 2, y0 + THUMB), f'{number}', fill='black', font=small_font)
                index_rows.append({'sheet': f'K-{code:06d}_p{p:02d}.jpg', 'number': number, 'category_id': code,
                                   'file_name': fn, 'bbox': json.dumps(box)})
            sheet.save(GALLERY_DIR / f'K-{code:06d}_p{p:02d}.jpg', quality=85)
    with open(index_path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['sheet', 'number', 'category_id', 'file_name', 'bbox'])
        writer.writeheader()
        writer.writerows(index_rows)
    (GALLERY_DIR / 'key.txt').write_text(gallery_key)
with open(index_path, encoding='utf-8') as f:
    gallery_index = list(csv.DictReader(f))
n_sheets = len(list(GALLERY_DIR.glob('K-*.jpg')))
show_table(['항목', '값'], [
    ['crop 수 (AI Hub, 2-2 수정 후)', f"{len(gallery_index):,}개 (이미지 {len(fixed_records):,}장)"],
    ['클래스 수', f'{len(gallery_items)}개 (대회 train crop 줄이 있는 클래스 {len(set(gallery_items) & set(train_classes))}개)'],
    ['시트 수', f'{n_sheets}장 (클래스당 {min(len(v) for v in gallery_items.values()) // 100 + 1}~{max(len(v) for v in gallery_items.values()) // 100 + 1}장)'],
    ['저장 위치', f'`{GALLERY_DIR.relative_to(PROJECT_ROOT)}`'],
])

- 시트 예시로 대회 train에도 있는 K-001900(보령부스파정 5mg)의 첫 시트를 보인다. (1-2의 예시 알약)

In [ ]:
example_sheet = Image.open(GALLERY_DIR / 'K-001900_p01.jpg')
fig, ax = plt.subplots(figsize=(14, 14 * example_sheet.height / example_sheet.width), layout='constrained')
ax.imshow(example_sheet)
ax.axis('off')
ax.set_title('crop 갤러리 시트 예시 (첫 줄 빨간 테두리 = 대회 train crop, 아래 = AI Hub crop을 box 넓이 순서로, 칸 아래 숫자 = 색인 번호)', fontsize=12)
save_fig(fig, '2-3_gallery_example')
plt.show()

**인사이트**
- crop 45,879개(11,989장, 116개 클래스)를 시트 512장에 늘어놓았다. 클래스당 1~14장이다.
- 예시 K-001900 시트 1쪽의 AI Hub crop 90개는 모두 BSP 각인의 흰 장방형 알약이고, 첫 줄의 대회 crop과 같은 알약이다.
    - 노란 box가 알약을 빈틈없이 감싼다. 넓이 순서의 처음과 끝에도 헐거운 box가 보이지 않는다.
- 대회 train과 같은 클래스 54개는 첫 줄에 대회 crop이 있어, drug_N으로 매긴 클래스가 대회의 같은 알약인지 바로 대조할 수 있다.

**결론**
- 시트에서 찾은 잘못된 클래스와 틀린 box를 2-4의 `LABEL_FINDINGS`에 적어 고친다. (클래스 바꿈 47개, box 다시 그림 17개)
- drug_N으로 매긴 클래스를 대회 train의 클래스 번호와 그대로 합친다. 알약 정보도 대회 값과 같다. (2-1)

### 2-4. 라벨 정제 목록
- 2-1~2-3에서 찾은 라벨 오류를 annotation 단위로 모으고 처리 방법을 정한다.
    - 2-1: copy 파일을 다시 배정해 바꾸거나 채우거나 남는 box를 지운 알약이다. (셀 62) 같은 알약의 2-2 수정보다 앞에 적는다.
    - 2-1, 2-2: 2-2의 처리 결과(`box_fixes`)이다. 눈으로 확인해 고친 box와, `-오류` 폴더 알약을 사진 속 실제 클래스로 붙인 것도 들어간다.
    - 2-3: crop 갤러리 시트에서 눈으로 확인한 오류이다. (이미지, K-코드)마다 상수(`LABEL_FINDINGS`)로 적는다.
        - 갤러리에서 찾은 칸은 시트 이름과 칸 번호로 `index.csv`에서 이미지를 찾는다. 주석의 시트와 번호가 그 칸이다.
        - 같은 사진 안에서 두 알약의 라벨이 서로 바뀌었으면 두 라벨을 맞바꾼다.
        - 사진 속 알약이 파일명에 없는 약이면 그 약의 클래스로 바꾼다. (`-오류` 폴더와 같은 방법)
        - box가 틀린 알약(다른 자리, 알약보다 너무 큼, 알약을 다 감싸지 않음)은 사진에서 알약 범위를 재고 클래스 여백(2-2의 `class_refs`)을 더해 다시 그린다.
            - 2-2의 자동 처리로 다시 그리면 같은 실수를 되풀이하므로, 잰 box를 눈으로 확인해 상수로 적는다.
            - box 맞음을 믿을 수 없는 클래스(셀 76의 `unreliable_classes`, 파란 알약 등)는 클래스 여백 대신 전체 여백 중앙값을 더한다.
- 정제 목록은 00과 같은 열로 `notebooks/eda/01_raewoo_aihub_label_fixes.csv`에 저장한다. 바뀌는 annotation 하나가 한 줄이고, 목록 순서대로 적용한다.
    - `category_id`: 1-3의 K-코드(파일명의 K-코드)이다. 같은 알약의 줄은 모두 이 값으로 찾는다.
    - `bbox_before`: 원본 JSON의 bbox 값 그대로이다. JSON이 없으면 비어 있고, 빈 값과 숫자가 붙은 값도 그대로 적는다.
        - copy 재배정에서 1-3이 같은 K-코드의 JSON을 여러 개 넣은 사진은 그 bbox 목록이다.
    - `bbox_after`: 고친 `[x, y, w, h]` 픽셀이다. 이미지를 빼는 줄은 비어 있다.
    - `area_before`, `area_after`: 원본 area와 너비×높이로 다시 계산한 area이다.
    - `category_id_after`: 클래스를 바꾸는 줄(2-2의 `-오류` 폴더 알약, 2-3의 라벨 바뀜)에만 적는다.
    - `evidence`: 근거가 된 절 번호이다.

In [ ]:
# 2-3 crop 갤러리를 눈으로 확인해 찾은 라벨 오류 (주석 = 2-3 갤러리의 시트와 칸 번호)
# (이미지, K-코드) → (문제, 바꿀 K-코드 또는 None, 다시 그린 box [x, y, w, h] 또는 None(box 그대로))
LABEL_FINDINGS = {
    # 같은 사진 안에서 라벨이 서로 바뀜
    ('K-001900-003544-006563-016548_0_2_0_2_90_000_200.png', 3544): ('같은 사진의 K-001900 라벨과 바뀜 (BSP 알약)', 1900, None),  # K-003544_p01 #1
    ('K-001900-003544-006563-016548_0_2_0_2_75_000_200.png', 3544): ('같은 사진의 K-001900 라벨과 바뀜 (BSP 알약)', 1900, None),  # K-003544_p01 #2
    ('K-001900-003544-006563-016548_0_2_0_2_90_000_200.png', 1900): ('같은 사진의 K-003544 라벨과 바뀜 (OG3 알약, 피드백의 짝)', 3544, None),  # K-001900_p02 #162 (피드백 짝)
    ('K-001900-003544-006563-016548_0_2_0_2_75_000_200.png', 1900): ('같은 사진의 K-003544 라벨과 바뀜 (OG3 알약, 피드백의 짝)', 3544, None),  # K-001900_p02 #161 (피드백 짝)
    ('K-006835-019881-020004-023319_0_2_0_2_75_000_200.png', 6835): ('같은 사진의 K-019881 라벨과 바뀜 (Z|Z 알약)', 19881, None),  # K-006835_p01 #1
    ('K-006835-019881-020004-023319_0_2_0_2_70_000_200.png', 6835): ('같은 사진의 K-019881 라벨과 바뀜 (Z|Z 알약)', 19881, None),  # K-006835_p01 #2
    ('K-006835-019881-020004-023319_0_2_0_2_75_000_200.png', 19881): ('같은 사진의 K-006835 라벨과 바뀜 (CKD 알약)', 6835, None),  # K-019881_p04 #359
    ('K-006835-019881-020004-023319_0_2_0_2_70_000_200.png', 19881): ('같은 사진의 K-006835 라벨과 바뀜 (CKD 알약)', 6835, None),  # K-019881_p04 #360
    ('K-013004-026993-038972-044266_0_2_0_2_75_000_200.png', 13004): ('같은 사진의 K-044266 라벨과 바뀜 (BTS 330 캡슐)', 44266, None),  # K-013004_p04 #344
    ('K-013004-026993-038972-044266_0_2_0_2_75_000_200.png', 44266): ('같은 사진의 K-013004 라벨과 바뀜 (이가탄 캡슐)', 13004, None),  # K-044266_p01 #74
    ('K-011220-016262-022347-025469_0_2_0_2_75_000_200.png', 16262): ('같은 사진의 K-022347 라벨과 바뀜 (112 자누비아정)', 22347, None),  # K-016262_p01 #1
    ('K-011220-016262-022347-025469_0_2_0_2_75_000_200.png', 22347): ('같은 사진의 K-016262 라벨과 바뀜 (ZD4522 크레스토정)', 16262, None),  # K-022347_p05 #468
    ('K-003832-016262-024752_0_2_0_2_75_000_200.png', 16262): ('같은 사진의 K-024752 라벨과 바뀜 (FS 알약)', 24752, None),  # K-016262_p09 #795
    ('K-003832-016262-024752_0_2_0_2_75_000_200.png', 24752): ('같은 사진의 K-016262 라벨과 바뀜 (ZD4522 크레스토정)', 16262, None),  # K-024752_p01 #5
    ('K-016206-016262-020014_0_2_0_2_75_000_200.png', 16262): ('같은 사진의 K-020014 라벨과 바뀜 (마도파정)', 20014, None),  # K-016262_p09 #796
    ('K-016206-016262-020014_0_2_0_2_75_000_200.png', 20014): ('같은 사진의 K-016262 라벨과 바뀜 (ZD4522 크레스토정)', 16262, None),  # K-020014_p01 #2
    ('K-016206-020014-020238_0_2_0_2_70_000_200.png', 20014): ('같은 사진의 K-020238 라벨과 바뀜 (75 플라빅스정)', 20238, None),  # K-020014_p01 #1
    ('K-016206-020014-020238_0_2_0_2_70_000_200.png', 20238): ('같은 사진의 K-020014 라벨과 바뀜 (마도파정)', 20014, None),  # K-020238_p07 #620
    ('K-016206-020238-032310_0_2_0_2_70_000_200.png', 20238): ('같은 사진의 K-032310 라벨과 바뀜 (글리아타민 연질캡슐)', 32310, None),  # K-020238_p07 #619
    ('K-016206-020238-032310_0_2_0_2_70_000_200.png', 32310): ('같은 사진의 K-020238 라벨과 바뀜 (75 플라빅스정)', 20238, None),  # K-032310_p01 #1
    ('K-006835-024941-030850-053384_0_2_0_2_70_000_200.png', 24941): ('같은 사진의 K-053384 라벨과 바뀜 (CT/LC 디카테오엘씨정)', 53384, None),  # K-024941_p04 #359
    ('K-006835-024941-030850-053384_0_2_0_2_70_000_200.png', 53384): ('같은 사진의 K-024941 라벨과 바뀜 (TG|L 액티프롤정)', 24941, None),  # K-053384_p01 #1
    ('K-016235-025367-027777-029667_0_2_0_2_90_000_200.png', 27777): ('같은 사진의 K-029667 라벨과 바뀜 (Kowa 203 리바로정)', 29667, None),  # K-027777_p05 #467
    ('K-016235-025367-027777-029667_0_2_0_2_90_000_200.png', 29667): ('같은 사진의 K-027777 라벨과 바뀜 (FMS6 카나브정)', 27777, [143, 811, 180, 211]),  # K-029667_p01 #1
    ('K-003614-027733-029667-030308_0_2_0_2_70_000_200.png', 29667): ('같은 사진의 K-030308 라벨과 바뀜 (D2/850 트라젠타듀오정)', 30308, None),  # K-029667_p07 #611
    ('K-003614-027733-029667-030308_0_2_0_2_70_000_200.png', 30308): ('같은 사진의 K-029667 라벨과 바뀜 (Kowa 203 리바로정)', 29667, None),  # K-030308_p01 #1
    # 사진 속 알약이 파일명에 없는 약 (그 약의 클래스로 바꾼다)
    ('K-016548-016551-023203-024850_0_2_0_2_75_000_200.png', 16548): ('사진 속 알약이 K-004543(Airtal 에어탈정)', 4543, None),  # K-016548_p01 #1
    ('K-016548-016551-023203-024850_0_2_0_2_90_000_200.png', 16548): ('사진 속 알약이 K-004543(Airtal 에어탈정)', 4543, None),  # K-016548_p01 #2
    ('K-016548-016551-023203-024850_0_2_0_2_70_000_200.png', 16548): ('사진 속 알약이 K-004543(Airtal 에어탈정)', 4543, None),  # K-016548_p01 #3
    ('K-016548-016551-023203-024850_0_2_0_2_75_000_200.png', 16551): ('사진 속 알약이 K-016548(GB 가바토파정)', 16548, None),  # K-016551_p01 #1
    ('K-016548-016551-023203-024850_0_2_0_2_90_000_200.png', 16551): ('사진 속 알약이 K-016548(GB 가바토파정)', 16548, None),  # K-016551_p01 #2
    ('K-016548-016551-023203-024850_0_2_0_2_70_000_200.png', 16551): ('사진 속 알약이 K-016548(GB 가바토파정)', 16548, None),  # K-016551_p01 #3
    ('K-011220-016232-027733-031885_0_2_0_2_75_000_200.png', 16232): ('사진 속 알약이 K-029667(Kowa 203 리바로정)', 29667, None),  # K-016232_p09 #802
    ('K-011220-016232-027733-031885_0_2_0_2_90_000_200.png', 16232): ('사진 속 알약이 K-029667(Kowa 203 리바로정)', 29667, None),  # K-016232_p09 #803
    ('K-011220-016232-027733-031885_0_2_0_2_70_000_200.png', 16232): ('사진 속 알약이 K-029667(Kowa 203 리바로정)', 29667, None),  # K-016232_p09 #804
    ('K-016206-018147-022074_0_2_0_2_70_000_200.png', 22074): ('사진 속 알약이 K-033026(JW|TH4 트루패스정)', 33026, None),  # K-022074_p02 #173
    ('K-016206-018147-022074_0_2_0_2_90_000_200.png', 22074): ('사진 속 알약이 K-033026(JW|TH4 트루패스정)', 33026, None),  # K-022074_p02 #175
    ('K-016206-018147-022074_0_2_0_2_75_000_200.png', 22074): ('사진 속 알약이 K-033026(JW|TH4 트루패스정)', 33026, None),  # K-022074_p02 #177
    ('K-015280-018147-022074_0_2_0_2_70_000_200.png', 22074): ('사진 속 알약이 K-033026(JW|TH4 트루패스정)', 33026, None),  # K-022074_p02 #172
    ('K-015280-018147-022074_0_2_0_2_90_000_200.png', 22074): ('사진 속 알약이 K-033026(JW|TH4 트루패스정)', 33026, None),  # K-022074_p02 #174
    ('K-015280-018147-022074_0_2_0_2_75_000_200.png', 22074): ('사진 속 알약이 K-033026(JW|TH4 트루패스정)', 33026, None),  # K-022074_p02 #176 (Claude가 찾음)
    ('K-003614-027777-028763-030308_0_2_0_2_75_000_200.png', 30308): ('사진 속 알약이 K-016262(ZD4522 크레스토정)', 16262, None),  # K-030308_p01 #2
    ('K-003614-027777-028763-030308_0_2_0_2_90_000_200.png', 30308): ('사진 속 알약이 K-016262(ZD4522 크레스토정)', 16262, None),  # K-030308_p01 #3
    ('K-003614-027777-028763-030308_0_2_0_2_70_000_200.png', 30308): ('사진 속 알약이 K-016262(ZD4522 크레스토정)', 16262, None),  # K-030308_p01 #4
    ('K-015280-016688-032310_0_2_0_2_75_000_200.png', 15280): ('사진 속 알약이 K-016206(GX CE2 아보다트연질캡슐)', 16206, None),  # K-015280_p03 #280
    ('K-015280-016688-032310_0_2_0_2_90_000_200.png', 15280): ('사진 속 알약이 K-016206(GX CE2 아보다트연질캡슐)', 16206, None),  # K-015280_p03 #293
    ('K-015280-016688-032310_0_2_0_2_70_000_200.png', 15280): ('사진 속 알약이 K-016206(GX CE2 아보다트연질캡슐)', 16206, None),  # K-015280_p03 #294 (Claude가 찾음)
    # box가 틀림 (사진에서 알약 범위를 재고 클래스 여백을 더해 다시 그린다)
    ('K-003614-027777-029667-030308_0_2_0_2_70_000_200.png', 3614): ('box 너비가 모자람', None, [195, 295, 164, 162]),  # K-003614_p01 #1
    ('K-003614-027777-029667-030308_0_2_0_2_70_000_200.png', 27777): ('box가 바닥을 가리킴 (알약은 오른쪽 위의 FMS6)', None, [632, 289, 175, 201]),  # K-027777_p01 #17
    ('K-006697-006835-026993-043233_0_2_0_2_75_000_200.png', 6697): ('box 너비가 모자람', None, [573, 728, 232, 210]),  # K-006697_p03 #283
    ('K-010221-016551-019607-033009_0_2_0_2_90_000_200.png', 10221): ('box가 왼쪽으로 쏠림', None, [177, 218, 222, 215]),  # K-010221_p02 #105
    ('K-010487-026455-044266-053384_0_2_0_2_75_000_200.png', 10487): ('box가 왼쪽으로 쏠림', None, [532, 33, 265, 417]),  # K-010487_p02 #164
    ('K-010487-020004-026455-026993_0_2_0_2_70_000_200.png', 10487): ('box가 다른 알약을 가리킴 (알약은 왼쪽 아래의 파란 캡슐)', None, [88, 712, 388, 385]),  # K-010487_p06 #532
    ('K-012638-024941-029711-043233_0_2_0_2_75_000_200.png', 12638): ('box가 다른 알약을 가리킴 (알약은 오른쪽 위의 파란 캡슐)', None, [577, 48, 256, 428]),  # K-012638_p03 #210
    ('K-012638-030850-035659-053384_0_2_0_2_70_000_200.png', 12638): ('box가 다른 알약을 가리킴 (알약은 왼쪽 아래의 파란 캡슐)', None, [54, 648, 270, 444]),  # K-012638_p03 #240
    ('K-005000-016262-032310_0_2_0_2_70_000_200.png', 16262): ('box가 왼쪽으로 밀림', None, [653, 213, 239, 229]),  # K-016262_p02 #117
    ('K-001900-004543-010224-016548_0_2_0_2_75_000_200.png', 16548): ('box 너비, 높이가 알약에 맞지 않음', None, [595, 778, 235, 230]),  # K-016548_p14 #1352
    ('K-012420-016548-018110-029451_0_2_0_2_70_000_200.png', 16548): ('box 너비, 높이가 알약에 맞지 않음', None, [169, 784, 232, 231]),  # K-016548_p14 #1353
    ('K-012247-016551-031705-033208_0_2_0_2_90_000_200.png', 16551): ('box 너비가 모자람', None, [636, 120, 285, 476]),  # K-016551_p01 #4
    ('K-000573-022627-025438-038954_0_2_0_2_75_000_200.png', 22627): ('box가 알약보다 너무 큼', None, [588, 79, 309, 470]),  # K-022627_p05 #438
    ('K-013004-024941-026993-041149_0_2_0_2_90_000_200.png', 24941): ('box 너비가 모자라고 오른쪽으로 쏠림', None, [635, 823, 169, 197]),  # K-024941_p01 #1
    ('K-013004-024941-026993-041149_0_2_0_2_90_000_200.png', 41149): ('box가 오른쪽으로 쏠림 (캡슐 왼쪽 끝이 빠지고 K-024941 알약을 덮음, 같은 사진에서 찾음)', None, [9, 651, 564, 406]),  # K-041149_p03 #291 (Claude가 찾음)
    ('K-016232-016235-027653-031885_0_2_0_2_70_000_200.png', 31885): ('box가 알약을 감싸지 못함 (위로 쏠림)', None, [627, 638, 317, 527]),  # K-031885_p02 #117
}
label_at = {(fn, a['category_id']): a for fn, r in fixed_records.items() for a in r['annotations']}  # 2-2 수정 뒤 라벨 (2-3)
for (fn, code), (problem, new_code, box) in LABEL_FINDINGS.items():
    assert (fn, code) in label_at and code in pill_codes(fn), f'{fn} K-{code:06d}: 2-2 수정 뒤 라벨에 없는 알약이다'
    assert new_code is None or new_code in aihub_example, f'K-{new_code:06d}: AI Hub에 메타데이터가 없는 클래스이다'
    assert box is None or usable_bbox(box), f'{fn} K-{code:06d}: 쓸 수 없는 box {box}'
for fn in {fn for fn, _ in LABEL_FINDINGS}:  # 바꾼 뒤에도 한 사진에 같은 클래스가 두 번 나오지 않는다
    codes = [(LABEL_FINDINGS.get((fn, a['category_id'])) or (None, None, None))[1] or a['category_id'] for a in fixed_records[fn]['annotations']]
    assert len(codes) == len(set(codes)), f'{fn}: 바꾼 뒤 같은 클래스가 두 번 있다 {codes}'


def area_of(box):
    """쓸 수 있는 box의 너비×높이를 구한다. 쓸 수 없는 box는 빈 칸이다."""
    return box[2] * box[3] if usable_bbox(box) else ''


FIX_COLUMNS = ['file_name', 'category_id', 'problem', 'action', 'bbox_before', 'bbox_after',
               'area_before', 'area_after', 'category_id_after', 'evidence']
aihub_fix_rows = []
for f in copy_fixes:  # 2-1의 copy 재배정 (2-5가 목록 순서로 적용하므로 2-2 수정보다 앞에 둔다)
    if f['action'] == '그대로':
        continue
    befores = [box for _, box in f['before']]  # 1-3이 이 사진에 넣은 그 K-코드의 JSON bbox (없거나 여러 개일 수 있다)
    aihub_fix_rows.append({
        'file_name': f['file_name'], 'category_id': f['code'], 'problem': 'JSON 이름의 각도가 틀림',
        'action': f"copy 재배정 ({f['action']}): {f['box']}",
        'bbox_before': json.dumps(befores[0] if len(befores) == 1 else befores) if befores else '', 'bbox_after': json.dumps(f['box']),
        'area_before': (area_of(befores[0]) if len(befores) == 1 else json.dumps([area_of(b) for b in befores])) if befores else '',
        'area_after': area_of(f['box']), 'category_id_after': '', 'evidence': '2-1',
    })
for fix in box_fixes:  # 2-2의 처리
    raw_box, box = fix['raw_box'], fix['box']
    relabel = fix['label_id'] if fix['label_id'] != fix['category_id'] else ''  # -오류 폴더 알약은 사진 속 실제 클래스로 붙였다
    if fix['action'] == '찾지 못함':
        action = '이미지 제외 (사진에서 알약을 찾지 못함)'
    elif fix['action'].startswith('그대로 둠') and not relabel:
        if fix['raw_area'] in (None, box[2] * box[3]):
            continue  # bbox와 area 모두 바뀌지 않음 (box가 똑같은 두 클래스 중 그 box의 알약인 쪽)
        action = 'area를 너비×높이로 다시 계산'
    else:
        action = f"{fix['action']}: {box}"
    aihub_fix_rows.append({
        'file_name': fix['file_name'], 'category_id': fix['category_id'], 'problem': fix['problem'], 'action': action,
        'bbox_before': json.dumps(raw_box) if raw_box is not None else '', 'bbox_after': json.dumps(box) if box else '',
        'area_before': fix['raw_area'] if fix['raw_area'] is not None else '', 'area_after': area_of(box) if box else '',
        'category_id_after': relabel, 'evidence': '2-1, 2-2' if fix['problem'].startswith('라벨 누락') else '2-2',
    })
for (fn, code), (problem, new_code, box) in LABEL_FINDINGS.items():  # 2-3의 눈 확인 (같은 알약의 2-2 수정 다음에 적용한다)
    raw_box, raw_area = raw_annotation(fn, code)
    after = box or label_at[(fn, code)]['bbox']
    action = ', '.join((['클래스 바꿈'] if new_code else []) + (['box 다시 그림'] if box else [])) + f': {after}'
    aihub_fix_rows.append({
        'file_name': fn, 'category_id': code, 'problem': problem, 'action': action,
        'bbox_before': json.dumps(raw_box) if raw_box is not None else '', 'bbox_after': json.dumps(after),
        'area_before': raw_area if raw_area is not None else '', 'area_after': area_of(after),
        'category_id_after': new_code or '', 'evidence': '2-3',
    })
AIHUB_FIXES_CSV = EDA_DIR / '01_raewoo_aihub_label_fixes.csv'  # 전처리 코드가 그대로 적용하도록 git으로 공유한다
with open(AIHUB_FIXES_CSV, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=FIX_COLUMNS)
    writer.writeheader()
    writer.writerows(aihub_fix_rows)

action_kind = lambda a: a.split(':')[0]
summary = collections.Counter((row['evidence'], action_kind(row['action'])) for row in aihub_fix_rows)
rows = [[evidence, action, n, len({r['file_name'] for r in aihub_fix_rows if r['evidence'] == evidence and action_kind(r['action']) == action})]
        for (evidence, action), n in sorted(summary.items())]
rows.append(['**합계**', '', f'**{len(aihub_fix_rows)}줄**', f"**{len({r['file_name'] for r in aihub_fix_rows})}장**"])
show_table(['근거', '처리', 'annotation 수', '이미지 수'], rows)

**인사이트**
- 정제 목록은 1,117줄(annotation 1,117개), 이미지 960장이다. 촬영 PNG 12,009장의 8.0%를 고친다.
- 근거별로는 2-2가 798줄로 가장 많고, 2-1·2-2 211줄, 2-3 63줄, 2-1 45줄이다.
- 처리별로는 area 다시 계산 337줄, 숫자 하나 고침 312줄, 사진에서 다시 그림 287줄 순서이다.
    - 이미지를 빼는 줄은 21줄(20장)이다.
    - 클래스를 바꾸는 줄은 56줄이다. (2-2의 `-오류` 폴더 9줄, 2-3의 47줄)

**결론**
- 이 CSV(`01_raewoo_aihub_label_fixes.csv`)를 AI Hub 라벨 정제의 원본으로 git에 올려 팀이 같이 쓴다.
- 학습 데이터를 만드는 코드는 1-3의 변환 뒤에 이 목록을 순서대로 적용한다. (2-5와 같은 방법)

### 2-5. 라벨 정제 처리
- 2-4가 저장한 정제 목록(CSV)을 1-3의 바꾼 AI Hub 라벨에 적용해 정제 라벨(`aihub_clean_records`)을 만든다. 3번부터는 이 라벨로 본다.
    - 학습 데이터를 만드는 코드도 같은 방법으로 적용하도록, 노트북 안의 변수가 아니라 저장한 CSV를 읽는다.
    - 1-3 라벨은 셀 62가 copy를 다시 배정하기 전의 라벨이다. copy 재배정도 목록의 2-1 줄로 적용한다.
- 적용 방법
    - 촬영 PNG(1-1)마다 파일명의 K-코드 하나에 annotation 하나를 만든다. 원본 JSON이 없는 촬영 PNG만 있는 이미지도 넣는다.
    - (이미지, `category_id`)의 줄이 없는 알약은 1-3의 annotation을 그대로 쓴다.
    - 줄이 있는 알약은 목록 순서대로 적용한다. 2-1이나 2-2의 줄 다음에 2-3의 줄이 온다.
        - `bbox_after`로 box를 바꾼다. 1-3이 같은 K-코드의 box를 여러 개 넣은 사진(copy 폴더)은 하나로 줄인다.
        - 1-3의 annotation이 없는 알약은 새로 만든다. 알약 메타데이터는 같은 클래스의 다른 AI Hub annotation에서 가져온다. (셀 93의 `drawn_annotation`)
        - `category_id_after`가 있으면 클래스와 알약 메타데이터를 그 클래스의 것으로 바꾼다.
    - "이미지 제외" 줄이 있는 이미지(2-2에서 알약을 찾지 못한 이미지)는 뺀다.
    - 모든 area는 너비×높이로 다시 계산한다.
- 결과가 노트북 안에서 고친 라벨(2-2 수정 뒤 라벨에 2-3의 `LABEL_FINDINGS`를 적용한 라벨)과 이미지마다 클래스와 box가 같은지 assert로 확인한다.
- 단계별 라벨의 이미지, 객체, 클래스 수와 남은 라벨 문제를 비교한다.
    - 쓸 수 있는 box 수가 파일명의 알약 수와 다른 이미지: 2-1의 라벨 누락과 남는 box가 있는 이미지이다.
    - 파일명에 없는 클래스가 있는 이미지: 사진 속 알약이 파일명에 없는 약이라 클래스를 바꾼 이미지이다. (2-2의 `-오류` 폴더, 2-3)
        - 이 이미지는 파일명의 K-코드로 클래스를 정하면 틀린다.
        -> 파일명 변경을 고려해야함

In [ ]:
with open(AIHUB_FIXES_CSV, encoding='utf-8') as f:  # 2-4가 저장한 정제 목록을 읽는다 (전처리 코드도 같은 파일을 쓴다)
    fix_table = list(csv.DictReader(f))
excluded_files = {row['file_name'] for row in fix_table if row['action'].startswith('이미지 제외')}
rows_of = collections.defaultdict(list)  # (이미지, 1-3의 K-코드) → 정제 목록의 줄 (목록 순서)
for row in fix_table:
    rows_of[(row['file_name'], int(row['category_id']))].append(row)
converted = {}  # 이미지 → 1-3이 넣은 annotation (셀 62가 copy를 다시 배정하기 전). 촬영 PNG만 있는 이미지는 없다
for j in aihub_kaggle_jsons:
    if j['data'] is not None:
        converted.setdefault(j['file_name'], []).append({**j['data']['annotations'][0], 'image': j['data']['images'][0], 'json_path': str(j['path'])})

aihub_clean_records = {}  # 정제 라벨: 이미지 → {'image', 'annotations'}. 3번부터 이 라벨로 본다
for fn in aihub_files:
    if fn in excluded_files:
        continue
    anns = []
    for code in pill_codes(fn):
        found = [a for a in converted.get(fn, []) if a['category_id'] == code]  # copy 파일이 있는 알약 폴더는 0~3개이다 (2-1)
        rows = rows_of[(fn, code)]
        if not rows:
            assert len(found) == 1 and usable_bbox(found[0]['bbox']), f'{fn} K-{code:06d}: 정제 목록에 줄이 없는데 1-3 라벨을 그대로 쓸 수 없다'
            anns.append(dict(found[0]))
            continue
        ann = dict(found[0]) if found else drawn_annotation(fn, code, json.loads(rows[0]['bbox_after']))
        for row in rows:  # 2-1이나 2-2의 줄 다음에 2-3의 줄을 적용한다
            ann['bbox'] = json.loads(row['bbox_after'])
            if row['category_id_after']:  # 클래스를 바꾸면 알약 메타데이터도 그 클래스의 것으로 바꾼다
                new_code = int(row['category_id_after'])
                ann['category_id'], ann['image'] = new_code, drawn_annotation(fn, new_code, ann['bbox'])['image']
        anns.append(ann)
    for ann in anns:
        ann['area'] = ann['bbox'][2] * ann['bbox'][3]
    aihub_clean_records[fn] = {'image': anns[0]['image'], 'annotations': anns}


def class_boxes(fn, anns, findings=None):
    """이미지의 (클래스, box) 목록. findings를 주면 2-3의 LABEL_FINDINGS를 적용한 값이다."""
    pairs = []
    for a in anns:
        _, new_code, box = (findings or {}).get((fn, a['category_id']), (None, None, None))
        pairs.append((new_code or a['category_id'], tuple(box or a['bbox'])))
    return sorted(pairs)


# 저장한 목록만으로 노트북 안에서 고친 라벨(2-2 수정 뒤 라벨 + 2-3)과 같은 라벨이 나오는지 확인한다
assert set(aihub_clean_records) == set(fixed_records), '정제 라벨과 2-2 수정 뒤 라벨의 이미지가 다르다'
assert all(class_boxes(fn, r['annotations']) == class_boxes(fn, fixed_records[fn]['annotations'], LABEL_FINDINGS)
           for fn, r in aihub_clean_records.items()), '정제 목록을 적용한 라벨이 2-2 수정 뒤 라벨에 2-3을 적용한 것과 다르다'


def label_summary(labels):
    """이미지 → annotation 목록에서 이미지, 객체, 클래스 수와 남은 라벨 문제를 센다."""
    objects = [a for anns in labels.values() for a in anns]
    usable = {fn: [a for a in anns if usable_bbox(a['bbox'])] for fn, anns in labels.items()}
    return [f'{len(labels):,}장', f'{len(objects):,}개', f"{len({a['category_id'] for a in objects})}개",
            f"{sum(not usable_bbox(a['bbox']) for a in objects):,}개",
            f"{sum(len(anns) != len(pill_codes(fn)) for fn, anns in usable.items()):,}장",
            f"{sum(any(a['category_id'] not in pill_codes(fn) for a in anns) for fn, anns in labels.items()):,}장"]


show_table(['라벨', '이미지', '객체', '클래스', '쓸 수 없는 bbox', '쓸 수 있는 box 수가 파일명의 알약 수와 다른 이미지', '파일명에 없는 클래스가 있는 이미지'], [
    ['바꾼 AI Hub 라벨 (1-3)', *label_summary(converted)],
    ['copy 재배정 뒤 (셀 62)', *label_summary({fn: r['annotations'] for fn, r in aihub_kaggle.items()})],
    ['2-2 수정 뒤 (셀 93)', *label_summary({fn: r['annotations'] for fn, r in fixed_records.items()})],
    ['**정제 라벨 (2-5, 정제 목록 적용)**', *[f'**{v}**' for v in label_summary({fn: r['annotations'] for fn, r in aihub_clean_records.items()})]],
])

**인사이트**
- 정제 라벨은 이미지 11,989장, 객체 45,879개, 클래스 116개이다.
    - 촬영 PNG 12,009장에서 알약을 찾지 못한 20장을 뺐다. 촬영 PNG만 있던 12장은 box를 그려 넣어 들어갔다.
- 쓸 수 없는 bbox와, box 수가 파일명의 알약 수와 다른 이미지가 모두 0이 되었다.
- 파일명에 없는 클래스가 있는 이미지는 27장이다.
    - 2-2의 `-오류` 폴더 9장과, 2-3에서 파일명에 없는 알약의 클래스로 바꾼 18장이다.
- 저장한 CSV만으로 만든 라벨이 노트북 안에서 고친 라벨과 같다. (assert 통과)

**결론**
- 3번부터는 정제 라벨(`aihub_clean_records`)로 보고, 학습에도 이 라벨을 쓴다.
- 클래스는 파일명의 K-코드가 아니라 정제 라벨의 category_id로 읽는다. 파일명으로 정하면 27장이 틀린다.

## 3. bbox 분포
- 2-5의 정제 라벨(`aihub_clean_records`)과 정제한 대회 train(`clean_records`, 00 2-5)을 비교한다.
- 00 3절에서 정한 입력 크기(640으로 시작해 1280과 비교), 크기 조절·모자이크 증강(기본값), NMS IoU(0.7)가 AI Hub를 섞은 뒤에도 맞는지 본다.

### 3-1. 객체 수와 크기
- 왼쪽: 이미지당 객체 수의 비율을 비교한다.
- 가운데: box의 짧은 변 길이(원본 픽셀) 분포를 비교한다. 긴 알약은 놓인 방향에 따라 너비와 높이가 바뀌므로 짧은 변으로 본다.
- 오른쪽: 입력 640(긴 변 1280을 640으로 줄임, 0.5배)에서의 짧은 변 길이이다. 00 3-1에서 대회 train은 최소 62, 중앙값 107픽셀이었다.
- 세 그림 모두 묶음의 크기가 달라서 비율(밀도)로 그린다.

In [ ]:
def box_sides(records):
    """box마다 (짧은 변, 긴 변) 픽셀을 모은다."""
    return np.array([sorted(a['bbox'][2:]) for r in records.values() for a in r['annotations']], dtype=float)


compare_sets = {'대회 train (정제)': clean_records, 'AI Hub (정제)': aihub_clean_records}
fig, axes = plt.subplots(1, 3, figsize=(21, 5.5), layout='constrained')
for (name, records), color in zip(compare_sets.items(), ['tab:blue', 'tab:green']):
    counts = collections.Counter(len(r['annotations']) for r in records.values())
    ks = np.arange(1, 5)
    share = np.array([counts[k] for k in ks]) / len(records)
    offset = -0.2 if color == 'tab:blue' else 0.2
    axes[0].bar(ks + offset, share, width=0.4, color=color, label=f'{name} {len(records):,}장')
    for k, s in zip(ks, share):
        if s > 0:
            axes[0].text(k + offset, s + 0.01, f'{s:.1%}', ha='center', fontsize=9)
    sides = box_sides(records)
    axes[1].hist(sides[:, 0], bins=np.arange(0, 560, 10), density=True, histtype='step', linewidth=2, color=color,
                 label=f'{name} 중앙값 {np.median(sides[:, 0]):.0f}, 최소 {sides[:, 0].min():.0f}')
    axes[2].hist(sides[:, 0] * 0.5, bins=np.arange(0, 280, 5), density=True, histtype='step', linewidth=2, color=color,
                 label=f'{name} 중앙값 {np.median(sides[:, 0]) * 0.5:.0f}, 최소 {sides[:, 0].min() * 0.5:.0f}')
axes[0].set_xticks(ks)
axes[0].set_xlabel('이미지당 객체 수 (개)')
axes[0].set_ylabel('이미지 비율')
axes[0].set_title('이미지당 객체 수')
axes[1].set_xlabel('box 짧은 변 (원본 픽셀)')
axes[1].set_title('box 짧은 변 길이 (원본 976×1280)')
axes[2].set_xlabel('box 짧은 변 (입력 640 기준 픽셀)')
axes[2].set_title('입력 640에서의 box 짧은 변 길이')
for ax in axes[1:]:
    ax.set_ylabel('밀도 (넓이 합 = 1)')
for ax in axes:
    ax.legend(fontsize=9)
fig.suptitle('대회 train과 AI Hub의 객체 수와 box 크기 (묶음마다 비율로 그림)', fontsize=14)
save_fig(fig, '3-1_objects_and_sizes')
plt.show()

**인사이트**
- AI Hub는 이미지의 82.7%에 알약이 4개 있다. 대회 train은 3개인 이미지가 67.7%이다.
- box 짧은 변 분포는 두 데이터가 비슷하다.
    - 중앙값은 AI Hub 236픽셀, 대회 train 214픽셀이다.
    - 최솟값은 AI Hub 120픽셀, 대회 train 125픽셀이다.
- 입력 640에서도 AI Hub box 짧은 변은 최소 60픽셀, 중앙값 118픽셀이다. (대회 train 62, 107픽셀)

**결론**
- 00에서 정한 입력 크기(640으로 시작해 1280과 비교)를 그대로 쓴다. AI Hub에도 작은 객체가 없다.
- 크기 조절과 모자이크 증강은 ultralytics 기본값을 그대로 쓴다.
- 이미지당 객체가 최대 4개라 최대 검출 수 기본값(300)으로 충분하다.

- 같은 알약을 같은 각도에서 찍으면 알약이 사진에 같은 크기로 찍히므로, box 크기 차이는 라벨을 그린 습관(알약 둘레 여백)의 차이이다.
    - 대회 train과 AI Hub에 모두 있는 클래스에서, (클래스, 촬영 각도)마다 box의 긴 변과 짧은 변 중앙값을 비교한다.
    - 점이 대각선 위에 있으면 두 데이터의 box 크기가 같다. 대각선에서 벗어난 거리가 여백 차이이다.
- 00 8-2에서 box를 4픽셀 옮기면 mAP@[0.75:0.95]가 0.87로 떨어졌다. 여백 습관이 몇 픽셀만 달라도 test 점수가 깎인다.

In [ ]:
def side_medians(records):
    """(K-코드, 촬영 각도)마다 box의 긴 변, 짧은 변 중앙값과 box 수를 구한다."""
    sides = collections.defaultdict(list)
    for record in records.values():
        for ann in record['annotations']:
            sides[(ann['category_id'], ann['image']['camera_la'])].append(sorted(ann['bbox'][2:], reverse=True))
    return {key: (np.median(v, axis=0), len(v)) for key, v in sides.items()}


kaggle_sides, aihub_sides = side_medians(clean_records), side_medians(aihub_clean_records)
common_keys = sorted(set(kaggle_sides) & set(aihub_sides))
diffs = np.array([aihub_sides[k][0] - kaggle_sides[k][0] for k in common_keys])  # (긴 변 차이, 짧은 변 차이)
fig, axes = plt.subplots(1, 2, figsize=(16, 7), layout='constrained')
for i, (ax, side) in enumerate(zip(axes, ['긴 변', '짧은 변'])):
    k_vals = np.array([kaggle_sides[k][0][i] for k in common_keys])
    a_vals = np.array([aihub_sides[k][0][i] for k in common_keys])
    ax.scatter(k_vals, a_vals, s=18, alpha=0.7)
    lo, hi = min(k_vals.min(), a_vals.min()) - 10, max(k_vals.max(), a_vals.max()) + 10
    ax.plot([lo, hi], [lo, hi], color='gray', linestyle='--', linewidth=1)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel(f'대회 train box {side} 중앙값 (픽셀)')
    ax.set_ylabel(f'AI Hub box {side} 중앙값 (픽셀)')
    d = diffs[:, i]
    ax.set_title(f'{side}: AI Hub - 대회 차이 중앙값 {np.median(d):+.1f}픽셀, 절댓값 4픽셀 넘는 묶음 {(np.abs(d) > 4).sum()}/{len(d)}')
fig.suptitle(f'같은 클래스·같은 각도의 box 크기 비교 ({len(common_keys)}개 묶음, 점 = (클래스, 각도) 하나, 회색 점선 = 크기가 같음)', fontsize=14)
save_fig(fig, '3-1b_margin_compare')
plt.show()

**인사이트**
- (클래스, 각도) 162개 묶음의 점이 대각선을 따라 놓여 있다. 두 데이터의 box 크기가 같다.
    - AI Hub - 대회 차이의 중앙값은 긴 변 +0.0픽셀, 짧은 변 -1.0픽셀이다.
- 차이가 4픽셀을 넘는 묶음도 긴 변 77개, 짧은 변 90개로 절반쯤이다.
    - 점이 대각선 위아래로 고르게 퍼져 있어, 한쪽으로 치우친 여백 습관은 아니다.
    - 대회 train은 묶음마다 box가 평균 3.5개(569개 ÷ 162묶음)뿐이라 중앙값이 쉽게 흔들린다.
    - 짧은 변이 더 퍼진 것은, 긴 알약의 box 짧은 변이 알약이 놓인 방향에 따라 크게 바뀌기 때문으로 보인다.

**결론**
- AI Hub box는 대회 box와 같은 여백으로 그려졌으므로, box 크기를 고치지 않고 그대로 섞는다.
- 학습 뒤 val에서 AI Hub 이미지와 대회 이미지의 IoU 0.95 AP를 따로 보면, 여백 차이가 점수에 영향을 주는지 확인할 수 있다.

### 3-2. 위치와 겹침
- 왼쪽: box 중심 위치의 분포를 AI Hub 전체로 본다. (색이 진할수록 box 중심이 많음)
- 오른쪽: 이미지마다 box 쌍의 IoU 최댓값을 구해 분포를 비교한다.
    - 00 3-2에서 대회 train은 최댓값이 0.021이었다. 알약끼리 거의 겹치지 않아 NMS IoU를 기본값 0.7로 두기로 했다.

- 둘째 그림: IoU 최댓값이 가장 큰 AI Hub 이미지 10장에 정제 라벨 box를 그린다.
    - box가 겹친 까닭이 알약 배치인지 라벨 오류인지 눈으로 확인한다.

In [ ]:
centers = np.array([[a['bbox'][0] + a['bbox'][2] / 2, a['bbox'][1] + a['bbox'][3] / 2]
                    for r in aihub_clean_records.values() for a in r['annotations']])


def max_pair_iou(records):
    """이미지마다 box 쌍의 IoU 최댓값을 구한다. box가 하나면 0이다."""
    values = []
    for record in records.values():
        boxes = [a['bbox'] for a in record['annotations']]
        values.append(max([box_iou(a, b) for i, a in enumerate(boxes) for b in boxes[i + 1:]], default=0))
    return np.array(values)


fig, axes = plt.subplots(1, 2, figsize=(16, 7), layout='constrained', gridspec_kw={'width_ratios': [0.8, 1.2]})
h = axes[0].hist2d(centers[:, 0], centers[:, 1], bins=[np.linspace(0, IMG_W, 25), np.linspace(0, IMG_H, 33)], cmap='Blues')
fig.colorbar(h[3], ax=axes[0], label='box 중심 수 (개)')
axes[0].set_xlim(0, IMG_W)
axes[0].set_ylim(IMG_H, 0)
axes[0].set_aspect('equal')
axes[0].set_xlabel('x (픽셀)')
axes[0].set_ylabel('y (픽셀)')
axes[0].set_title(f'AI Hub box 중심 위치 ({len(centers):,}개)')
bins = np.linspace(0, 1, 51)
pair_iou = {name: max_pair_iou(records) for name, records in compare_sets.items()}
for (name, values), color in zip(pair_iou.items(), ['tab:blue', 'tab:green']):
    axes[1].hist(values, bins=bins, histtype='step', linewidth=2, color=color,
                 label=f'{name} {len(values):,}장: 최댓값 {values.max():.3f}, 0.05 넘는 이미지 {(values > 0.05).sum()}장')
axes[1].set_yscale('log')
axes[1].set_xlabel('이미지 안 box 쌍의 IoU 최댓값')
axes[1].set_ylabel('이미지 수 (장, 로그 눈금)')
axes[1].set_title('이미지마다 가장 많이 겹친 box 쌍의 IoU')
axes[1].legend()
fig.suptitle('box 위치와 겹침', fontsize=14)
save_fig(fig, '3-2_position_overlap')
plt.show()

In [ ]:
aihub_pair_iou = pair_iou['AI Hub (정제)']
top_overlap = sorted(zip(aihub_pair_iou, aihub_clean_records), key=lambda t: (-t[0], t[1]))[:10]
fig, axes = plt.subplots(2, 5, figsize=(25, 14), layout='constrained')
for rank, (ax, (iou, fn)) in enumerate(zip(axes.flat, top_overlap), start=1):
    anns = aihub_clean_records[fn]['annotations']
    pair = max(((a, b) for i, a in enumerate(anns) for b in anns[i + 1:]), key=lambda p: box_iou(p[0]['bbox'], p[1]['bbox']))
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for ann in anns:
        color = 'red' if any(ann is p for p in pair) else 'lime'
        x, y, w, h = ann['bbox']
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2))
        # 겹친 쌍의 라벨 글자가 같은 자리에 겹치지 않게, 두 번째 box는 box 아래에 적는다
        ty, va = (y + h + 8, 'top') if ann is pair[1] else (y - 8, 'bottom')
        ax.text(x, ty, pill_label(ann), fontsize=7, va=va, bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0))
    ax.set_title(f"{rank}. IoU {iou:.3f} | {aihub_clean_records[fn]['image']['camera_la']}°\n{fn}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f'box 쌍의 IoU가 가장 큰 AI Hub 이미지 10장 (정제 라벨 {len(aihub_pair_iou):,}장 중 box가 겹친 이미지 {(aihub_pair_iou > 0).sum()}장, '
             f'IoU 0.05 초과 {(aihub_pair_iou > 0.05).sum()}장 | 빨강 = 가장 많이 겹친 box 쌍, 연두 = 나머지 box)', fontsize=14)
save_fig(fig, '3-2b_top_overlap')
plt.show()

**인사이트**
- box 중심은 사진의 네 자리(2×2 격자)에 몰려 있다. 가운데 위·아래에도 옅은 자리가 있다.
    - 중심은 대부분 x 150~850, y 150~1,050 안에 있다. 사진 가장자리에 걸친 알약은 없다.
- 이미지 안 box 쌍의 IoU 최댓값은 AI Hub 0.151로 대회 train(0.021)보다 크다. 0.05를 넘는 이미지는 42장이다.
- IoU가 가장 큰 10장은 모두 70° 사진이고, 모두 긴 캡슐 K-041149(Hanwha LCR5)가 대각선으로 놓여 있다.
    - box는 알약을 맞게 감싼다. 대각선으로 놓인 긴 캡슐의 box가 옆 알약 box의 모서리와 겹친 것이다.
    - 라벨 오류로 겹친 box는 보이지 않는다.

**결론**
- 실제 box끼리의 겹침이 최대 0.151이므로 NMS IoU는 00에서 정한 대로 0.7(기본값)로 시작한다. 0.5로 낮춰도 실제 box가 지워지지 않는다.
- 최대 검출 수도 기본값(300)을 그대로 쓴다. (3-1)
- 라벨 박스를 알약에 딱맞게 전처리하는 건 데이터 전처리 실험의 후보로 둔다.

## 4. 클래스 분포
- 2-5의 정제 라벨(`aihub_clean_records`)로 본다. 대회 train은 00 2-5의 정제 라벨(`clean_records`)이다.

### 4-1. 클래스별 객체 수와 등장 조합 수
- 위: 대회 train 클래스는 대회 train과 AI Hub의 객체 수를 나란히 놓고(로그 눈금), train에 없는 클래스는 AI Hub 객체 수만 본다.
- 아래: 같은 클래스들이 나오는 조합 수이다. 조합 하나는 같은 알약 배치를 각도만 바꿔 찍은 사진 3장이다. (00 4-3)
    - 객체 수가 많아도 조합 수가 적으면, 같은 배치의 사진만 반복해서 보는 셈이다.

In [ ]:
def class_stats(records):
    """클래스별 객체 수와 등장 조합 수를 센다."""
    objects, combos = collections.Counter(), collections.defaultdict(set)
    for fn, record in records.items():
        for ann in record['annotations']:
            objects[ann['category_id']] += 1
            combos[ann['category_id']].add(combo_name(fn))
    return objects, {c: len(s) for c, s in combos.items()}


aihub_object_count, aihub_combo_count = class_stats(aihub_clean_records)
_, train_combo_count = class_stats(clean_records)
aihub_classes = set(aihub_object_count)
new_classes = sorted(aihub_classes - set(train_classes), key=lambda c: (-aihub_object_count[c], c))
absent_train = sorted(set(train_classes) - aihub_classes)

fig, axes = plt.subplots(2, 2, figsize=(22, 10), layout='constrained', gridspec_kw={'width_ratios': [1, 1.1]})
order = sorted(train_classes, key=lambda c: (-clean_object_count[c], c))
x = np.arange(len(order))
for row, (train_values, aihub_values, unit) in enumerate([(clean_object_count, aihub_object_count, '객체 수'),
                                                           (train_combo_count, aihub_combo_count, '등장 조합 수')]):
    ax = axes[row, 0]
    ax.bar(x - 0.2, [train_values.get(c, 0) for c in order], width=0.4, label='대회 train (00 2-5 정제 라벨)')
    ax.bar(x + 0.2, [aihub_values.get(c, 0) for c in order], width=0.4, label='AI Hub (2-5 정제 라벨)')
    ax.set_yscale('log')
    ax.set_xticks(x, [f'K-{c:06d}' for c in order], rotation=90, fontsize=7)
    ax.set_ylabel(f'{unit} (개, 로그 눈금)')
    ax.set_title(f'대회 train 클래스 {len(train_classes)}개의 {unit}' +
                 (f'\nAI Hub에 없는 train 클래스: {", ".join(f"K-{c:06d}" for c in absent_train)} (막대 높이 0)' if row == 0 else ''))
    ax.legend()
    ax = axes[row, 1]
    values = [aihub_values[c] for c in new_classes]
    ax.bar([f'K-{c:06d}' for c in new_classes], values, color='tab:green')
    ax.tick_params(axis='x', rotation=90, labelsize=7)
    ax.set_ylabel(f'{unit} (개)')
    ax.set_title(f'train에 없는 클래스 {len(new_classes)}개의 AI Hub {unit} (최소 {min(values)}, 중앙값 {np.median(values):g})')
fig.suptitle(f'AI Hub 정제 라벨 {len(aihub_clean_records):,}장의 클래스별 객체 수와 등장 조합 수 (오른쪽은 객체 수 순서로 정렬)', fontsize=14)
save_fig(fig, '4-1_class_coverage')
plt.show()

**인사이트**
- 위 왼쪽 그림은 로그 눈금이라 AI Hub 막대끼리의 차이가 작아 보인다.
    - AI Hub에 있는 대회 train 클래스 54개의 AI Hub 객체 수는 159~1,353개로 8.5배 차이 나지만, 로그 눈금에서는 한 자릿수(10배)도 안 되는 높이 차이이다.
    - 바로 옆 대회 train 막대(3~157개)는 같은 클래스의 AI Hub 막대보다 8~447배 작아서, 주황 막대가 모두 비슷하게 커 보인다.
- 대회 train과 AI Hub를 나란히 놓은 그림이므로, 합친 뒤의 균형은 이 그림이 아니라 4-4의 선형 눈금 그림과 표에서 본다.
    - AI Hub에 없는 K-003351(157개), K-003483(45개)은 이 그림에서 가장 큰 파란 막대 두 개지만, 합치면 대회 train 클래스 56개 중 가장 적은 두 클래스가 된다. (4-4)
    - K-003483은 합친 클래스 118개 중에서도 가장 적다. (4-4 표의 최솟값 45개)
- AI Hub에는 대회 train 클래스 56개 중 54개와, train에 없는 클래스 62개가 있다. 합치면 118개이다.
    - AI Hub에 없는 train 클래스는 K-003351, K-003483이다.
- train에 없는 클래스 62개의 AI Hub 객체 수는 최소 90개(K-020753), 중앙값 374개이다.
- AI Hub는 클래스마다 등장 조합 수도 많다. train에 없는 클래스는 최소 30개, 중앙값 125개이다.
    - 대회 train은 클래스별 조합 수의 중앙값이 3.5개이다. (4-4 표)

**결론**
- 학습 클래스는 대회 train 56개와 AI Hub에만 있는 62개를 합친 118개로 한다.
    - test에만 나오는 알약이 AI Hub 클래스에 있는지는 5절에서 확인한다.
- AI Hub에 없는 K-003351, K-003483은 대회 train 객체만으로 학습한다. 두 클래스의 AP는 val에서 따로 본다.
- 합친 뒤의 불균형은 4-4와 4-5에서 판단한다.

### 4-2. 조합당 이미지 수
- AI Hub 소개대로 조합마다 촬영 각도(camera_la) 70°, 75°, 90° 사진이 한 장씩 있는지 센다.
- 2-5에서 뺀 이미지 때문에 3장이 다 남지 않은 조합도 센다.

In [ ]:
def combo_angles(records):
    """조합마다 남은 사진의 촬영 각도를 모은다."""
    angles = collections.defaultdict(list)
    for fn, record in records.items():
        angles[combo_name(fn)].append(record['image']['camera_la'])
    return angles


rows = []
for name, records in [('AI Hub 촬영 PNG 전체 (1-1)', {fn: {'image': {'camera_la': int(fn.split('_')[5])}} for fn in aihub_files}),
                      ('AI Hub 정제 라벨 (2-5)', aihub_clean_records), ('대회 train (참고)', clean_records)]:
    angles = combo_angles(records)
    per_combo = collections.Counter(len(v) for v in angles.values())
    full = sum(sorted(v) == [70, 75, 90] for v in angles.values())
    rows.append([name, f'{len(records):,}장', f'{len(angles):,}개', ', '.join(f'{k}장 {v:,}개' for k, v in sorted(per_combo.items())),
                 f'{full:,}개 ({full / len(angles):.1%})'])
show_table(['라벨', '이미지', '조합', '조합당 이미지 수', '70°·75°·90°가 모두 있는 조합'], rows)

**인사이트**
- AI Hub 촬영 PNG는 조합 4,003개 모두 70°·75°·90° 사진이 한 장씩 있다.
- 정제 라벨에서는 조합 4,002개 중 3,986개(99.6%)에 세 각도가 모두 남았다.
    - 2-5에서 뺀 20장 때문에 2장만 남은 조합 15개, 1장만 남은 조합 1개, 모두 빠진 조합 1개가 생겼다.
- 대회 train은 세 각도가 다 있는 조합이 35.1%뿐이다. 같은 조합의 나머지 각도 사진 일부는 test에 있다. (00 5-2)

**결론**
- 조합 하나는 같은 배치를 세 각도로 찍은 사진 3장이므로, 학습과 검증을 조합 단위로 나누도록 분할 담당 팀원에게 전달한다.
- 같은 사진이 든 두 조합 쌍(1-6)은 한 묶음으로 나눈다.

### 4-3. 같은 조합 사진 비교
- 무작위로 고른 조합 2개의 각도별 사진 3장을 나란히 놓고, 정제 라벨 box를 겹쳐 그린다. (seed 고정)
- 00 4-3에서 대회 train의 같은 조합 사진은 알약 배치가 같고, 70° 사진은 배치가 180° 돌아가 있었다. AI Hub도 같은지 본다.

In [ ]:
full_combos = sorted(c for c, v in combo_angles(aihub_clean_records).items() if sorted(v) == [70, 75, 90])
picked = [full_combos[i] for i in np.random.default_rng(SEED).choice(len(full_combos), size=2, replace=False)]
fig, axes = plt.subplots(2, 3, figsize=(20, 17), layout='constrained')
for row_axes, combo in zip(axes, picked):
    files = sorted((fn for fn in aihub_clean_records if combo_name(fn) == combo),
                   key=lambda fn: aihub_clean_records[fn]['image']['camera_la'])
    for ax, fn in zip(row_axes, files):
        ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
        for ann in aihub_clean_records[fn]['annotations']:
            x, y, w, h = ann['bbox']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=2))
            ax.text(x, y - 8, pill_label(ann), fontsize=8, va='bottom', bbox=dict(facecolor='lime', alpha=0.85, pad=1.5, linewidth=0))
        ax.set_title(f"{aihub_clean_records[fn]['image']['camera_la']}° | {fn}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
fig.suptitle('같은 조합의 각도별 사진 (행 = 조합, 열 = 촬영 각도 70°·75°·90°, 연두 box = 정제 라벨)', fontsize=14)
save_fig(fig, '4-3_same_combo')
plt.show()

**인사이트**
- AI Hub도 대회 사진(00 4-3)과 같다. 75°와 90° 사진은 알약 배치가 같고 자리만 조금 다르다.
- 70° 사진은 배치가 180° 돌아가 있다.
    - 첫 조합에서 70° 사진의 왼쪽 위 SUSPEN ER이 75°·90° 사진에서는 오른쪽 아래에 있다.
- box는 세 각도 모두 알약을 빈틈없이 감싼다.

**결론**
- 같은 조합의 사진은 같은 장면이므로 조합 단위로 나눈다. (4-2)
- AI Hub를 섞으면 배치가 180° 돌아간 70° 사진이 약 4,000장 들어온다. 00의 추가 실험 후보(180° 돌린 복사본)는 AI Hub를 섞은 기준선과 비교할 때만 한다.

### 4-4. 대회 train과 합친 분포
- 정제한 대회 train에 AI Hub 정제 라벨을 모두 더했을 때 클래스 분포가 어떻게 바뀌는지 표로 본다.
- 클래스별 객체 수의 최대÷최소와 변동계수(표준편차÷평균, 0이면 모든 클래스의 객체 수가 같음)로 불균형 정도를 잰다.
- 목적: 클래스 가중치나 반복 샘플링 같은 불균형 대응이 필요한지 정한다.

In [ ]:
def class_list(objects, keep):
    """조건에 맞는 클래스를 객체 수 순서로 적는다. 6개 이상이면 개수만 적는다."""
    picked = sorted((n, c) for c, n in objects.items() if keep(n))
    if len(picked) > 5:
        return f'{len(picked)}개'
    return ', '.join(f'K-{c:06d} {n}개' for n, c in picked) or '없음'


def distribution_column(records):
    """클래스 분포 표의 한 열을 만든다."""
    objects, combos = class_stats(records)
    values = np.array(sorted(objects.values()))
    total = values.sum()
    top_code, top_n = objects.most_common(1)[0]
    return [
        f'{len(records):,}장', f'{total:,}개', f'{len(objects)}개',
        f'{values[0]} / {np.median(values):g} / {values[-1]}', f'{values[-1] / values[0]:.1f}배', f'{values.std() / values.mean():.2f}',
        f'K-{top_code:06d} {top_n}개 ({top_n / total:.1%})',
        ' / '.join(str(sum(lo <= n < hi for n in values)) for lo, hi in BINS),
        class_list(objects, lambda n: n < BINS[0][1]),
        f'{min(combos.values())} / {np.median(list(combos.values())):g}',
    ]


BINS = [(0, 100), (100, 200), (200, 400), (400, 800), (800, float('inf'))]  # 객체 수 구간
combined_records = {**clean_records, **aihub_clean_records}
assert not set(clean_records) & set(aihub_clean_records), '대회 train과 AI Hub에 같은 파일명이 있다 (1-6)'
row_names = ['이미지 수', '객체 수', '클래스 수', '클래스별 객체 수 (최소 / 중앙값 / 최대)', '최대 ÷ 최소', '변동계수 (표준편차 ÷ 평균)',
             '객체가 가장 많은 클래스 (비중)', '객체 수 구간별 클래스 수 (100 미만 / 100~199 / 200~399 / 400~799 / 800 이상)',
             '객체 100개 미만 클래스', '클래스별 등장 조합 수 (최소 / 중앙값)']
show_table(['항목', '대회 train만 (00 2-5)', f'대회 train + AI Hub 전부 (2-5)'],
           [[name, before, after] for name, before, after in zip(row_names, distribution_column(clean_records),
                                                                 distribution_column(combined_records))])

**인사이트**
- AI Hub를 모두 더하면 객체가 771개에서 46,650개로 약 60배, 클래스가 56개에서 118개로 늘어난다.
- 불균형이 크게 줄었다.
    - 변동계수 1.53 → 0.47, 최대÷최소 52.3배 → 30.5배
    - 가장 많은 클래스의 비중 20.4%(K-003351) → 2.9%(K-016548)
- 객체 100개 미만 클래스는 55개에서 2개(K-003483 45개, K-020753 90개)로 줄었다.
- 클래스별 등장 조합 수는 최소 1 → 15개, 중앙값 3.5 → 124개로 늘었다. 같은 배치만 반복해서 보는 문제도 줄었다.

**결론**
- 남은 불균형(30.5배)은 양 끝 클래스에서 생긴다.
    - 가장 적은 K-003483(45개)은 중앙값(370.5개)의 8분의 1이고, AI Hub에 없어 채울 수 없다.
    - 가장 많은 K-016548(1,371개)은 중앙값의 3.7배이다. 4-5에서 흔한 클래스를 덜 넣는 선택을 비교한다.

- 합친 분포를 클래스마다 막대 하나로 그린다.
    - 대회 train 클래스와 train에 없는 클래스를 색으로 나누고, 그룹 안에서는 객체 수 순서로 놓는다.
    - 대회 train 클래스 막대 아래쪽의 짙은 남색은 대회 원본 객체이다.

In [ ]:
def class_bar(ax, objects, title, line=None):
    """클래스별 객체 수를 그룹(대회 train 클래스, train에 없는 클래스)으로 나눠 막대로 그린다."""
    start, ticks = 0, []
    for name, classes, color in [('대회 train 클래스', set(train_classes), 'tab:blue'),
                                 ('train에 없는 클래스', set(objects) - set(train_classes), 'tab:green')]:
        order = sorted(classes, key=lambda c: (-objects[c], c))
        xs = np.arange(start, start + len(order))
        before = np.array([clean_object_count[c] for c in order])
        after = np.array([objects[c] for c in order])
        if before.any():
            ax.bar(xs, before, color='#08306b', label='대회 원본 객체 (00 2-5 정제 라벨)')
        ax.bar(xs, after - before, bottom=before, color=color,
               label=f'{name} {len(order)}개 (객체 {after.min()}~{after.max()}개, 중앙값 {np.median(after):g}개)')
        if start:
            ax.axvline(start - 0.5, color='gray', linewidth=0.8, linestyle=':')
        ticks += order
        start += len(order)
    if line:
        ax.axhline(line, color='black', linewidth=1, linestyle='--', label=f'목표 객체 수 {line}개')
    ax.set_xticks(np.arange(len(ticks)), [f'K-{c:06d}' for c in ticks], rotation=90, fontsize=6)
    ax.set_xlim(-1, len(ticks))
    ax.set_ylabel('객체 수 (개)')
    ax.set_title(title)
    ax.legend(loc='upper right', fontsize=9)


combined_objects, _ = class_stats(combined_records)
fig, ax = plt.subplots(figsize=(24, 7), layout='constrained')
class_bar(ax, combined_objects, f'대회 train {len(clean_records)}장 + AI Hub 전부 {len(aihub_clean_records):,}장의 클래스 {len(combined_objects)}개 객체 수 '
          f'(최대÷최소 {max(combined_objects.values()) / min(combined_objects.values()):.1f}배)')
save_fig(fig, '4-4_combined_classes')
plt.show()

**인사이트**
- K-016548(1,371개)과 K-016551(1,345개)이 다른 클래스보다 크게 많다. 그다음은 K-016232, K-016262로 약 820개이다.
    - 이 클래스들은 AI Hub의 많은 조합에 함께 들어 있다. (4-1 아래 그림의 조합 수)
- 대회 train 클래스 56개의 중앙값은 348개, train에 없는 클래스 62개는 374개로 비슷하다.
- 대회 원본 객체(짙은 남색)는 대부분의 막대에서 아래쪽에 얇게만 보인다.
    - K-003351(157개)과 K-003483(45개)만 막대 전체가 대회 원본 객체이다.
- train에 없는 클래스 중에서는 K-020753(90개)이 가장 적다.

**결론**
- 객체 수를 고르게 하려면 K-016548, K-016551이 든 이미지를 덜 넣어야 한다. 4-5에서 목표 객체 수로 고르는 방법을 비교한다.

### 4-5. 가장 균일하게 고르기
- 클래스마다 목표 객체 수를 정하고, 목표에 못 미친 클래스를 채우는 AI Hub 이미지만 고른다.
    - 사진에는 알약 3~4개가 함께 있어서, 이미지를 넣으면 그 안의 알약이 모두 함께 들어간다. 그래서 완전히 같은 수로 맞출 수는 없다.
    - 00 6-6의 방법(목표에 못 미친 클래스가 하나라도 있으면 넣음)은 함께 찍힌 흔한 클래스도 계속 늘린다.
    - 그래서 객체가 적은 클래스부터 하나씩, 그 클래스가 목표를 채울 때까지 그 클래스가 있는 이미지를 넣는다.
        - 한 클래스 안에서는 이미 목표를 채운 클래스가 적게 함께 찍힌 이미지부터 넣는다. 흔한 클래스가 덜 늘어난다.
        - 모든 클래스가 목표나, 목표보다 적으면 가진 객체 전부를 채운다.
- 목표 객체 수별로 고른 이미지 수와 불균형 정도(최대÷최소, 변동계수)를 AI Hub를 전부 쓸 때와 비교한다.
    - 00 6-6은 디스크 용량 때문에 목표를 골랐다. 지금은 이미지를 모두 받았으므로, 학습에 무엇을 넣을지의 문제이다.

In [ ]:
TARGETS = [100, 200, 300, 400]
image_classes = {fn: [a['category_id'] for a in r['annotations']] for fn, r in aihub_clean_records.items()}
all_classes = sorted(set(train_classes) | aihub_classes)


images_by_class = collections.defaultdict(list)  # K-코드 → 그 클래스가 있는 AI Hub 이미지
for fn, classes in sorted(image_classes.items()):
    for c in set(classes):
        images_by_class[c].append(fn)
total_objects = collections.Counter(clean_object_count) + collections.Counter(c for cs in image_classes.values() for c in cs)


def plan_uniform(target):
    """객체가 적은 클래스부터 목표 객체 수를 채우도록 AI Hub 이미지를 고른다."""
    count = collections.Counter(clean_object_count)  # 대회 train(00 2-5 정제 라벨)은 모두 넣고 시작한다
    chosen = set()
    for c in sorted(all_classes, key=lambda c: (total_objects[c], c)):
        # 이미 목표를 채운 클래스가 적게 함께 찍힌 이미지부터 넣는다
        pool = sorted((fn for fn in images_by_class[c] if fn not in chosen),
                      key=lambda fn: (sum(count[o] >= target for o in image_classes[fn]), fn))
        for fn in pool:
            if count[c] >= target:
                break
            chosen.add(fn)
            count.update(image_classes[fn])
    return sorted(chosen), count


def balance_row(name, objects, n_images):
    values = np.array([objects[c] for c in all_classes])
    return [name, f'{n_images:,}장', f'{values.sum():,}개', f'{values.min()} / {np.median(values):g} / {values.max()}',
            f'{values.max() / values.min():.1f}배', f'{values.std() / values.mean():.2f}']


uniform_plans = {t: plan_uniform(t) for t in TARGETS}
rows = [balance_row('AI Hub 전부', combined_objects, len(aihub_clean_records))]
for t, (chosen, count) in uniform_plans.items():
    short = sum(count[c] < t for c in all_classes)
    rows.append(balance_row(f'목표 {t}개 (못 미친 클래스 {short}개)', count, len(chosen)))
show_table(['AI Hub 이미지 선택', 'AI Hub 이미지 수', '객체 수 (대회 train 포함)', '클래스별 객체 수 (최소 / 중앙값 / 최대)', '최대 ÷ 최소', '변동계수'], rows)

**인사이트**
- 목표가 클수록 고른 이미지가 늘고, 목표 400개에서는 AI Hub 거의 전부(11,523장)를 고른다.
- 최솟값은 모든 행에서 45개(K-003483)이다. AI Hub에 없는 클래스라 고르는 방법과 상관없다.
- 변동계수는 0.43~0.47로 크게 바뀌지 않는다. 목표 200개가 0.43으로 가장 낮다.
    - 목표 100개에서도 최댓값이 572개이다. 적은 클래스를 채우려고 넣은 사진에 흔한 클래스가 함께 찍혀 있기 때문이다.
- 목표 200개는 AI Hub 6,993장(전부의 58%)으로 클래스 중앙값을 정확히 200개로 맞춘다. 최대÷최소는 20.9배이다.
- 목표 100개는 최대÷최소가 12.7배로 가장 작지만, AI Hub 이미지가 전부의 30%(3,644장)로 줄어든다.

**결론**
- 목표 200개로 고른 AI Hub 6,993장을 기본 보충 데이터로 쓴다. (아래 셀의 `SELECTED_TARGET`)
    - 변동계수가 가장 낮고, 이미지가 전부 쓸 때의 58%라 학습 시간도 줄어든다.
- 변동계수 차이(0.43과 0.47)가 작으므로, AI Hub 전부(11,989장)로 학습한 결과와 val에서 비교해 확정한다. (7번의 추가 실험 후보)

- 4-5 표에서 고른 목표 객체 수로 AI Hub 이미지 목록을 `data/external/aihub_targets.csv`에 저장하고, 전부 쓸 때와 클래스별 객체 수를 나란히 그린다.
    - 목록의 열은 00 6-6과 같다. (라벨 zip, 이미지 zip, filekey, 이미지 파일명)
    - zip은 PNG가 나온 이미지 zip(1-1)과 그 짝인 라벨 zip이다. 촬영 PNG만 있는 이미지는 JSON이 없어서, 라벨은 정제 목록의 줄로만 만든다. (2-5)

In [ ]:
SELECTED_TARGET = 200  # 4-5 표를 보고 정한다
FILEKEY = {  # 이미지 zip → filekey (script/download_aihub.py로 받는 번호)
    'TS_1_조합.zip': 66154, 'TS_3_조합.zip': 66156, 'TS_4_조합.zip': 66157, 'TS_5_조합.zip': 66158,
    'TS_6_조합.zip': 66159, 'TS_7_조합.zip': 66160, 'TS_8_조합.zip': 66161, 'VS_1_조합.zip': 66254,
}
label_zip_of_image_zip = {image_zip: label_zip for label_zip, image_zip in PAIR.items()}  # 1-1의 zip 쌍
selected, selected_count = uniform_plans[SELECTED_TARGET]
with open(EXTERNAL_DIR / 'aihub_targets.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['label_zip', 'image_zip', 'filekey', 'file_name'])
    # 촬영 PNG만 있는 이미지(1-1)는 JSON이 없으므로 PNG가 나온 이미지 zip(셀 13의 image_zip_of)으로 zip을 정한다
    for fn in sorted(selected, key=lambda n: (label_zip_of_image_zip[image_zip_of[n]], n)):
        writer.writerow([label_zip_of_image_zip[image_zip_of[fn]], image_zip_of[fn], FILEKEY[image_zip_of[fn]], fn])

fig, axes = plt.subplots(2, 1, figsize=(24, 13), layout='constrained', sharex=False)
class_bar(axes[0], combined_objects, f'AI Hub 전부 {len(aihub_clean_records):,}장 (최대÷최소 {max(combined_objects.values()) / min(combined_objects.values()):.1f}배)')
class_bar(axes[1], selected_count, f'목표 {SELECTED_TARGET}개로 고른 AI Hub {len(selected):,}장 '
          f'(최대÷최소 {max(selected_count[c] for c in all_classes) / min(selected_count[c] for c in all_classes):.1f}배)', line=SELECTED_TARGET)
fig.suptitle('대회 train + AI Hub의 클래스별 객체 수: 전부 쓸 때(위)와 목표 객체 수로 고를 때(아래)', fontsize=14)
save_fig(fig, '4-5_uniform_selection')
plt.show()

**인사이트**
- 목표 200개로 고르면 대회 train 클래스는 45~940개(중앙값 200개), train에 없는 클래스는 90~406개(중앙값 207개)가 된다.
- 대부분의 클래스가 200개(점선) 근처에 모인다.
    - K-016548(940개)과 K-016551(약 730개)은 여전히 많다. 다른 클래스를 채우려고 넣은 사진에 함께 찍혀 있다.
    - 목표에 못 미친 15개 클래스는 가진 객체를 모두 넣었다. (K-003483, K-020753 등)
- 고른 이미지 목록을 `data/external/aihub_targets.csv`(6,993줄)에 저장했다. 열은 라벨 zip, 이미지 zip, filekey, 이미지 파일명이다.

**결론**
- 학습 데이터는 대회 train 232장과 `aihub_targets.csv`의 AI Hub 6,993장으로 만든다. 라벨은 정제 목록(2-4)을 적용한 것을 쓴다.
- 클래스 가중치나 반복 샘플링 없이 시작한다. val의 클래스별 AP에서 적은 클래스(K-003483, K-020753)가 특히 낮으면 반복 샘플링을 실험한다.

## 5. test 전용 클래스 채우기
- 00 5-3에서 test 사진을 직접 보고 train에 없는 알약 18종(각인 19종, 213과 30은 같은 알약의 앞뒤)을 찾았다. test 842장 중 335장(39.8%)에 나온다.
- 이 알약들이 AI Hub로 채워지는지 정제 라벨(2-5)로 확인한다.

### 5-1. test에서 읽은 각인 대조
- 00 5-3 기록표의 각인을 AI Hub 라벨의 각인 필드(print_front, print_back)와 마크 설명 필드(mark_code_front_anal, mark_code_back_anal)에서 찾는다.
    - 회사 로고처럼 그림으로 된 각인은 각인 필드에 "마크"라고만 적고, 로고의 글자는 마크 설명 필드에 적는다. (예: 각인 필드 "마크", 마크 설명 "Hanmi")
    - 00 6-5는 각인 필드만 찾아서 Hanmi를 찾지 못했다.
    - 띄어쓰기와 대소문자는 무시하고, 필드 값 전체가 같은 경우를 먼저 찾는다.
    - 같은 값이 없으면 필드 값에 각인이 들어 있는 경우(부분 일치)를 찾는다.
    - AI Hub는 분할선을 "분할선"이라는 글자로 적으므로, 기록표의 `|`를 바꿔서 찾는다. (K|50 → `K분할선50`)

In [ ]:
def normalize(text):
    """띄어쓰기와 대소문자를 무시하고 비교하도록 각인 글자를 정리한다."""
    return str(text or '').replace(' ', '').lower()


aihub_pill_info = {}  # K-코드 → 그 알약 JSON의 이미지 메타데이터
for record in aihub_clean_records.values():
    for ann in record['annotations']:
        aihub_pill_info.setdefault(ann['category_id'], ann['image'])
filled = sum(bool(im['print_front'] or im['print_back']) for im in aihub_pill_info.values())
IMPRINT_KEYS = ['print_front', 'print_back', 'mark_code_front_anal', 'mark_code_back_anal']  # 각인 필드와 마크 설명 필드
imprint_fields = {c: tuple(normalize(im.get(key)) for key in IMPRINT_KEYS) for c, im in aihub_pill_info.items()}

files_by_imprint = collections.defaultdict(set)  # 기록표의 각인 → test 파일 번호
look_by_imprint = {}  # 기록표의 각인 → test에서 본 색과 모양
imprint_matches = {}  # 기록표의 각인 → AI Hub에서 찾은 K-코드
for row in review_rows:
    if row['imprint'] not in ('', '?'):  # 각인을 읽지 못한 알약은 글자로 대조할 수 없어 뺀다
        files_by_imprint[row['imprint']].add(int(row['test_file'].split('.')[0]))
        look_by_imprint.setdefault(row['imprint'], f"{row['color']} {row['shape']}")

rows = []
for seen, numbers in sorted(files_by_imprint.items(), key=lambda kv: (-len(kv[1]), kv[0])):
    query = normalize(seen.replace('|', '분할선'))
    match, hits = '같음', [c for c, fields in imprint_fields.items() if query in fields]
    if not hits and len(query) >= 2:
        match, hits = '부분 일치', [c for c, fields in imprint_fields.items() if any(query in f for f in fields)]
    imprint_matches[seen] = hits
    shown = ', '.join(map(str, sorted(numbers)[:6])) + (' …' if len(numbers) > 6 else '')
    for c in hits:
        info = aihub_pill_info[c]
        rows.append([seen, look_by_imprint[seen], shown, f'K-{c:06d}', info['dl_name'], f"{info['color_class1']} {info['drug_shape']}",
                     match, '있음' if c in train_classes else '없음', aihub_object_count[c]])
    if not hits:
        rows.append([seen, look_by_imprint[seen], shown, '찾지 못함', '-', '-', '-', '-', '-'])
rows.append([f'**각인 필드가 있는 클래스 {filled}/{len(aihub_pill_info)}**', '', '', '', '', '', '', '', ''])
show_table(['test에서 읽은 각인', 'test 색, 모양', 'test 파일 번호', 'AI Hub K-코드', '제품명', 'AI Hub 색, 모양', '일치',
            'train에 있음', 'AI Hub 객체 수'], rows)

**인사이트**
- test에서 읽은 각인 19종을 모두 AI Hub 클래스에서 찾았다.
    - 18종은 필드 값 전체가 같고, 213만 부분 일치(K-018110의 각인 "마크213/30")이다.
    - 213과 30은 같은 클래스 K-018110(란스톤엘에프디티정 30mg)의 앞뒤이다.
- Hanmi는 회사 로고라 두 클래스가 찾아졌다. (K-031705 낙소졸정, K-025469 아모잘탄정)
    - K-025469는 대회 train에 이미 있는 클래스이다.
- K-025469를 빼면 찾은 클래스는 모두 train에 없는 클래스이고, AI Hub 객체가 293~467개씩 있다.
- 기록표와 AI Hub의 색 이름이 다른 경우가 있다. (C77 빨강과 갈색, PE 하양과 노랑, Q25 빨강과 분홍)
    - 사람마다 색을 부르는 이름이 달라서로 보인다. 같은 알약인지는 5-2에서 사진으로 본다.
- AI Hub 116개 클래스 중 114개에 각인 필드가 있다.

**결론**
- 각인으로 찾은 클래스가 test 알약과 같은 약인지 5-2에서 사진으로 확인한다. 색 이름이 달라도 후보에서 빼지 않는다.

### 5-2. test 알약과 AI Hub 알약 나란히 보기
- 각인마다 한 줄에 왼쪽 2칸은 test 사진, 오른쪽 4칸은 5-1에서 찾은 AI Hub 클래스의 crop이다.
    - test 사진은 00 5-3처럼 기록표의 위치 쪽 모서리에서 620×780픽셀을 자른다. 각인마다 서로 다른 조합의 사진 2장을 고른다.
    - AI Hub crop은 그 클래스의 정제 라벨 box를 정사각형으로 자른 것이다. 서로 다른 조합에서 4개를 고른다. (seed 고정)
- 같은 알약인지 색, 모양, 크기, 각인으로 대조한다.

In [ ]:
POS_CROP_W, POS_CROP_H = 620, 780  # 기록표 위치 쪽 모서리에서 자를 크기 (00 5-3과 같음)
N_AIHUB = 4  # 각인마다 보여 줄 AI Hub crop 수


def crop_position(image, position):
    """기록표의 위치(예: 왼쪽 위) 쪽 모서리에서 POS_CROP_W×POS_CROP_H 크기로 자른다."""
    x0 = 0 if '왼쪽' in position else IMG_W - POS_CROP_W
    y0 = 0 if '위' in position else IMG_H - POS_CROP_H
    return image.crop((x0, y0, x0 + POS_CROP_W, y0 + POS_CROP_H))


def test_examples(seen, n=2):
    """기록표에서 그 각인이 나오는 줄을 서로 다른 조합에서 n개 고른다. (번호 3개가 한 조합, 00 5-2)"""
    first_of_combo = {}
    for row in sorted((r for r in review_rows if r['imprint'] == seen), key=lambda r: int(r['test_file'].split('.')[0])):
        first_of_combo.setdefault((int(row['test_file'].split('.')[0]) - 1) // 3, row)
    return list(first_of_combo.values())[:n]


aihub_class_anns = collections.defaultdict(list)  # K-코드 → (이미지, annotation) 목록
for fn in sorted(aihub_clean_records):
    for ann in aihub_clean_records[fn]['annotations']:
        aihub_class_anns[ann['category_id']].append((fn, ann))


def aihub_examples(code, n=N_AIHUB, seed=SEED):
    """그 클래스의 annotation을 서로 다른 조합에서 n개 무작위로 고른다."""
    by_combo = {}
    for fn, ann in aihub_class_anns[code]:
        by_combo.setdefault(combo_name(fn), []).append((fn, ann))
    combos = sorted(by_combo)
    picks = np.random.default_rng(seed).choice(len(combos), size=min(n, len(combos)), replace=False)
    return [by_combo[combos[i]][0] for i in sorted(picks)]


lines_to_show = [(seen, c) for seen in sorted(files_by_imprint, key=lambda s: (-len(files_by_imprint[s]), s))
                 for c in (imprint_matches[seen] or [None])]
fig, axes = plt.subplots(len(lines_to_show), 2 + N_AIHUB, figsize=(20, 3.4 * len(lines_to_show)), layout='constrained',
                         gridspec_kw={'width_ratios': [1.6, 1.6] + [1] * N_AIHUB})
for row_axes, (seen, c) in zip(axes, lines_to_show):
    for ax, row in zip(row_axes[:2], test_examples(seen)):
        ax.imshow(crop_position(Image.open(TEST_IMG_DIR / row['test_file']).convert('RGB'), row['position']))
        ax.set_title(f"test {row['test_file']} {row['position']}\n기록표: {seen} ({row['color']} {row['shape']})", fontsize=9)
    if c is None:
        row_axes[2].text(0.5, 0.5, 'AI Hub 각인 필드에서\n찾지 못함', ha='center', va='center', fontsize=12)
    else:
        for ax, (fn, ann) in zip(row_axes[2:], aihub_examples(c)):
            ax.imshow(crop_square(Image.open(AIHUB_IMG_DIR / fn).convert('RGB'), ann['bbox']))
            ax.set_title(f"AI Hub {pill_label(ann)}\n{ann['image']['color_class1']} {ann['image']['drug_shape']}", fontsize=9)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('train에 없는 test 알약(왼쪽 2칸, 기록표 위치 쪽 모서리)과 각인으로 찾은 AI Hub 알약(오른쪽 4칸, 정제 라벨 box)', fontsize=14)
save_fig(fig, '5-2_test_vs_aihub')
plt.show()

**인사이트**
- Hanmi는 5-1에서 AI Hub 클래스 두 개가 찾아져서 두 행으로 그렸다.
    - Hanmi는 한미약품 로고이고, K-031705(낙소졸정)와 K-025469(아모잘탄정) 모두 마크 설명 필드에 "Hanmi"가 있다.
    - 두 행의 test 사진(88.png, 298.png)은 같고, 오른쪽 AI Hub crop만 다르다.
- Hanmi 각인의 test 알약은 K-031705와 같은 약이다.
    - 둘 다 노랑 타원형이고, 타원 테두리 안에 Hanmi가 새겨져 있다.
    - K-031705 crop 하나에는 뒷면 각인 Naxozol 500/20이 보인다.
- K-025469는 test 알약과 다른 약이다.
    - 분홍 타원형이고, crop에 보이는 면의 각인은 Amosartan 5/100이다.
    - 대회 train에 이미 있는 클래스라서 train에 없는 알약일 수도 없다. (5-1 표)
    - 5-3에서 `DIFFERENT_PILLS`에 적어 뺀다.
- Hanmi를 뺀 나머지 각인 18개는 찾은 AI Hub 클래스가 test 알약과 같은 약으로 보인다.
    - 각인, 모양, 색이 맞는다. 213과 30은 K-018110의 앞면(마크 213)과 뒷면(30)이다.

**결론**
- test에만 나오는 알약 18종은 모두 AI Hub 클래스로 학습할 수 있다. 이 클래스들은 118개 클래스 체계에 들어 있다. (4-1)
- Hanmi 각인은 K-031705로만 센다. K-025469는 5-3의 `DIFFERENT_PILLS`로 뺀다.

### 5-3. test 사진 단위 커버리지
- test 사진마다 기록표에 적힌 train에 없는 알약이 모두 학습 클래스로 채워지는지 센다.
    - 기록표에 줄이 없는 test 사진은 모든 알약이 train 갤러리에 있는 사진이다. (00 5-2)
    - 보충 후에는 5-1에서 K-코드를 찾고 5-2에서 같은 알약으로 확인한 각인을 채워진 것으로 본다.
        - 5-1은 각인 글자만으로 찾으므로, 다른 약이 함께 나올 수 있다. 5-2 그림에서 test 알약과 다른 약으로 확인한 K-코드는 상수(`DIFFERENT_PILLS`)로 적어 뺀다.
        - 빼고 남은 K-코드가 하나인 각인만 채워진 것으로 본다.
- 목적: test 점수에서 보충으로도 맞힐 수 없는 알약이 얼마나 남는지 확인한다.

In [ ]:
# 5-2 그림에서 test 알약과 다른 약으로 확인한 (각인, K-코드). 각인 글자만 같다
DIFFERENT_PILLS = {
    ('Hanmi', 25469): '아모잘탄정은 분홍 타원형이다. test의 Hanmi 알약은 노랑 타원형인 K-031705(낙소졸정, 뒷면 Naxozol 500/20)이다',
}
same_pills = {seen: [c for c in hits if (seen, c) not in DIFFERENT_PILLS] for seen, hits in imprint_matches.items()}
code_of_imprint = {seen: codes[0] for seen, codes in same_pills.items() if len(codes) == 1}  # 5-2에서 같은 알약으로 확인한 각인
new_pills_by_file = collections.defaultdict(set)  # test 파일 → 그 사진의 train에 없는 알약 각인
for row in review_rows:
    new_pills_by_file[row['test_file']].add(row['imprint'])
covered_after = [f for f, imprints in new_pills_by_file.items() if all(i in code_of_imprint for i in imprints)]
left = collections.Counter(i for f, imprints in new_pills_by_file.items() for i in imprints if i not in code_of_imprint)
n_test = len(test_files)
rows = [
    ['모든 알약이 학습 클래스에 있는 test 사진', f'{n_test - len(new_pills_by_file)}장 ({(n_test - len(new_pills_by_file)) / n_test:.1%})',
     f'{n_test - len(new_pills_by_file) + len(covered_after)}장 ({(n_test - len(new_pills_by_file) + len(covered_after)) / n_test:.1%})'],
    ['train에 없는 알약이 하나라도 있는 test 사진', f'{len(new_pills_by_file)}장', f'{len(new_pills_by_file) - len(covered_after)}장'],
    ['학습 클래스에 없는 알약 (각인: 나오는 test 사진 수)', f'{len(files_by_imprint)}종',
     ', '.join(f'{i}: {len({r["test_file"] for r in review_rows if r["imprint"] == i})}장' for i in sorted(left)) or '없음'],
]
show_table(['항목', '보충 전 (대회 train만)', '보충 후 (+ AI Hub)'], rows)

**인사이트**
- 보충 전에는 test 842장 중 507장(60.2%)만 모든 알약이 학습 클래스에 있다. 나머지 335장에는 train에 없는 알약이 하나 이상 있다.
- AI Hub로 보충하면 842장(100.0%) 모두 알약이 학습 클래스에 있다.
    - 기록표의 각인 19종(알약 18종)이 모두 5-2에서 같은 알약으로 확인한 AI Hub 클래스로 채워진다.
- 이 수치는 00 5-2의 test 기록표 기준이다. 기록표에서 놓친 알약이 있으면 그만큼 줄어든다.

**결론**
- AI Hub 보충은 필수이다. 보충하지 않으면 test 사진의 39.8%에 맞힐 수 없는 알약이 있다.
- 보충 데이터에는 test 전용 알약 18종의 클래스가 반드시 들어가야 한다.
    - 4-5의 목표 200개 목록에도 이 클래스들은 200개 이상씩 들어간다. (AI Hub에 293~467개씩 있음, 5-1)

## 6. 평가 기준선
- 목적: 대회 train과 AI Hub를 합친 118개 클래스 체계로도 로컬 평가 코드가 맞게 동작하는지 확인한다.

### 6-1. 변환 라벨로 mAP 계산 검증
- 00 8-1의 `coco_map`(pycocotools, IoU 기준 0.75~0.95)을 그대로 쓴다.
- 정답은 정제한 대회 train(00 2-5)과 정제한 AI Hub(2-5)를 합친 라벨이다.
- 정답 box를 그대로 예측으로 넣으면 1.0, 클래스를 모두 틀리게 넣으면 0이 나와야 한다.
    - AI Hub 라벨의 category_id를 drug_N으로 만들었으므로(1-3), 대회 train과 같은 알약이 같은 번호로 묶여야 1.0이 된다.

In [ ]:
def coco_map(gt_objects, pred_objects):
    """mAP@[0.75:0.95]를 계산한다. 정답은 (파일명, K-코드, xywh), 예측은 (파일명, K-코드, xywh, 점수)이다. (00 8-1과 같음)"""
    image_ids = {fn: i + 1 for i, fn in enumerate(sorted({o[0] for o in gt_objects}))}
    categories = sorted({o[1] for o in gt_objects} | {o[1] for o in pred_objects})
    gt = {
        'images': [{'id': i, 'width': IMG_W, 'height': IMG_H} for i in image_ids.values()],
        'annotations': [{'id': k + 1, 'image_id': image_ids[fn], 'category_id': c, 'bbox': list(b),
                         'area': b[2] * b[3], 'iscrowd': 0} for k, (fn, c, b) in enumerate(gt_objects)],
        'categories': [{'id': c} for c in categories],
    }
    if not pred_objects:
        return 0.0
    with contextlib.redirect_stdout(io.StringIO()):  # pycocotools의 진행 메시지 숨김
        coco_gt = COCO()
        coco_gt.dataset = gt
        coco_gt.createIndex()
        coco_dt = coco_gt.loadRes([{'image_id': image_ids[fn], 'category_id': c, 'bbox': list(b), 'score': s}
                                   for fn, c, b, s in pred_objects])
        evaluator = COCOeval(coco_gt, coco_dt, 'bbox')
        evaluator.params.iouThrs = np.linspace(0.75, 0.95, 5)  # 대회 지표의 IoU 기준
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    return float(evaluator.stats[0])  # stats[0]: 모든 IoU 기준과 클래스의 AP 평균


eval_objects = ([(fn, a['category_id'], a['bbox']) for fn, r in sorted(clean_records.items()) for a in r['annotations']]
                + [(fn, a['category_id'], a['bbox']) for fn, r in sorted(aihub_clean_records.items()) for a in r['annotations']])
eval_classes = sorted({c for _, c, _ in eval_objects})
wrong_class = {c: eval_classes[(i + 1) % len(eval_classes)] for i, c in enumerate(eval_classes)}
show_table(['예측', 'mAP@[0.75:0.95]'], [
    [f'정답 box와 클래스를 그대로 넣음 (이미지 {len({o[0] for o in eval_objects}):,}장, 객체 {len(eval_objects):,}개, 클래스 {len(eval_classes)}개)',
     f'{coco_map(eval_objects, [(*o, 1.0) for o in eval_objects]):.3f}'],
    ['box는 그대로, 클래스를 모두 다른 클래스로 바꿈', f'{coco_map(eval_objects, [(fn, wrong_class[c], b, 1.0) for fn, c, b in eval_objects]):.3f}'],
    ['예측 없음', f'{coco_map(eval_objects, []):.3f}'],
])

**인사이트**
- 정답 box와 클래스를 그대로 넣으면 mAP@[0.75:0.95]가 1.000이다. (이미지 12,221장, 객체 46,650개, 클래스 118개)
- 클래스를 모두 다른 클래스로 바꾸거나 예측이 없으면 0.000이다.
- 대회 train과 AI Hub에 함께 있는 54개 클래스도 같은 번호로 묶여 계산된다.

**결론**
- 118개 클래스 체계에서도 00의 `coco_map`을 그대로 로컬 평가에 쓴다.
- 제출 CSV의 category_id는 00과 같이 K-코드 숫자로 쓴다. AI Hub 클래스도 drug_N의 숫자라 같은 규칙이다.

## 7. EDA 결론
- 1~6번의 결과를 항목별 결정 사항으로 정리한다. 괄호 안 번호는 근거가 된 셀의 번호이다.
- 대회 원본 데이터의 결정(입력 크기, 증강, 후처리, 평가 방법 등)은 00 노트북 9절에 있다. 여기에는 AI Hub 보충에 관한 결정만 적는다.

| 항목 | 결정 | 근거 |
| --- | --- | --- |
| 보충 데이터 범위 | AI Hub 576 조합 데이터의 허용 zip 8쌍(TL_1·TS_1, TL_3~8·TS_3~8, VL_1·VS_1)만 사용. TL_2, TS_2는 받지 않음. 단일 데이터는 사용하지 않음 | 대회 규칙이 TL_2, TS_2만 금지 (1-1, RULE.md), 단일 이미지 합계 약 3,701GB (1-1) |
| 1차 정제 | JSON만 있는 이미지 파일명 6개(JSON 24개)와 `_index.png` 4,003장을 뺌 | 촬영 PNG가 없어 고칠 수 없음, `_index.png`는 정답을 그린 참고 사진 (1-1) |
| 라벨 변환 | 키 8개만 Kaggle 규칙으로 바꿈: category_id·categories.id·dl_idx = drug_N 숫자, categories.name = dl_name, images.id 1501부터, annotations.id 5692부터, iscrowd = 0 | 원본 dl_idx는 K-코드 - 1이라 그대로 쓰면 클래스가 하나씩 어긋남 (1-2), 바꾼 뒤 Kaggle 형식과 45,909/45,909 일치 (1-3) |
| 라벨 정제 | `01_raewoo_aihub_label_fixes.csv`(1,117줄, 이미지 960장)를 1-3 변환 뒤에 순서대로 적용 | copy 재배정 45줄, 좌표 오류와 라벨 누락 1,009줄, 갤러리에서 찾은 클래스·box 오류 63줄 (2-4) |
| 라벨 정제: 이미지 제외 | 사진에서 알약을 찾지 못한 20장을 뺌 | 2-2 수정 뒤에도 라벨 누락 21개가 남음 (2-2), box가 없는 알약은 배경으로 학습됨 |
| 라벨 정제: `-오류` 폴더 | 사진 속 실제 클래스로 붙여 사용 (K-023319 → K-029711, K-016551 → K-016548) | 9장 모두 파일명의 알약 대신 다른 알약이 놓여 있음 (2-2) |
| 클래스 읽기 | 파일명의 K-코드가 아니라 정제 라벨의 category_id를 사용 | 파일명에 없는 클래스가 있는 이미지 27장 (2-5) |
| 대회 사진과 섞기 | 밝기 전처리와 box 크기 보정 없이 그대로 섞음 | 촬영 조건 필드가 같음 (1-4), 밝기 분포가 test 범위를 덮음 (1-4), 같은 클래스·각도의 box 크기 차이 중앙값 0~1픽셀 (3-1) |
| 입력 크기, 증강, 후처리 | 00 노트북 9절의 결정을 그대로 사용 (입력 640으로 시작, NMS IoU 0.7, 최대 검출 수 300 등) | 입력 640에서 box 짧은 변 최소 60픽셀 (3-1), box 쌍 IoU 최대 0.151 (3-2) |
| 클래스 체계 | 118개 (대회 train 56개 + AI Hub에만 있는 62개) | AI Hub에 train 클래스 54개와 새 클래스 62개가 있음 (4-1), test 전용 알약 18종이 새 클래스에 있음 (5-2) |
| 보충 이미지 선택 | 목표 객체 수 200개로 고른 AI Hub 6,993장 (`data/external/aihub_targets.csv`) | 목표별 비교에서 변동계수 0.43으로 가장 낮음, 이미지는 전부의 58% (4-5) |
| 클래스 불균형 | 클래스 가중치와 반복 샘플링 없이 시작 | 변동계수 1.53(대회 train만) → 0.43, 객체 100개 미만 클래스는 K-003483(45개), K-020753(90개)뿐 (4-4, 4-5) |
| 분할 (팀원 작업) | 조합 단위로 나누고, 같은 사진이 든 조합 쌍 2개는 한 묶음으로 둠 | 같은 조합 = 같은 배치 3장 (4-2, 4-3), 다른 조합 이름의 같은 사진 6쌍 (1-6) |
| 평가 방법 | 00의 `coco_map`을 118개 클래스 정답으로 그대로 사용. 제출 category_id는 K-코드 숫자 | 정답을 넣으면 1.000, 클래스를 틀리면 0.000 (6-1) |

**추가 실험 후보**
- AI Hub 전부(11,989장)와 목표 200개 목록(6,993장)을 같은 설정으로 학습해 val mAP를 비교한다. (4-5)
    - 변동계수 차이가 0.04로 작아서, 데이터 양과 균형 중 무엇이 더 중요한지 결과로 정한다.
- val의 클래스별 AP에서 K-003483, K-020753처럼 객체가 적은 클래스가 낮으면, 그 클래스가 든 이미지를 반복 샘플링한다. (4-4)
- test에 나오지 않는 AI Hub 클래스 44개(62개 중 test 전용 18종을 뺀 것)를 빼고 74개 클래스로 학습해 118개 클래스와 비교한다. (4-1, 5-2)
    - 비슷한 알약끼리 헷갈리는 오답이 줄어드는지 본다.
- val에서 AI Hub 이미지와 대회 이미지의 IoU 0.95 AP를 따로 보고, 라벨 여백 차이가 점수에 영향을 주는지 확인한다. (3-1)

In [ ]:
_v = pair_iou['AI Hub (정제)']
print(len(aihub_clean_records), len(_v), (_v > 0).sum(), (_v > 0.05).sum(), _v.max())

In [ ]:
_rows = []
for _fn, _r in aihub_clean_records.items():
    _anns = _r['annotations']
    _best = max(((box_iou(a['bbox'], b['bbox']), a['category_id'], b['category_id']) for i, a in enumerate(_anns) for b in _anns[i + 1:]), default=(0, None, None))
    _rows.append((_best[0], _fn, _best[1], _best[2], len(_anns)))
_rows.sort(key=lambda t: (-t[0], t[1]))
for t in _rows[:12]: print(f'{t[0]:.3f}', t[1], t[2], t[3], t[4], aihub_clean_records[t[1]]['image']['camera_la'])
print(sum(t[0] > 0 for t in _rows), sum(t[0] > 0.05 for t in _rows))
del _rows, _best, _fn, _r, _anns, _v

In [ ]:
aihub_pair_iou = pair_iou['AI Hub (정제)']
top_overlap = sorted(zip(aihub_pair_iou, aihub_clean_records), key=lambda t: (-t[0], t[1]))[:10]
fig, axes = plt.subplots(2, 5, figsize=(25, 14), layout='constrained')
for rank, (ax, (iou, fn)) in enumerate(zip(axes.flat, top_overlap), start=1):
    anns = aihub_clean_records[fn]['annotations']
    pair = max(((a, b) for i, a in enumerate(anns) for b in anns[i + 1:]), key=lambda p: box_iou(p[0]['bbox'], p[1]['bbox']))
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for ann in anns:
        color = 'red' if any(ann is p for p in pair) else 'lime'
        x, y, w, h = ann['bbox']
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2))
        ax.text(x, y - 8, pill_label(ann), fontsize=7, va='bottom', bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0))
    ax.set_title(f"{rank}. IoU {iou:.3f} | {aihub_clean_records[fn]['image']['camera_la']}°\n{fn}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f'box 쌍의 IoU가 가장 큰 AI Hub 이미지 10장 (정제 라벨 {len(aihub_pair_iou):,}장 중 box가 겹친 이미지 {(aihub_pair_iou > 0).sum()}장, '
             f'IoU 0.05 초과 {(aihub_pair_iou > 0.05).sum()}장 | 빨강 = 가장 많이 겹친 box 쌍, 연두 = 나머지 box)', fontsize=14)
save_fig(fig, '3-2b_top_overlap')
plt.close(fig)
for iou, fn in top_overlap: print(f'{iou:.3f}', fn)

In [ ]:
aihub_pair_iou = pair_iou['AI Hub (정제)']
top_overlap = sorted(zip(aihub_pair_iou, aihub_clean_records), key=lambda t: (-t[0], t[1]))[:10]
fig, axes = plt.subplots(2, 5, figsize=(25, 14), layout='constrained')
for rank, (ax, (iou, fn)) in enumerate(zip(axes.flat, top_overlap), start=1):
    anns = aihub_clean_records[fn]['annotations']
    pair = max(((a, b) for i, a in enumerate(anns) for b in anns[i + 1:]), key=lambda p: box_iou(p[0]['bbox'], p[1]['bbox']))
    ax.imshow(Image.open(AIHUB_IMG_DIR / fn))
    for ann in anns:
        color = 'red' if any(ann is p for p in pair) else 'lime'
        x, y, w, h = ann['bbox']
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2))
        # 겹친 쌍의 라벨 글자가 같은 자리에 겹치지 않게, 두 번째 box는 box 아래에 적는다
        ty, va = (y + h + 8, 'top') if ann is pair[1] else (y - 8, 'bottom')
        ax.text(x, ty, pill_label(ann), fontsize=7, va=va, bbox=dict(facecolor=color, alpha=0.85, pad=1.5, linewidth=0))
    ax.set_title(f"{rank}. IoU {iou:.3f} | {aihub_clean_records[fn]['image']['camera_la']}°\n{fn}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f'box 쌍의 IoU가 가장 큰 AI Hub 이미지 10장 (정제 라벨 {len(aihub_pair_iou):,}장 중 box가 겹친 이미지 {(aihub_pair_iou > 0).sum()}장, '
             f'IoU 0.05 초과 {(aihub_pair_iou > 0.05).sum()}장 | 빨강 = 가장 많이 겹친 box 쌍, 연두 = 나머지 box)', fontsize=14)
save_fig(fig, '3-2b_top_overlap')
plt.close(fig)
for lo in (0, 0.05):
    sel = [fn for v, fn in zip(aihub_pair_iou, aihub_clean_records) if v > lo]
    print(lo, len(sel), collections.Counter(aihub_clean_records[f]['image']['camera_la'] for f in sel),
          sum(41149 in [a['category_id'] for a in aihub_clean_records[f]['annotations']] for f in sel))
print(collections.Counter(r['image']['camera_la'] for r in aihub_clean_records.values()))